# Large-FOV Nuclear Pipeline v19 (P150_CC1) — Vulcan 3.1.1, single-nucleus selection, growth analysis

**This copy is set up for the P150 CC1 condition** (`Condition/P150_CC1/Extract_Two.tif`): fresh inference, no segmentation reuse, and §25 compares it against the latest control v19 run. Code is identical to `Large_FOV_Nuclear_Pipeline_v19.ipynb`; only §3 and §25a differ. There is no gold for P150, so read the §16 rule counts and gallery against the control before trusting the curves.

Pipeline **and** analysis for one replicate, plus the condition × replicate comparison at the end.
Validation against the hand-traced gold ROIs, and all troubleshooting audits, live in the companion
notebook `Vulcan_Validation_v19.ipynb`, which loads this notebook's definition cells (tag `v19-def`)
and bootstrap cell (tag `v19-bootstrap`) so both notebooks run identical code.

Start with a fresh kernel and run top to bottom. Settings are in §2 (`PipelineConfig`) and §3 (bootstrap).

**Method (validated against gold, 2026-11-04)**

| step | rule | where |
|---|---|---|
| 1 | Vulcan 3.1.1 e55 inference: 2.5D input, Hann-blended tiles, 8-view TTA, all 8 heads saved | §13 |
| 2 | Per-plane nuclear instances (edge-guided watershed) | §13–14 |
| 3 | Droplet population: fitted droplet geometry, parent assignment, NLS enrichment ≥ 1.20, solidity ≥ 0.80, 5 % droplet-overlap exclusion; components with status `accepted` go on | §15 |
| 4 | Timepoints from `sel_min_timepoint` (T0 has no real nuclei) and planes ≥ `focus_min_z` | §16 |
| 5 | **Edge rule**: drop a section whose bounding box is within `sel_edge_px` of the frame border (truncated nuclei) | §16 |
| 6 | **Section selection without Z-linking**: keep a section unless a section within ±`sel_window_planes` planes, with centroid closer than the larger radius, is larger | §16 |
| 7 | **Area floor**: `sel_min_area_um2` | §16 |
| 8 | **Single-nucleus rule**: drop a section with solidity < `sel_min_solidity` (bilobed: touching nuclei segmented together) or with another nuclear instance within `sel_touch_px` on its plane (separated pieces of a cluster) | §16 |
| 9 | True acquisition time = `time_zero_offset_min` + t·frame + tile index (2×3 serpentine, 1 min/tile) | §17 |
| 10 | Median logistic growth fit in log area; phases at tm ± 1.317/k; absolute and relative growth velocity per phase | §21 |

**Removed relative to v18.2** (dead weight from earlier methods): Z-linking and best-Z by linked ID,
the §15 1 µm close-nuclei rule, legacy stage-2 NPC/membrane gate, fragmentation repair, focus scoring,
probability re-extraction, single-plane debug cells, §38 audits, checkpoint comparison, hard-negative
export cells, scratch cells. The droplet-population library (`droplet_population`, `droplet_geometry`)
is inlined in §10 so all analysis lives in this notebook.

**Reusing a v18.2 segmentation.** Set `SEGMENTATION_SOURCE_RUN` in §3 to a finished v18.2/v19 run
directory. Inference is then skipped; the source run's checkpoint is checked field-by-field against the
segmentation-relevant settings here and refused on any difference. The source run is only read.

**Stage map**

| § | stage | output (this run) |
|---|---|---|
| 1–12 | imports, config, definitions | — |
| 13–14 | segmentation (or reuse), per-plane objects | `objects/plane_objects.pkl` |
| 15 | droplet population | `analysis/selected_population/` |
| 16 | single-nucleus selection | `objects/sections_all.pkl`, `objects/nuclei_selected.pkl` |
| 17–19 | timing, tracking, halos, radial sweep | `tracking/`, `analysis/` |
| 20 | export of the single-nucleus table | `exports/single_nuclei/` |
| 21 | growth curve, phases, velocities (this replicate) | `exports/growth/` |
| 22–23 | membrane / envelope figures (population = selected single nuclei) | `exports/figures/` |
| 24 | database export | `database_export/` |
| 25 | conditions × replicates | `CONDITION_RUNS` → `COMPARISON_DIR` |

## 1. Imports

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import time
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import shutil

from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage import filters, measure, morphology

try:
    import tensorflow as tf
except Exception:
    tf = None

try:
    import tifffile as tiff
except Exception:
    tiff = None

try:
    from sklearn.cluster import DBSCAN
except Exception:
    DBSCAN = None

import copy
from types import SimpleNamespace
from tqdm.auto import tqdm


## 2. Configuration
Calibration: 0.1625 µm/pixel, 2.0 µm Z step. Selection rules (`sel_*`) and the time offset are here.

In [ ]:
def get_n_workers() -> int:
    """Return the number of CPU cores available for parallel processing."""
    return int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))


@dataclass
class PipelineConfig:
    # ── Inputs ────────────────────────────────────────────────────────────
    model_name: str = "Vulcan_3.1.1_iezqw_s42_final.keras"   # overridden with the full path in §3
    input_image_name: str = "control_extract_1.1.tif"
    image_subdir: str = "Condition/Control"
    pipeline_revision: str = "v19.0-vulcan311-selection1"
    model_sha256: str = ""
    input_fingerprint: str = ""

    # ── Reuse of an existing segmentation (location only; see §3) ─────────
    # A finished run directory whose segmentation/ and mask_tifs/ are read instead of
    # running inference. Not part of the run signature; checked field-by-field instead.
    segmentation_source_run: Optional[str] = None

    # ── Model input contract (Vulcan 2.5.x / 3.x) ─────────────────────────
    n_z_context: int = 5
    normalization_mode: str = "global_t"
    norm_stats_stride: int = 4
    norm_stats_z_stride: int = 4
    tta: bool = True
    patch_size: int = 512
    patch_stride: int = 384
    batch_size: int = 8          # tune down if GPU OOM
    model_output_channels: int = 8
    model_probability_output_indices: Tuple[int, ...] = (0, 1, 3, 4)
    model_outputs_logits: bool = True

    # ── Post-processing (instances) ───────────────────────────────────────
    watershed_core_thresh: float = 0.7
    watershed_edge_weight: float = 1.0
    watershed_min_marker_dist_px: int = 20
    sharpness_window_px: int = 7
    droplet_threshold: float = 0.5
    npc_threshold: float = 0.5
    nucleus_threshold: float = 0.5
    droplet_class_index: int = 1     # display label map only
    npc_class_index: int = 2
    nucleus_class_index: int = 3
    num_classes: int = 4

    # ── Eligible planes ───────────────────────────────────────────────────
    focus_min_z: int = 6                 # planes below are coverslip artefacts; context only
    focus_max_z: Optional[int] = None

    # ── Imaging ───────────────────────────────────────────────────────────
    pixel_size_um: float = 0.1625
    z_step_um: float = 2.0
    nuclear_channel_index: int = 1
    membrane_channel_index: int = 0
    npc_channel_index: int = 2

    # ── Acquisition layout and time ───────────────────────────────────────
    tile_rows: int = 2
    tile_cols: int = 3
    minutes_per_tile: float = 1.0
    serpentine_scan: bool = True
    # Minutes from the biological zero (extract activation / encapsulation) to the start of
    # frame 0. Analysis-only: excluded from the run signature, so editing it re-times the
    # tables without forcing new inference. Record it per replicate.
    time_zero_offset_min: float = 0.0

    # ── Single-nucleus selection (§16; validated against gold 2026-11-04) ─
    sel_min_timepoint: int = 1           # T0 has no real nuclei
    sel_edge_px: int = 2                 # border-truncated nuclei
    sel_window_planes: int = 2           # local area-maximum window, ± planes
    sel_min_area_um2: float = 50.0       # fragment floor
    sel_min_solidity: float = 0.90       # below: bilobed (touching nuclei as one object)
    sel_touch_px: int = 3                # another instance this close on the plane: cluster

    # ── Halo analysis ─────────────────────────────────────────────────────
    halo_step_px: int = 4
    n_halos: int = 4
    erosion_px: int = 4

    # ── Radial sweep ──────────────────────────────────────────────────────
    radial_channel_index: int = 0        # membrane / NE channel
    radial_n_angles: int = 180
    radial_step_size_px: float = 1.0
    radial_exclude_nucleus: bool = True
    radial_max_ray_steps: int = 4000

    # ── Tracking across time ──────────────────────────────────────────────
    time_track_tolerance_um: float = 5.0
    track_dbscan_eps_um: float = 6.0
    track_dbscan_min_samples: int = 1
    track_crowded_distance_scale: float = 0.65
    track_area_log_weight: float = 12.0
    track_area_ratio_max: float = 5.0

    # ── HPC ───────────────────────────────────────────────────────────────
    n_workers: Optional[int] = None

    # Fields that determine the segmentation (instances, heads). A reused segmentation must
    # match these exactly; everything else only affects downstream tables.
    SEGMENTATION_FIELDS = (
        "model_sha256", "input_fingerprint", "n_z_context", "normalization_mode",
        "norm_stats_stride", "norm_stats_z_stride", "tta", "patch_size", "patch_stride",
        "model_output_channels", "model_probability_output_indices", "model_outputs_logits",
        "watershed_core_thresh", "watershed_edge_weight", "watershed_min_marker_dist_px",
        "sharpness_window_px", "droplet_threshold", "npc_threshold", "nucleus_threshold",
        "focus_min_z", "focus_max_z", "nuclear_channel_index", "membrane_channel_index",
        "npc_channel_index")
    # Legacy options that must have been OFF in a reused v18.x segmentation.
    LEGACY_MUST_BE_OFF = ("require_stage2_gate", "repair_enabled", "use_focus_z_selection",
                          "probability_hyperstack_input")
    # Excluded from the run signature: locations and analysis-only settings.
    _NON_SIGNATURE_FIELDS = ("segmentation_source_run", "time_zero_offset_min", "n_workers")

    def __post_init__(self):
        if self.n_workers is None:
            self.n_workers = min(get_n_workers(), 16)
        self.paths = None   # set by the bootstrap cell

    # ── Unit conversions ──────────────────────────────────────────────────
    @property
    def time_track_tolerance_px(self) -> float:
        return self.time_track_tolerance_um / self.pixel_size_um

    @property
    def track_dbscan_eps_px(self) -> float:
        return self.track_dbscan_eps_um / self.pixel_size_um

    @property
    def frame_interval_min(self) -> float:
        return self.tile_rows * self.tile_cols * self.minutes_per_tile

    # ── Signatures ────────────────────────────────────────────────────────
    def to_serializable_dict(self) -> Dict[str, object]:
        return asdict(self)

    def to_run_signature(self) -> Dict[str, object]:
        """Config subset that defines this run's RESULTS (hashed into the run id)."""
        d = asdict(self)
        for k in self._NON_SIGNATURE_FIELDS:
            d.pop(k, None)
        if self.segmentation_source_run:
            d["segmentation_source_run_id"] = Path(self.segmentation_source_run).name
        return d

    def segmentation_signature(self) -> Dict[str, object]:
        return json.loads(json.dumps({k: getattr(self, k) for k in self.SEGMENTATION_FIELDS}))

    # ── Paths (delegated to the run-scoped resolver) ──────────────────────
    def _require_paths(self):
        if getattr(self, "paths", None) is None:
            raise RuntimeError("cfg.paths is not set — run the bootstrap cell (§3) first.")
        return self.paths

    @property
    def model_path(self) -> Path:
        return self._require_paths().project.models_dir / self.model_name

    @property
    def input_image_path(self) -> Path:
        return self._require_paths().project.raw_images_dir / self.image_subdir / self.input_image_name

    @property
    def output_dir(self) -> Path:
        return self._require_paths().run_dir

    # Segmentation artefacts come from the source run when one is set (read-only).
    @property
    def seg_dir(self) -> Path:
        if self.segmentation_source_run:
            return Path(self.segmentation_source_run) / "segmentation"
        return self._require_paths().seg_dir

    @property
    def mask_tif_dir(self) -> Path:
        if self.segmentation_source_run:
            return Path(self.segmentation_source_run) / "mask_tifs"
        return self._require_paths().mask_tif_dir

    @property
    def obj_dir(self) -> Path:
        return self._require_paths().obj_dir

    @property
    def track_dir(self) -> Path:
        return self._require_paths().track_dir

    @property
    def analysis_dir(self) -> Path:
        return self._require_paths().analysis_dir

    @property
    def qc_dir(self) -> Path:
        return self._require_paths().qc_dir

    @property
    def exports_dir(self) -> Path:
        return self._require_paths().exports_dir

    @property
    def segmentation_index_path(self) -> Path:
        return self.seg_dir / "segmentation_index.pkl"

    @property
    def segmentation_roi_table_path(self) -> Path:
        return self.seg_dir / "segmentation_roi_table.pkl"

    @property
    def segmentation_class_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_class_hyperstack.tif"

    @property
    def segmentation_label_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_label_hyperstack.tif"

    @property
    def nucleus_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "nucleus_instance_hyperstack.tif"

    @property
    def droplet_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "droplet_instance_hyperstack.tif"

    @property
    def segmentation_probability_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_probability_hyperstack.tif"

    def __repr__(self) -> str:
        keys = ["model_name", "input_image_name", "segmentation_source_run", "time_zero_offset_min",
                "pixel_size_um", "sel_edge_px", "sel_window_planes", "sel_min_area_um2",
                "sel_min_solidity", "sel_touch_px", "n_workers"]
        return "\n".join(f"{k}: {getattr(self, k)}" for k in keys)


def segmentation_signature_diff(stored: dict, config: PipelineConfig) -> dict:
    """Differences between a checkpoint's stored config and the live segmentation fields.
    Also reports legacy options that were ON in the stored config."""
    stored = json.loads(json.dumps(stored or {}, default=str))
    live = config.segmentation_signature()
    diffs = {k: (stored.get(k, "<missing>"), v) for k, v in live.items() if stored.get(k, "<missing>") != v}
    for k in config.LEGACY_MUST_BE_OFF:
        if stored.get(k) not in (None, False, 0, "", "None"):
            diffs[k] = (stored.get(k), "must be off")
    return diffs


## 3. Path resolver and run bootstrap

In [ ]:
# =====================================================================
# Path resolver — single source of truth for every filesystem location.
# Reads the logical layout in STRUCTURE and materialises it under two
# physical roots. On Cheaha: code_root=$HOME/Projects (small, git),
# data_root=/data/user/$USER/<project> (5 TB, all inputs + outputs).
# Every run writes under Runs/<run_id>/, so a stale pickle from a
# different config can never be loaded by accident. (Mirrors structure.json.)
# =====================================================================
import getpass, hashlib, datetime as _dt
import json as _json
from pathlib import Path as _Path

STRUCTURE = {
    "code_root": {"Python": {"Environments": None,
                             "Model_Training": {"Training_Scripts": None},
                             "Image_Segmentation": None, "Image_Analysis": None},
                  "R_Projects": None},
    "data_root": {"Inputs": {"Raw_Images": None, "Models": None, "Training_Data": None},
                  "Runs": None, "Legacy": None},
    "run_template": {"segmentation": None, "objects": None, "tracking": None,
                     "analysis": None, "qc": None, "mask_tifs": None, "exports": None},
}

_RUN_STAGE_ATTRS = {"segmentation": "seg_dir", "objects": "obj_dir",
                    "tracking": "track_dir", "analysis": "analysis_dir",
                    "qc": "qc_dir", "mask_tifs": "mask_tif_dir",
                    "exports": "exports_dir"}


def _build_tree(base, spec):
    """Recursively create a directory tree under base according to the spec dict."""
    base.mkdir(parents=True, exist_ok=True)
    if not spec:
        return
    for name, child in spec.items():
        if str(name).startswith("_"):
            continue
        _build_tree(base / name, child)


class ProjectPaths:
    """Resolve all filesystem locations for a project, given two roots:"""
    def __init__(self, code_root, data_root, structure=STRUCTURE, create=True):
        self.spec = structure
        self.code_root = _Path(code_root).expanduser().resolve()
        self.data_root = _Path(data_root).expanduser().resolve()
        self.run_template = structure.get("run_template", {})
        self.raw_images_dir    = self.data_root / "Inputs" / "Raw_Images"
        self.models_dir        = self.data_root / "Inputs" / "Models"
        self.training_data_dir = self.data_root / "Inputs" / "Training_Data"
        self.runs_dir          = self.data_root / "Runs"
        self.legacy_dir        = self.data_root / "Legacy"
        if create:
            _build_tree(self.code_root, self.spec.get("code_root"))
            _build_tree(self.data_root, self.spec.get("data_root"))

    @classmethod
    def for_cheaha(cls, project_name, code_subdir="Projects", create=True):
        """Resolve paths for a Cheaha HPC. The code root is $HOME/<code_subdir>,"""
        user = getpass.getuser()
        home = _Path(os.environ.get("HOME", f"/home/{user}"))
        user_data = _Path(os.environ.get("USER_DATA", f"/data/user/{user}")).expanduser()
        return cls(home / code_subdir, user_data / project_name, create=create)

    @staticmethod
    def make_run_id(label, config_dict, timestamp=False):
        """Stable per config: <label>__cfg-<hash>. Same config -> same dir, so
        RUN_*=False loads reuse the existing run. Any config change flips the
        hash -> a new dir -> no stale mixing. Set timestamp=True to force a
        distinct dir for same-config reruns."""
        h = hashlib.sha1(
            _json.dumps(config_dict, sort_keys=True, default=str).encode()).hexdigest()[:8]
        safe = "".join(c if (c.isalnum() or c in "-._") else "_" for c in str(label))
        if timestamp:
            return f"{safe}__{_dt.datetime.now():%Y%m%d_%H%M}__cfg-{h}"
        return f"{safe}__cfg-{h}"

    def for_run(self, run_id, create=True):
        run_dir = self.runs_dir / run_id
        if create:
            _build_tree(run_dir, self.run_template)
        return RunPaths(self, run_id, run_dir)

    def list_runs(self):
        if not self.runs_dir.exists():
            return []
        return sorted(p.name for p in self.runs_dir.iterdir() if p.is_dir())


class RunPaths:
    def __init__(self, project, run_id, run_dir):
        self.project, self.run_id, self.run_dir = project, run_id, run_dir
        for stage, attr in _RUN_STAGE_ATTRS.items():
            setattr(self, attr, run_dir / stage)
        self.config_snapshot_path = run_dir / "config.json"
        self.manifest_path = run_dir / "manifest.json"

    def snapshot_config(self, config_dict):
        self.config_snapshot_path.write_text(
            _json.dumps(config_dict, indent=2, sort_keys=True, default=str))

    def load_snapshot(self):
        if self.config_snapshot_path.exists():
            return _json.loads(self.config_snapshot_path.read_text())
        return None

    def assert_config_matches(self, config_dict, strict=True):
        """Refuse to trust cached stages if the live config drifted from the
        snapshot that produced this run's data. Call before any RUN_*=False load."""
        config_dict = _json.loads(_json.dumps(config_dict, default=str))
        snap = self.load_snapshot()
        if snap is None:
            return
        diffs = {k: (snap.get(k), config_dict.get(k))
                 for k in set(snap) | set(config_dict)
                 if snap.get(k) != config_dict.get(k)}
        if diffs:
            msg = ("[stale-guard] live config differs from snapshot in run '%s':\n%s"
                   % (self.run_id, "\n".join(f"    {k}: snapshot={s!r} live={l!r}"
                                             for k, (s, l) in diffs.items())))
            if strict:
                raise RuntimeError(msg)
            print("WARNING:", msg)

    def write_manifest(self, **fields):
        m = {}
        if self.manifest_path.exists():
            m = _json.loads(self.manifest_path.read_text())
        m.update(fields)
        m["updated_at"] = _dt.datetime.now().isoformat(timespec="seconds")
        self.manifest_path.write_text(_json.dumps(m, indent=2, default=str))


In [ ]:
# §3 BOOTSTRAP. Set the options below, then run. The model SHA256, the image fingerprint and
# every result-affecting field determine the run directory (prior runs stay untouched).
import hashlib
cfg = PipelineConfig()

# Model: Vulcan 3.1.1 final checkpoint (epoch 55; standalone hard negatives).
cfg.model_name = ('/home/tdeibert/Projects/Nuclear_Scaling/Models/runs/Vulcan_3.1.1_iezqw_2d87f0ec3c/'
                  'Vulcan_3.1.1_iezqw_s42_final.keras')
cfg.input_image_name = "Extract_Two.tif"
cfg.image_subdir = "Condition/P150_CC1"      # /data/user/tdeibert/Nuclear_Scaling/Inputs/Raw_Images/Condition/P150_CC1/

# Minutes from extract activation / encapsulation to the start of frame 0 for THIS replicate.
# Must be on the same biological zero as the control run it is compared with in §25.
cfg.time_zero_offset_min = 0.0               # TODO: set from the P150 acquisition record

# Acquisition layout. These are the control's values (control_extract_1.1.json); there is no P150
# metadata JSON in Configs/, so CONFIRM them against the P150 acquisition before trusting true time.
cfg.tile_rows, cfg.tile_cols = 2, 3
cfg.minutes_per_tile = 1.0
cfg.serpentine_scan = True
cfg.pixel_size_um, cfg.z_step_um = 0.1625, 2.0

# New image: segment in this run (no reuse).
SEGMENTATION_SOURCE_RUN = None
cfg.segmentation_source_run = None if SEGMENTATION_SOURCE_RUN is None else str(SEGMENTATION_SOURCE_RUN)

paths = ProjectPaths.for_cheaha(project_name="Nuclear_Scaling")
_model_path = paths.models_dir / cfg.model_name
assert _model_path.exists(), f"Missing model: {_model_path}"
_image_path = paths.raw_images_dir / cfg.image_subdir / cfg.input_image_name
_sha = hashlib.sha256()
with _model_path.open("rb") as _stream:
    for _block in iter(lambda: _stream.read(8 * 1024**2), b""):
        _sha.update(_block)
cfg.model_sha256 = _sha.hexdigest()
_st = _image_path.stat()
cfg.input_fingerprint = f"{_image_path.resolve()}:{_st.st_size}:{_st.st_mtime_ns}"

if cfg.segmentation_source_run is not None:
    _src = Path(cfg.segmentation_source_run)
    assert (_src / "segmentation" / "_checkpoint.json").exists(), f"No segmentation checkpoint in {_src}"
    _src_manifest = json.loads((_src / "manifest.json").read_text()) if (_src / "manifest.json").exists() else {}
    if _src_manifest.get("model_sha256") not in (None, cfg.model_sha256):
        raise RuntimeError(f"Source run used a different model ({_src_manifest.get('model_sha256')}).")

run = paths.for_run(ProjectPaths.make_run_id(
    label=Path(cfg.input_image_name).stem + "_v19", config_dict=cfg.to_run_signature()))
cfg.paths = run
run.assert_config_matches(cfg.to_run_signature())
run.snapshot_config(cfg.to_run_signature())
run.write_manifest(input_image=cfg.input_image_name, model=cfg.model_name,
                   model_sha256=cfg.model_sha256, run_id=run.run_id,
                   pipeline_revision=cfg.pipeline_revision,
                   segmentation_source_run=cfg.segmentation_source_run,
                   time_zero_offset_min=cfg.time_zero_offset_min)
print("Run:", run.run_dir)
print("Segmentation read from:", cfg.seg_dir.parent)
print("Model SHA256:", cfg.model_sha256)
cfg


## 4. Configuration I/O and GPU setup

In [ ]:
def save_config(config: PipelineConfig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))


def load_config(in_path: Path) -> PipelineConfig:
    data = json.loads(in_path.read_text())
    return PipelineConfig(**data)


In [ ]:
def configure_tensorflow_for_gpu(config: PipelineConfig) -> None:
    if tf is None:
        print("TensorFlow not available — skipping GPU setup.")
        return

    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs found:", gpus)

    if getattr(config, "use_mixed_precision", False):
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy("mixed_float16")
            print("Mixed precision enabled.")
        except Exception as exc:
            print("Could not enable mixed precision:", exc)

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print("Could not set memory growth:", exc)


## 5. Image loading helpers

In [ ]:
def load_image_5d(image_path: Union[str, Path]) -> np.ndarray:
    if tiff is None:
        raise ImportError("tifffile is required to load TIFF data.")
    arr = tiff.imread(image_path)
    print("Loaded image shape:", arr.shape)
    return arr


def get_nuclear_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.nuclear_channel_index]


def get_membrane_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.membrane_channel_index]


def get_full_plane_yxc(
    img_5d: np.ndarray,
    t_idx: int,
    z_idx: int,
    config: "PipelineConfig" = None,
) -> np.ndarray:
    """
    Return a 3-channel plane in (Y, X, C) format ordered to match training input.

    Training channel order: (NLS/nucleus, NPC, Membrane)
    Hyperstack channel order: Ch0=Membrane, Ch1=NLS, Ch2=NPC

    If config is provided, uses config channel indices for explicit ordering.
    Falls back to default training order (1, 2, 0) if config is None.
    """
    plane_cyx = img_5d[t_idx, z_idx]  # shape: (C, Y, X)
    if config is not None:
        ch_order = [
            config.nuclear_channel_index,   # NLS first  (training ch0)
            config.npc_channel_index,        # NPC second (training ch1)
            config.membrane_channel_index,   # Mem third  (training ch2)
        ]
    else:
        ch_order = [1, 2, 0]  # default: NLS, NPC, Membrane
    return np.moveaxis(plane_cyx[ch_order], 0, -1)  # (Y, X, C)

def context_plane_indices(z_idx: int, n_z: int, n_context: int) -> List[int]:
    """Training-faithful Z context, edge-replicated at stack boundaries."""
    if n_context < 1 or n_context % 2 == 0:
        raise ValueError(f"n_z_context must be a positive odd integer, got {n_context}")
    half = n_context // 2
    return [int(np.clip(z_idx + dz, 0, n_z - 1))
            for dz in range(-half, half + 1)]


def get_full_context_yxc(
    img_5d: np.ndarray,
    t_idx: int,
    z_idx: int,
    config: "PipelineConfig",
) -> np.ndarray:
    """Return plane-major 2.5D input: [NLS, NPC, Mem] for each context Z."""
    planes = [get_full_plane_yxc(img_5d, t_idx, zz, config)
              for zz in context_plane_indices(z_idx, img_5d.shape[1],
                                              config.n_z_context)]
    return np.concatenate(planes, axis=-1)


def normalization_stats_for_timepoint(
    img_5d: np.ndarray,
    t_idx: int,
    config: "PipelineConfig",
) -> Optional[List[Tuple[float, float]]]:
    """Training-faithful p1/p99.8 stats pooled across eligible Z planes.

    Returns stats in the same plane-major channel order as get_full_context_yxc.
    With per_patch mode, returns None and _preprocess_patch computes each channel.
    """
    if config.normalization_mode == "per_patch":
        return None
    if config.normalization_mode != "global_t":
        raise ValueError(f"Unsupported normalization_mode: {config.normalization_mode!r}")
    n_z = img_5d.shape[1]
    z0 = max(0, int(config.focus_min_z or 0))
    zs = list(range(z0, n_z, int(config.norm_stats_z_stride))) or [n_z // 2]
    sy = sx = max(1, int(config.norm_stats_stride))
    base_order = [config.nuclear_channel_index, config.npc_channel_index,
                  config.membrane_channel_index]
    base_stats = []
    for channel_idx in base_order:
        pool = np.concatenate([
            np.asarray(img_5d[t_idx, z, channel_idx, ::sy, ::sx],
                       dtype=np.float32).ravel()
            for z in zs])
        lo, hi = np.percentile(pool, [1.0, 99.8])
        base_stats.append((float(lo), float(hi)))
    return base_stats * int(config.n_z_context)


## 6. Segmentation helpers (Vulcan inference and instances)

In [ ]:
def load_unet_model(model_path: Union[str, Path]):
    if tf is None:
        raise ImportError("TensorFlow is not available.")
    return tf.keras.models.load_model(model_path, compile=False)


def normalize_channel(
    ch2d: np.ndarray,
    stats: Optional[Tuple[float, float]] = None,
) -> np.ndarray:
    """Training-faithful p1/p99.8 normalization, optionally with fixed stats."""
    ch = ch2d.astype(np.float32)
    lo, hi = np.percentile(ch, [1.0, 99.8]) if stats is None else stats
    if hi <= lo:
        return np.zeros_like(ch)
    return np.clip((ch - lo) / (hi - lo), 0.0, 1.0)

def _preprocess_patch(
    patch_yxc: np.ndarray,
    channel_stats: Optional[List[Tuple[float, float]]] = None,
) -> np.ndarray:
    """Normalize every input channel with the same contract used in training.

    Vulcan 2.5.3 receives 15 plane-major channels: [NLS, NPC, Membrane]
    repeated over five Z planes. Fixed global_t statistics preserve relative
    intensity across Z; per_patch normalization remains supported.
    """
    x = patch_yxc.astype(np.float32)
    if channel_stats is not None and len(channel_stats) != x.shape[-1]:
        raise ValueError(
            f"normalization stats have {len(channel_stats)} channels, "
            f"but the patch has {x.shape[-1]}")
    return np.stack([
        normalize_channel(x[..., c], None if channel_stats is None else channel_stats[c])
        for c in range(x.shape[-1])
    ], axis=-1)

def collapse_label_map(droplet: np.ndarray, npc: np.ndarray,
                       nucleus: np.ndarray, config: PipelineConfig) -> np.ndarray:
    """Single-int DISPLAY label map. Priority Nucleus>NPC>Droplet>BG (v9
    collapse_to_integer_hwc). Display/QC only -- never an extraction source."""
    out = np.zeros(droplet.shape, dtype=np.uint8)
    out[droplet] = config.droplet_class_index
    out[npc]     = config.npc_class_index
    out[nucleus] = config.nucleus_class_index   # last write wins = highest priority
    return out


def _generate_patch_starts(length: int, patch_size: int, stride: int) -> List[int]:
    """Generate patch start positions that always include the image end."""
    starts = list(range(0, max(length - patch_size + 1, 1), stride)) or [0]
    last = max(0, length - patch_size)
    if starts[-1] != last:
        starts.append(last)
    return sorted(set(starts))


# =====================================================================
# Stage-2 gate — NPC-shell organisation + membrane co-localisation
# PORTED FROM vulcan_training_1.1.ipynb (sections 4b, 5b, 6)      [v16.2]
# ---------------------------------------------------------------------
# The training-set generator has always applied this gate. The segmentation
# block never did — it accepted any component surviving clean_nucleus_mask,
# which tests area, solidity and holes: all properties of the MASK, every one
# of which a droplet cap passes trivially (in a single Z a cap is a small,
# round, solid, filled disc). That asymmetry is the t = 0-3 contamination.
#
# The gate tests ORGANISATION, NOT PRESENCE:
#   1. Shell     — NPC puncta in an annulus around the nucleus edge must admit
#                  a coherent RANSAC circle. An envelope fits; scattered cap or
#                  antibody-clump signal does not.
#   2. Membrane  — each punctum needs membrane signal within R, and a minimum
#                  fraction must be supported. True NPC sits on the NE, which
#                  the dim membrane probe also marks.
# Both must fire.
#
# The functions below are transcribed from the generator so the two can be
# diffed. ONE deliberate deviation, marked inline: _punctum_centroids reshapes
# to (-1, 2) so the empty case has a 2-D shape. Behaviourally identical.
#
# Normalisation regions are the DROPLET INTERIOR in every case, exactly as in
# the generator, which is why the adapter pairs each candidate with its parent
# droplet instance rather than using a local annulus.
# =====================================================================


# ---------------------------------------------------------------------
# v16 adapter — per-candidate, parent-droplet-scoped
# ---------------------------------------------------------------------


VULCAN_HEADS = ('background', 'droplet_interior', 'droplet_edge', 'npc',
                'nucleus_interior', 'nucleus_edge', 'nucleus_equatorial', 'z_offset')

def file_sha256(path):
    import hashlib
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            h.update(block)
    return h.hexdigest()


def eligible_z_planes(n_z, config):
    lo = max(0, int(config.focus_min_z))
    hi = n_z - 1 if config.focus_max_z is None else min(n_z - 1, config.focus_max_z)
    return list(range(lo, hi + 1))


def full_head_path(config):
    return config.seg_dir / 'vulcan_all_heads.tif'


def full_head_metadata_path(config):
    return config.seg_dir / 'vulcan_all_heads.json'


def validate_vulcan_model(model, config):
    if tuple(model.input_shape[1:]) != (config.patch_size, config.patch_size, 15):
        raise ValueError('Vulcan input must be (512, 512, 15): ' + str(model.input_shape))
    if int(model.output_shape[-1]) != 8:
        raise ValueError('Expected seven mask logits and one Z-offset output')
    masks = model.get_layer('masks_logits')
    zhead = model.get_layer('z_offset')
    if masks.filters != 7 or masks.activation.__name__ != 'linear':
        raise ValueError('Unexpected mask-head contract')
    if zhead.filters != 1 or zhead.activation.__name__ != 'softplus':
        raise ValueError('Unexpected Z-offset contract')
    if config.n_z_context != 5 or config.model_probability_output_indices != (0, 1, 3, 4):
        raise ValueError('Unsupported input context or semantic projection')


def predict_vulcan_plane(img_5d, t, z, model, config, channel_stats=None):
    """Streaming Hann-blended dihedral TTA, matching training H1 inference."""
    if z not in eligible_z_planes(img_5d.shape[1], config):
        raise ValueError('Target plane is outside the configured Z range')
    if channel_stats is None:
        channel_stats = normalization_stats_for_timepoint(img_5d, t, config)
    full = get_full_context_yxc(img_5d, t, z, config)
    if config.normalization_mode != 'global_t':
        raise ValueError('This validated model requires global_t normalization')
    full = _preprocess_patch(full, channel_stats)
    h, w = full.shape[:2]
    ps = config.patch_size
    if min(h, w) < ps:
        raise ValueError('Image must contain at least one full model patch')
    origins = [(y, x) for y in _generate_patch_starts(h, ps, config.patch_stride)
               for x in _generate_patch_starts(w, ps, config.patch_stride)]
    win = (np.hanning(ps)[:, None] * np.hanning(ps)[None, :]).astype(np.float32) + 1e-6
    acc = np.zeros((h, w, 8), np.float32)
    weight = np.zeros((h, w), np.float32)
    transforms = [(k, f) for k in range(4) for f in (False, True)] if config.tta else [(0, False)]
    for start in range(0, len(origins), config.batch_size):
        chunk = origins[start:start + config.batch_size]
        batch = np.stack([full[y:y+ps, x:x+ps] for y, x in chunk])
        mean = np.zeros(batch.shape[:3] + (8,), np.float32)
        for k, flip in transforms:
            xb = np.stack([np.rot90(np.flip(a, 1) if flip else a, k) for a in batch])
            p = np.asarray(model(xb, training=False).numpy(), np.float32)
            if p.shape != mean.shape or not np.isfinite(p).all():
                raise ValueError('Invalid or nonfinite model output')
            p[..., :7] = 1 / (1 + np.exp(-np.clip(p[..., :7], -80, 80)))
            for i, a in enumerate(p):
                a = np.rot90(a, -k)
                mean[i] += (np.flip(a, 1) if flip else a) / len(transforms)
        for (y, x), p in zip(chunk, mean):
            acc[y:y+ps, x:x+ps] += p * win[..., None]
            weight[y:y+ps, x:x+ps] += win
    return acc / np.maximum(weight[..., None], 1e-6)


def vulcan_instances(heads, nls, config, npc_raw=None, mem_raw=None, t_idx=None):
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed
    core, edge = heads[..., 4], heads[..., 5]
    seed = morphology.binary_erosion(core > config.watershed_core_thresh, morphology.disk(2))
    dist = ndi.distance_transform_edt(seed)
    comp = measure.label(seed)
    peaks = peak_local_max(dist, min_distance=config.watershed_min_marker_dist_px,
                           labels=comp, exclude_border=False)
    markers = np.zeros(seed.shape, np.int32)
    for i, (y, x) in enumerate(peaks, 1):
        markers[y, x] = i
    if not markers.any():
        markers = comp
    rng = ndi.maximum_filter(nls, config.sharpness_window_px) - ndi.minimum_filter(nls, config.sharpness_window_px)
    sharp = filters.sobel(nls) / (rng + 1e-3 * (nls.max() - nls.min() + 1e-9))
    lab = watershed(sharp + config.watershed_edge_weight * edge, markers,
                    mask=(core > config.nucleus_threshold) | (edge > config.nucleus_threshold))
    droplet = heads[..., 1] > config.droplet_threshold
    drops = measure.label(droplet, connectivity=2)
    # v19: the legacy stage-2 NPC/membrane gate is removed (npc_raw/mem_raw/t_idx kept for call compatibility).
    if max(int(lab.max()), int(drops.max())) > np.iinfo(np.uint16).max:
        raise OverflowError('Instance count exceeds uint16; no truncated labels written')
    return lab.astype(np.uint16), drops.astype(np.uint16)


## 7. TIFF helpers, segmentation checkpoint, region props, segmentation loop, objects

In [ ]:
def _write_hyperstack_tiff(arr: np.ndarray, out_path: Path, axes: str,
                           prefer_imagej: bool = True) -> None:
    if tiff is None:
        raise ImportError("tifffile is required.")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(arr)
    # BigTIFF for arrays >= ~2 GB to avoid ImageJ truncation.
    use_imagej = prefer_imagej and arr.nbytes < (2 * 1024**3 - 32 * 1024**2)
    tiff.imwrite(out_path, arr,
                 imagej=use_imagej, bigtiff=not use_imagej,
                 metadata={"axes": axes})


def save_probability_hyperstack_tiff(probability_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(probability_5d, out_path, axes="TZCYX", prefer_imagej=False)


def save_class_hyperstack_tiff(class_mask_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff((class_mask_5d.astype(np.uint8) * 255), out_path,
                           axes="TZCYX", prefer_imagej=True)


def save_label_hyperstack_tiff(label_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(label_4d.astype(np.uint8), out_path,
                           axes="TZYX", prefer_imagej=True)


def save_instance_hyperstack_tiff(instance_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(instance_4d, out_path, axes="TZYX", prefer_imagej=True)


def create_probability_memmap(
    shape: Tuple[int, int, int, int, int],
    out_dir: Path,
    dtype: np.dtype = np.float16,
) -> Tuple[np.memmap, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    temp_path = out_dir / "probability_hyperstack_temp.dat"
    if temp_path.exists():
        temp_path.unlink()
    return np.memmap(temp_path, mode="w+", dtype=dtype, shape=shape), temp_path



def open_hyperstack_memmap(out_path: Path,
                           shape: Tuple[int, ...],
                           dtype,
                           axes: str) -> np.memmap:
    """Open a hyperstack TIFF for in-place plane writes, creating a
    zero-filled one of ``shape`` if it does not already exist.

    Backs the resumable segmentation path: planes are written straight into
    the final TIFF through the returned memmap, so an interrupted run keeps
    every finished plane on disk and a restart only fills the missing
    timepoints. No ``.dat`` scratch copy is used.

    An existing file whose shape/dtype does not match (an older run with
    different dims, or a non-memmappable ImageJ TIFF) is discarded and
    recreated.
    """
    if tiff is None:
        raise ImportError("tifffile is required.")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    dtype = np.dtype(dtype)
    shape = tuple(int(s) for s in shape)

    if out_path.exists():
        mm = tiff.memmap(str(out_path), mode="r+")
        if tuple(mm.shape) != shape or np.dtype(mm.dtype) != dtype:
            raise RuntimeError(f"Incompatible existing artifact: {out_path}; choose a new run")
        return mm

    return tiff.memmap(str(out_path), shape=shape, dtype=dtype,
                       bigtiff=True, photometric="minisblack", metadata={"axes": axes})


In [ ]:
# ============================================================================
# 8b. Segmentation checkpoint — single source of truth for stage state
# ============================================================================
# `seg_dir/_checkpoint.json` records what segmentation has produced and whether
# every artifact a downstream cell needs is still on disk. It supersedes the old
# `_resume/manifest.json` (the per-timepoint `_resume/t{NNN}.pkl` sidecars stay).
#
# The full-run vs resume vs already-complete decision is made by
# `assess_segmentation_state()` BEFORE the model is loaded and BEFORE the focus
# warm-up, so a finished (or half-finished) run costs nothing to detect.
#
#   status "in_progress"  some timepoints done, run not finalised
#   status "complete"     all timepoints done, seg_index + roi_table written,
#                         every artifact fingerprinted
#
# A "complete" checkpoint whose artifacts no longer verify -> mode "inconsistent"
# -> hard fail with recovery text (no implicit GPU work).

_SEG_DTYPES = {"class": "uint8", "label": "uint8",
               "nucleus": "uint16", "droplet": "uint16", "prob": "float16"}


@dataclass
class SegmentationState:
    mode: str                       # "complete" | "resume" | "fresh" | "inconsistent"
    completed_t: List[int]
    todo_t: List[int]
    checkpoint: Optional[dict]
    missing_artifacts: List[str]
    reason: str


def _seg_artifact_targets(config, save_probability_tiff: bool) -> Dict[str, tuple]:
    """artifact key -> ('file', Path) | ('npy_dir', Path). The set of files a
    completed segmentation must leave on disk for the downstream cells."""
    t = {
        "class_hyperstack":            ("file", config.segmentation_class_hyperstack_path),
        "label_hyperstack":            ("file", config.segmentation_label_hyperstack_path),
        "nucleus_instance_hyperstack": ("file", config.nucleus_instance_hyperstack_path),
        "droplet_instance_hyperstack": ("file", config.droplet_instance_hyperstack_path),
        "seg_index":                   ("file", config.segmentation_index_path),
        "roi_table":                   ("file", config.segmentation_roi_table_path),
        "npy_planes":                  ("npy_dir", config.seg_dir),
    }
    t["full_heads"] = ("file", full_head_path(config))
    t["full_heads_metadata"] = ("file", full_head_metadata_path(config))
    if save_probability_tiff:
        t["probability_hyperstack"] = ("file", config.segmentation_probability_hyperstack_path)
    return t


def _verify_seg_artifacts(cp: dict, needs=None) -> List[str]:
    """Return the artifact keys that are missing or changed since the checkpoint
    was written. `needs` limits the check to a subset of keys."""
    arts = cp.get("artifacts", {})
    keys = list(needs) if needs else list(arts.keys())
    missing: List[str] = []
    for k in keys:
        ent = arts.get(k)
        if not ent:
            missing.append(k); continue
        if "dir" in ent:                                   # npy plane directory
            d = Path(ent["dir"])
            n = len(list(d.glob("nuclear_mask_t*_z*.npy"))) if d.is_dir() else 0
            if n < int(ent.get("count", 0)):
                missing.append(k)
        else:                                              # single file
            p = Path(ent["path"])
            if not p.exists():
                missing.append(k)
            elif ent.get("bytes") is not None and p.stat().st_size != ent["bytes"]:
                missing.append(k)
    return missing


def _mask_stores_ok(config, T: int, Z: int, Y: int, X: int) -> tuple:
    """Are the 4 mask hyperstacks present and memmappable at the expected shape?
    A resume cannot write into a deleted store."""
    checks = [
        (full_head_path(config), (T, Z, 8, Y, X)),
        (config.segmentation_probability_hyperstack_path, (T, Z, 4, Y, X)),
        (config.segmentation_class_hyperstack_path, (T, Z, config.num_classes, Y, X)),
        (config.segmentation_label_hyperstack_path, (T, Z, Y, X)),
        (config.nucleus_instance_hyperstack_path,   (T, Z, Y, X)),
        (config.droplet_instance_hyperstack_path,   (T, Z, Y, X)),
    ]
    for p, want in checks:
        p = Path(p)
        if not p.exists():
            return False, f"{p.name} missing"
        try:
            mm = tiff.memmap(str(p), mode="r")
            shp = tuple(mm.shape); del mm
        except Exception as exc:
            return False, f"{p.name} not memmappable ({exc})"
        if shp != want:
            return False, f"{p.name} shape {shp} != {want}"
    return True, "ok"


def _recovery_hint(config) -> str:
    prob = Path(config.segmentation_probability_hyperstack_path)
    if prob.exists():
        return ("Recovery: a probability hyperstack is present — set "
                "cfg.probability_hyperstack_input to it and re-run §27 to rebuild the "
                "masks with no GPU; or run reset_segmentation(cfg, scope='all') and "
                "re-run §27 for a full rebuild.")
    return ("Recovery: run reset_segmentation(cfg, scope='all') in the §27 reset cell, "
            "then re-run §27 — no probability hyperstack remains, so this re-runs full "
            "GPU inference for every timepoint.")


def _inconsistent_reason(cp: dict, missing: List[str], config) -> str:
    lines = ["Segmentation checkpoint says status=complete, but these artifacts are "
             "missing or changed on disk:"]
    for k in missing:
        ent = cp.get("artifacts", {}).get(k, {})
        lines.append(f"    - {k}: {ent.get('path', ent.get('dir', '?'))}")
    lines.append(f"  checkpoint finalised: {cp.get('finalized_at', '?')}")
    lines.append(_recovery_hint(config))
    return "\n".join(lines)


def assess_segmentation_state(config, dims_tzyx: tuple,
                              save_probability_tiff: bool = True) -> SegmentationState:
    """Decide fresh / resume / complete / inconsistent from on-disk state only.
    No image load beyond the dims tuple, no TF, no GPU, no focus scoring."""
    T, Z, Y, X = (int(v) for v in dims_tzyx)
    all_t = list(range(T))
    cp_path = config.seg_dir / "_checkpoint.json"
    resume_dir = config.seg_dir / "_resume"
    legacy_manifest = resume_dir / "manifest.json"

    cp = None
    if cp_path.exists():
        try:
            cp = json.loads(cp_path.read_text())
        except Exception:
            cp = None

    if cp is None and legacy_manifest.exists():          # migrate old resume state
        try:
            _m = json.loads(legacy_manifest.read_text())
            cp = {"schema": 1, "stage": "segmentation", "status": "in_progress",
                  "dims": {"T": T, "Z": Z, "Y": Y, "X": X},
                  "completed_t": sorted(int(t) for t in _m.get("completed_t", [])),
                  "artifacts": {}, "note": "migrated from legacy _resume/manifest.json"}
        except Exception:
            cp = None

    if cp is None:
        return SegmentationState("fresh", [], all_t, None, [],
            f"No segmentation checkpoint ({cp_path.name}) — fresh run, "
            f"all {T} timepoints to segment.")

    # v19: compare only the fields that determine the segmentation, so a v18.x or v19 segmentation
    # can be reused by a run whose analysis settings differ. Legacy options must have been off.
    _sig_diffs = segmentation_signature_diff(cp.get("config_signature"), config)
    if _sig_diffs:
        return SegmentationState("inconsistent", [], [], cp, list(_sig_diffs),
            "Segmentation checkpoint was made with different segmentation settings:\n" +
            "\n".join(f"    {k}: checkpoint={a!r} live={b!r}" for k, (a, b) in _sig_diffs.items()))
    want = {"T": T, "Z": Z, "Y": Y, "X": X}
    cp_dims = cp.get("dims", {})
    if any(cp_dims.get(k) != v for k, v in want.items()):
        return SegmentationState("fresh", [], all_t, cp, [],
            f"Checkpoint dims {cp_dims} != current {want} — stale checkpoint ignored, "
            f"re-running all {T} timepoints.")

    completed = sorted(int(t) for t in cp.get("completed_t", []) if 0 <= int(t) < T)
    status = cp.get("status")

    if status == "complete":
        missing = _verify_seg_artifacts(cp)
        if not missing:
            return SegmentationState("complete", all_t, [], cp, [],
                f"Segmentation checkpoint complete and all {len(cp.get('artifacts', {}))} "
                f"artifacts verified (finalised {cp.get('finalized_at', '?')}). "
                f"Loading saved tables — no model load, no focus scoring.")
        return SegmentationState("inconsistent", completed, [], cp, missing,
            _inconsistent_reason(cp, missing, config))

    # status == "in_progress" (or a partial checkpoint with completed_t)
    ok, msg = _mask_stores_ok(config, T, Z, Y, X)
    if completed and not ok:
        return SegmentationState("inconsistent", completed, [], cp, ["mask_hyperstacks"],
            f"Checkpoint says timepoints {completed} were segmented, but the mask "
            f"hyperstacks are gone or the wrong shape ({msg}). Cannot resume into a "
            f"deleted store.\n" + _recovery_hint(config))

    survived = [t for t in completed if (resume_dir / f"t{t:03d}.pkl").exists()]
    lost = sorted(set(completed) - set(survived))
    todo = [t for t in all_t if t not in survived]

    if not todo:
        return SegmentationState("resume", survived, [], cp, [],
            f"Checkpoint lists all {T} timepoints done but was never finalised "
            f"(status={status!r}). Re-running finalisation only.")

    reason = (f"Resuming segmentation: {len(survived)}/{T} timepoints already done "
              f"(t={survived}); {len(todo)} to segment (t={todo}).")
    if lost:
        reason += f"\n  sidecars missing for t={lost} — those will be re-segmented."
    return SegmentationState("resume", survived, todo, cp, [], reason)


def write_segmentation_checkpoint(config, *, status: str, dims_tzyx: tuple,
                                  dtypes: dict, save_probability_tiff: bool,
                                  completed_t) -> dict:
    """Write `seg_dir/_checkpoint.json` atomically, fingerprinting every artifact
    that exists right now (st_size for files, plane count for the npy dir)."""
    T, Z, Y, X = (int(v) for v in dims_tzyx)
    cp_path = config.seg_dir / "_checkpoint.json"
    prev = {}
    if cp_path.exists():
        try:
            prev = json.loads(cp_path.read_text())
        except Exception:
            prev = {}

    artifacts = {}
    for key, (kind, p) in _seg_artifact_targets(config, save_probability_tiff).items():
        p = Path(p)
        if kind == "npy_dir":
            artifacts[key] = {"dir": str(p),
                              "count": len(list(p.glob("nuclear_mask_t*_z*.npy")))}
        else:
            artifacts[key] = {"path": str(p),
                              "bytes": p.stat().st_size if p.exists() else None}

    now = datetime.now().isoformat(timespec="seconds")
    cp = {
        "schema": 1,
        "stage": "segmentation",
        "run_id": getattr(getattr(config, "paths", None), "run_id", None),
        "code_version": config.pipeline_revision,
        "config_signature": config.to_run_signature(),
        "model_sha256": config.model_sha256,
        "status": status,
        "dims": {"T": T, "Z": Z, "Y": Y, "X": X},
        "dtypes": dtypes,
        "save_probability_tiff": bool(save_probability_tiff),
        "completed_t": sorted(int(t) for t in completed_t),
        "artifacts": artifacts,
        "started_at": prev.get("started_at") or now,
        "updated_at": now,
    }
    if status == "complete":
        cp["finalized_at"] = now

    tmp = cp_path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(cp, indent=2, default=str))
    os.replace(tmp, cp_path)
    return cp


def require_segmentation(config, needs=()) -> dict:
    """Guard for downstream cells. Raise unless the segmentation checkpoint is
    'complete' and every artifact key in `needs` still verifies on disk."""
    cp_path = config.seg_dir / "_checkpoint.json"
    if not cp_path.exists():
        raise RuntimeError(
            f"{cp_path} not found — segmentation (§27) has not completed in this run. "
            f"Run §27 first.")
    cp = json.loads(cp_path.read_text())
    if cp.get("status") != "complete":
        raise RuntimeError(
            f"Segmentation checkpoint status={cp.get('status')!r}, not 'complete'. "
            f"Re-run §27 to finish it.")
    if cp.get("config_signature") != json.loads(json.dumps(config.to_run_signature())):
        raise RuntimeError("Configuration changed after segmentation; create a new run")
    missing = _verify_seg_artifacts(cp, needs=needs or None)
    if missing:
        raise RuntimeError(_inconsistent_reason(cp, missing, config))
    return cp


def reset_segmentation(config, scope: str) -> None:
    """Wipe segmentation state so the next §27 run rebuilds.

    scope="resume"  delete only the checkpoint + _resume/ (keep the TIFFs).
                    Forces a full recompute; use after editing a seg function.
    scope="all"     also delete the 5 hyperstack TIFFs, the per-plane .npy
                    masks, the seg index / ROI pickles, AND the stale downstream
                    object pickles. Nothing half-deleted.
    """
    if scope not in ("resume", "all"):
        raise ValueError("scope must be 'resume' or 'all'")
    if config.segmentation_source_run:
        raise RuntimeError("Segmentation is read from a source run; it is never reset from here.")
    removed: List[str] = []

    def _rm(p):
        p = Path(p)
        if p.is_dir():
            shutil.rmtree(p); removed.append(str(p) + "/")
        elif p.exists():
            p.unlink(); removed.append(str(p))

    _rm(config.seg_dir / "_checkpoint.json")
    _rm(config.seg_dir / "_checkpoint.json.tmp")
    _rm(config.seg_dir / "_resume")
    _rm(config.seg_dir / "segmentation_checkpoint.json")          # legacy

    if scope == "all":
        for p in (full_head_path(config), full_head_metadata_path(config), config.segmentation_class_hyperstack_path,
                  config.segmentation_label_hyperstack_path,
                  config.nucleus_instance_hyperstack_path,
                  config.droplet_instance_hyperstack_path,
                  config.segmentation_probability_hyperstack_path,
                  config.segmentation_index_path,
                  config.segmentation_roi_table_path):
            _rm(p)
        for name in ("class_mask_5d.dat", "class_label_4d.dat",
                     "nucleus_instance_4d.dat", "droplet_instance_4d.dat"):
            _rm(config.mask_tif_dir / name)
        _npy = list(config.seg_dir.glob("nuclear_mask_t*_z*.npy"))
        for p in _npy:
            p.unlink()
        if _npy:
            removed.append(f"{config.seg_dir}/nuclear_mask_t*_z*.npy  ({len(_npy)} files)")
        for name in ("plane_objects.pkl", "sections_all.pkl", "nuclei_selected.pkl"):
            _rm(config.obj_dir / name)

    print(f"reset_segmentation(scope={scope!r}) removed {len(removed)} path(s):")
    for r in removed:
        print("   ", r)
    if not removed:
        print("    (nothing to remove)")
    if scope == "all":
        print("Re-run §27 to rebuild. Stale downstream object pickles were also cleared.")


In [ ]:
def regionprops_to_rows(
    labeled_mask: np.ndarray,
    t_idx: int,
    z_idx: int,
    class_id: int,
    class_name: str,
    min_area_px: int = 0,
    extra_by_label: Optional[Dict[int, Dict]] = None,
) -> List[Dict]:
    """
    Convert a labelled mask to a list of row dicts.

    Uses regionprops_table for a single vectorised C-level pass over all
    objects.  Perimeter is NOT computed here — it requires binary_erosion
    on every object which is catastrophically slow when the model produces
    large merged regions (e.g. droplet lattice at mid-stack Z planes).
    Circularity filtering happens earlier in clean_nucleus_mask; by the time
    we reach this function it is not needed again.

    extra_by_label (v16.2) merges per-object measurements keyed by label. Used
    to carry the Stage-2 gate metrics into the ROI table so a run can be audited
    — and thresholds re-tuned — without re-running segmentation.
    """
    if labeled_mask.max() == 0:
        return []

    tbl = measure.regionprops_table(
        labeled_mask,
        properties=("label", "area", "centroid", "bbox"),
    )

    labels   = tbl["label"]
    areas    = tbl["area"]
    cy_arr   = tbl["centroid-0"]
    cx_arr   = tbl["centroid-1"]
    rmin     = tbl["bbox-0"]
    cmin     = tbl["bbox-1"]
    rmax     = tbl["bbox-2"]
    cmax     = tbl["bbox-3"]

    rows = []
    for i in range(len(labels)):
        if areas[i] < min_area_px:
            continue
        row = {
            "t": t_idx, "z": z_idx,
            "class_id": class_id, "class_name": class_name,
            "label":          int(labels[i]),
            "centroid_x_px":  float(cx_arr[i]),
            "centroid_y_px":  float(cy_arr[i]),
            "area_px":        int(areas[i]),
            "bbox_min_row":   int(rmin[i]),
            "bbox_min_col":   int(cmin[i]),
            "bbox_max_row":   int(rmax[i]),
            "bbox_max_col":   int(cmax[i]),
        }
        if extra_by_label:
            row.update(extra_by_label.get(int(labels[i]), {}))
        rows.append(row)
    return rows


In [ ]:
# Interactive timing only: these displays do not change the run signature.
from contextlib import contextmanager
import threading
from tqdm.auto import tqdm

@contextmanager
def segmentation_timer(description):
    """Show elapsed time even during a long inference or filesystem operation."""
    stop = threading.Event()
    bar = tqdm(total=None, desc=description, position=2, leave=False,
               bar_format='{desc} | elapsed {elapsed}')
    def refresh():
        while not stop.wait(1.0):
            bar.refresh()
    worker = threading.Thread(target=refresh, daemon=True)
    worker.start()
    started = time.perf_counter()
    try:
        yield
    finally:
        stop.set()
        worker.join(timeout=2)
        bar.close()
        tqdm.write(f'{description}: {time.perf_counter() - started:.1f}s')

def run_segmentation_for_all_planes(img_5d, model, config, state=None,
                                    save_binary_masks=True, save_probability_tiff=True):
    """All eligible planes; fail-fast; timepoint resume; eight preserved heads."""
    if not save_binary_masks or not save_probability_tiff:
        raise ValueError('The reproducible baseline requires masks and probabilities')
    config.paths.assert_config_matches(config.to_run_signature())
    if file_sha256(config.model_path) != config.model_sha256:
        raise RuntimeError('Model changed since bootstrap; create a new run')
    T, Z, C, H, W = img_5d.shape
    if C != 3:
        raise ValueError('Expected microscopy axes TZCYX with three channels')
    zs = eligible_z_planes(Z, config)
    if not zs:
        raise ValueError('No eligible target planes')
    done = set(state.completed_t if state is not None else [])
    if len(done) < T:
        validate_vulcan_model(model, config)
    stores = [
        open_hyperstack_memmap(config.segmentation_class_hyperstack_path,(T,Z,4,H,W),np.uint8,'TZCYX'),
        open_hyperstack_memmap(config.segmentation_label_hyperstack_path,(T,Z,H,W),np.uint8,'TZYX'),
        open_hyperstack_memmap(config.nucleus_instance_hyperstack_path,(T,Z,H,W),np.uint16,'TZYX'),
        open_hyperstack_memmap(config.droplet_instance_hyperstack_path,(T,Z,H,W),np.uint16,'TZYX'),
        open_hyperstack_memmap(config.segmentation_probability_hyperstack_path,(T,Z,4,H,W),np.float16,'TZCYX'),
        open_hyperstack_memmap(full_head_path(config),(T,Z,8,H,W),np.float16,'TZCYX')]
    classes, display_map, nuclei, droplets, canonical, all_heads = stores
    meta = dict(channels=list(VULCAN_HEADS), axes='TZCYX', z_offset_units='um',
                included_z=zs, model_sha256=config.model_sha256, config=config.to_run_signature())
    full_head_metadata_path(config).write_text(json.dumps(meta,indent=2),encoding='utf8')
    records, rows = [], []
    resume = config.seg_dir / '_resume'
    resume.mkdir(parents=True,exist_ok=True)
    pending_t = [t for t in range(T) if t not in done]
    print(f'Resume: {len(done)}/{T} timepoints already saved; remaining {pending_t}', flush=True)
    print('Plane ETA covers remaining inference; timepoint ETA also includes normalization and writes.', flush=True)
    plane_progress = tqdm(total=len(pending_t) * len(zs), desc='Remaining planes',
                          unit='plane', position=0, leave=True)
    timepoint_progress = tqdm(total=T, initial=len(done), desc='Checkpointed timepoints',
                              unit='timepoint', position=1, leave=True)
    try:
        for t in range(T):
            sidecar = resume / ('t%03d.pkl' % t)
            if t in done:
                saved = pd.read_pickle(sidecar)
                records.extend(saved['records']); rows.extend(saved['roi_rows'])
                continue
            tr, rr = [], []
            timepoint_started = time.perf_counter()
            with segmentation_timer(f'T{t}: normalization'):
                stats = normalization_stats_for_timepoint(img_5d,t,config)
            (resume / ('normalization_t%03d.json' % t)).write_text(json.dumps(stats),encoding='utf8')
            # Clear any unfinished timepoint so retry cannot retain stale planes.
            with segmentation_timer(f'T{t}: clear unfinished output'):
                for mm in stores:
                    mm[t] = 0
            for z in range(Z):
                path = config.seg_dir / ('nuclear_mask_t%03d_z%03d.npy' % (t,z))
                if z not in zs:
                    np.save(path,np.zeros((H,W),np.uint8))
                    tr.append(dict(t=t,z=z,included=False,reason='outside_z_range',mask_path=str(path)))
                    continue
                plane_started = time.perf_counter()
                plane_progress.set_description(f'Remaining planes — T{t} Z{z}')
                try:
                    with segmentation_timer(f'T{t} Z{z}: inference and instance extraction'):
                        heads = predict_vulcan_plane(img_5d,t,z,model,config,stats)
                        lab, drop = vulcan_instances(heads,np.asarray(img_5d[t,z,config.nuclear_channel_index],np.float32),
                            config,img_5d[t,z,config.npc_channel_index],img_5d[t,z,config.membrane_channel_index],t)
                except Exception as exc:
                    (config.seg_dir/'last_error.json').write_text(json.dumps(dict(t=t,z=z,error=repr(exc))),encoding='utf8')
                    raise RuntimeError('Segmentation failed at t=%d z=%d; timepoint not committed' % (t,z)) from exc
                all_heads[t,z] = np.moveaxis(heads,-1,0).astype(np.float16)
                canonical[t,z] = np.moveaxis(heads[..., [0,1,3,4]],-1,0).astype(np.float16)
                dm, nm, npc = drop>0, lab>0, heads[...,3]>config.npc_threshold
                classes[t,z] = np.stack([~dm,dm,npc,nm]).astype(np.uint8)
                display_map[t,z] = collapse_label_map(dm,npc,nm,config)
                nuclei[t,z], droplets[t,z] = lab,drop
                extra = {}
                for r in measure.regionprops(lab):
                    m = lab[r.slice] == r.label
                    extra[r.label] = dict(mean_p=float(heads[r.slice][...,4][m].mean()),
                        mean_z_offset=float(heads[r.slice][...,7][m].mean()),
                        mean_equatorial=float(heads[r.slice][...,6][m].mean()))
                rr.extend(regionprops_to_rows(lab,t,z,3,'nucleus',extra_by_label=extra))
                rr.extend(regionprops_to_rows(drop,t,z,1,'droplet'))
                np.save(path,nm.astype(np.uint8))
                tr.append(dict(t=t,z=z,included=True,reason='segmented',mask_path=str(path),
                    nucleus_pixels=int(nm.sum()),droplet_pixels=int(dm.sum())))
                plane_progress.update(1)
                tqdm.write('t=%d z=%d: %d nuclei | %.1fs' % (t,z,len(extra),time.perf_counter()-plane_started))
            with segmentation_timer(f'T{t}: flush and save checkpoint'):
                for mm in stores: mm.flush()
                tmp = sidecar.with_suffix('.tmp')
                pd.to_pickle(dict(records=tr,roi_rows=rr),tmp); os.replace(tmp,sidecar)
                records.extend(tr); rows.extend(rr); done.add(t)
                write_segmentation_checkpoint(config,status='in_progress',dims_tzyx=(T,Z,H,W),
                    dtypes=_SEG_DTYPES,save_probability_tiff=True,completed_t=done)
            timepoint_progress.update(1)
            tqdm.write(f'Timepoint {t} committed in {(time.perf_counter()-timepoint_started)/60:.1f} min')
        with segmentation_timer('Finalize segmentation tables and checkpoint'):
            index = pd.DataFrame(records).sort_values(['t','z']).reset_index(drop=True)
            if len(index)!=T*Z or index.duplicated(['t','z']).any():
                raise RuntimeError('Incomplete or duplicated plane index')
            index.to_pickle(config.segmentation_index_path)
            pd.DataFrame(rows,columns=None if rows else
                ['t','z','label','class_name','class_id','area_px','centroid_x_px','centroid_y_px',
                 'mean_p','mean_z_offset','mean_equatorial']).to_pickle(config.segmentation_roi_table_path)
            write_segmentation_checkpoint(config,status='complete',dims_tzyx=(T,Z,H,W),
                dtypes=_SEG_DTYPES,save_probability_tiff=True,completed_t=done)
        return index
    finally:
        plane_progress.close()
        timepoint_progress.close()
        for mm in stores: mm.flush()


In [ ]:
def extract_objects_from_saved_masks(seg_index_df, config):
    """Use measured instance IDs, never relabel a binary watershed mask."""
    require_segmentation(config)
    roi = pd.read_pickle(config.segmentation_roi_table_path)
    result = roi[roi.class_name.eq('nucleus')].copy()
    result['nucleus_area_um2'] = result.area_px * config.pixel_size_um**2
    result.to_pickle(config.obj_dir/'plane_objects.pkl')
    return result


## 8. Single-nucleus selection (definitions)

In [ ]:
# §8 SINGLE-NUCLEUS SELECTION (definitions). Validated against the hand-traced gold ROIs on
# 2026-11-04 (median logistic A(30)/A(55) within the ±0.5 µm-diameter margin). The order of the
# rules matters for parity with that validation: edge -> local area max -> floor -> clusters.
def touches_edge_flags(df: pd.DataFrame, H: int, W: int, edge_px: int) -> np.ndarray:
    """Bounding box within edge_px of the frame border (bbox max is exclusive)."""
    return ((df.bbox_min_row.to_numpy() <= edge_px) | (df.bbox_min_col.to_numpy() <= edge_px)
            | (df.bbox_max_row.to_numpy() >= H - edge_px) | (df.bbox_max_col.to_numpy() >= W - edge_px))


def local_area_max(df: pd.DataFrame, window_planes: int) -> Tuple[np.ndarray, np.ndarray]:
    """Keep a section unless a section of the same timepoint within ±window_planes planes, whose
    centroid is closer than the larger of the two equivalent radii, has a strictly larger area.
    No chaining across planes, so a droplet moving between planes cannot merge or split nuclei
    beyond this local window. Returns (kept, suppressed_by) aligned with df rows;
    suppressed_by is the df index of the largest section that beats a dropped one (-1 if kept)."""
    kept = np.ones(len(df), bool)
    by = np.full(len(df), -1, dtype=np.int64)
    pos = {ix: k for k, ix in enumerate(df.index)}
    for _, g in df.groupby("t", sort=False):
        z = g.z.to_numpy(); cy = g.centroid_y_px.to_numpy(); cx = g.centroid_x_px.to_numpy()
        area = g.area_px.to_numpy(float); r = np.sqrt(area / np.pi)
        for a in range(len(g)):
            near = (np.abs(z - z[a]) <= window_planes) & (np.hypot(cy - cy[a], cx - cx[a]) < np.maximum(r, r[a]))
            near[a] = False
            beat = near & (area > area[a])
            if beat.any():
                k = pos[g.index[a]]
                kept[k] = False
                by[k] = g.index[np.flatnonzero(beat)[np.argmax(area[beat])]]
    return kept, by


def cluster_flags(df: pd.DataFrame, instance_stack, touch_px: int) -> pd.DataFrame:
    """Solidity of each section's own mask, and whether ANY other nuclear instance on the same
    plane lies within touch_px of it. Reads the saved instance hyperstack (TZYX)."""
    H, W = instance_stack.shape[-2:]
    disk = morphology.disk(touch_px)
    sol, touch = [], []
    for r in df.itertuples():
        a, b, c, d = int(r.bbox_min_row), int(r.bbox_min_col), int(r.bbox_max_row), int(r.bbox_max_col)
        A, B, C, D = max(0, a - touch_px), max(0, b - touch_px), min(H, c + touch_px), min(W, d + touch_px)
        lab = np.asarray(instance_stack[int(r.t), int(r.z), A:C, B:D])
        me = lab == int(r.label)
        if not me.any():
            raise ValueError(f"t{r.t} z{r.z} label {r.label}: instance missing from the saved masks")
        sol.append(measure.regionprops(me.astype(np.uint8))[0].solidity)
        ring = ndi.binary_dilation(me, structure=disk) & ~me
        touch.append(bool(((lab[ring] > 0) & (lab[ring] != int(r.label))).any()))
    return pd.DataFrame({"solidity": sol, "touching": touch}, index=df.index)


def select_single_nuclei(objects_df: pd.DataFrame, components: pd.DataFrame, H: int, W: int,
                         config: PipelineConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Apply the v19 rules. Returns (sections, nuclei):
    sections — every candidate section with one boolean column per rule and the final `reason`;
    nuclei   — the selected single nuclei (one row per nucleus per timepoint), with nucleus_id."""
    keys = ["t", "z", "label"]
    d = objects_df.merge(components[keys + ["status", "parent"]], on=keys, how="left", validate="one_to_one")
    d = d[(d.t >= config.sel_min_timepoint) & (d.z >= config.focus_min_z)].copy()
    d["area_um2"] = d.area_px * config.pixel_size_um ** 2
    d["r_population"] = d.status.eq("accepted")
    d["r_edge"] = touches_edge_flags(d, H, W, config.sel_edge_px)

    cand = d[d.r_population & ~d.r_edge]
    kept, by = local_area_max(cand, config.sel_window_planes)
    d["r_local_max"] = False
    d.loc[cand.index, "r_local_max"] = kept
    d["suppressed_by"] = -1
    d.loc[cand.index, "suppressed_by"] = by

    d["r_area"] = d.area_um2 >= config.sel_min_area_um2
    stage = d[d.r_population & ~d.r_edge & d.r_local_max & d.r_area]
    inst = tiff.memmap(str(config.nucleus_instance_hyperstack_path), mode="r")
    cl = cluster_flags(stage, inst, config.sel_touch_px)
    d["solidity"] = np.nan; d["touching"] = False
    d.loc[cl.index, "solidity"] = cl.solidity; d.loc[cl.index, "touching"] = cl.touching
    d["r_single"] = (d.solidity >= config.sel_min_solidity) & ~d.touching

    d["selected"] = d.r_population & ~d.r_edge & d.r_local_max & d.r_area & d.r_single
    d["reason"] = np.select(
        [~d.r_population, d.r_edge, ~d.r_local_max, ~d.r_area,
         d.solidity < config.sel_min_solidity, d.touching],
        ["population:" + d.status.fillna("no decision").astype(str), "edge", "not local max",
         "below area floor", "cluster: bilobed", "cluster: touching"], "selected")

    nuclei = d[d.selected].sort_values(["t", "centroid_y_px", "centroid_x_px"]).copy()
    nuclei["nucleus_id"] = nuclei.groupby("t").cumcount() + 1
    nuclei["nucleus_area_um2"] = nuclei.area_um2
    nuclei = nuclei.drop(columns=[c for c in nuclei.columns if c.startswith("r_")] + ["selected", "reason"])
    return d, nuclei.reset_index(drop=True)


def selection_rule_counts(sections: pd.DataFrame) -> pd.DataFrame:
    """Per timepoint: candidates, how many each rule removed (first failing rule), and kept."""
    tab = pd.crosstab(sections.t, sections.reason)
    order = [c for c in tab.columns if c != "selected"] + (["selected"] if "selected" in tab else [])
    tab = tab[order]
    tab.insert(0, "candidates", tab.sum(axis=1))
    return tab


## 9. Timing, tracking, halos, radial sweep, export helpers (definitions)

In [ ]:
# §9 TILE TIMING (definitions). True acquisition time of a nucleus = offset + t·frame + tile·minutes.
def assign_tile_index_from_xy(x_px: float, y_px: float, image_width_px: int, image_height_px: int,
                              config: PipelineConfig) -> Tuple[int, int, int]:
    col = min(int(x_px // (image_width_px / config.tile_cols)), config.tile_cols - 1)
    row = min(int(y_px // (image_height_px / config.tile_rows)), config.tile_rows - 1)
    if config.serpentine_scan and row % 2 != 0:
        tile_index = row * config.tile_cols + (config.tile_cols - 1 - col)
    else:
        tile_index = row * config.tile_cols + col
    return row, col, tile_index


def add_tile_timing_metadata(nuclei_df: pd.DataFrame, image_hw: Tuple[int, int],
                             config: PipelineConfig, out_path: Optional[Path] = None) -> pd.DataFrame:
    if nuclei_df.empty:
        return nuclei_df.copy()
    H, W = image_hw
    tiles = [assign_tile_index_from_xy(x, y, W, H, config)
             for x, y in zip(nuclei_df.centroid_x_px, nuclei_df.centroid_y_px)]
    out = nuclei_df.copy()
    out["tile_row"], out["tile_col"], out["tile_index"] = map(list, zip(*tiles))
    out["tile_offset_min"] = out.tile_index * config.minutes_per_tile
    out["true_time_min"] = (config.time_zero_offset_min + out.t * config.frame_interval_min
                            + out.tile_offset_min)
    if out_path is not None:
        out.to_pickle(out_path)
    return out


In [ ]:
def _safe_area_ratio(area_a: float, area_b: float) -> float:
    a, b = max(float(area_a), 1.0), max(float(area_b), 1.0)
    return max(a, b) / min(a, b)


def build_dbscan_candidate_clusters(
    prev_df: pd.DataFrame,
    curr_df: pd.DataFrame,
    config: PipelineConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if DBSCAN is None:
        raise ImportError("scikit-learn is required for DBSCAN-assisted tracking.")

    prev_local = prev_df.copy().reset_index(drop=True)
    curr_local = curr_df.copy().reset_index(drop=True)
    prev_local["frame_role"] = "prev"
    curr_local["frame_role"] = "curr"
    prev_local["frame_local_index"] = np.arange(len(prev_local))
    curr_local["frame_local_index"] = np.arange(len(curr_local))

    pooled = pd.concat([prev_local, curr_local], ignore_index=True)
    if pooled.empty:
        for col in ("dbscan_cluster_id", "dbscan_cluster_size", "dbscan_is_crowded"):
            pooled[col] = pd.Series(dtype=int if col != "dbscan_is_crowded" else bool)
        return prev_local, curr_local, pooled

    coords = pooled[["centroid_x_px", "centroid_y_px"]].to_numpy()
    labels = DBSCAN(eps=config.track_dbscan_eps_px,
                    min_samples=config.track_dbscan_min_samples).fit_predict(coords)
    pooled["dbscan_cluster_id"] = labels

    crowded_flags = []
    cluster_sizes = []
    for _, df_c in pooled.groupby("dbscan_cluster_id", sort=False):
        n_prev = int((df_c["frame_role"] == "prev").sum())
        n_curr = int((df_c["frame_role"] == "curr").sum())
        is_crowded = n_prev > 1 or n_curr > 1
        crowded_flags.extend([is_crowded] * len(df_c))
        cluster_sizes.extend([len(df_c)] * len(df_c))
    pooled["dbscan_cluster_size"] = cluster_sizes
    pooled["dbscan_is_crowded"]   = crowded_flags

    prev_out = pooled[pooled["frame_role"] == "prev"].copy().reset_index(drop=True)
    curr_out = pooled[pooled["frame_role"] == "curr"].copy().reset_index(drop=True)
    return prev_out, curr_out, pooled


def match_cluster_with_assignment(
    prev_cluster: pd.DataFrame,
    curr_cluster: pd.DataFrame,
    config: PipelineConfig,
) -> List[dict]:
    if prev_cluster.empty or curr_cluster.empty:
        return []

    n_prev, n_curr = len(prev_cluster), len(curr_cluster)
    cost_matrix = np.full((n_prev, n_curr), np.nan, dtype=float)
    max_dist_base = float(config.time_track_tolerance_px)

    for i, prev_row in prev_cluster.iterrows():
        for j, curr_row in curr_cluster.iterrows():
            dx  = float(curr_row["centroid_x_px"]) - float(prev_row["centroid_x_px"])
            dy  = float(curr_row["centroid_y_px"]) - float(prev_row["centroid_y_px"])
            dist_px = math.hypot(dx, dy)

            max_dist = max_dist_base
            if prev_row.get("dbscan_is_crowded", False) or curr_row.get("dbscan_is_crowded", False):
                max_dist *= config.track_crowded_distance_scale

            prev_area = max(float(prev_row["area_px"]), 1.0)
            curr_area = max(float(curr_row["area_px"]), 1.0)
            area_ratio = max(prev_area, curr_area) / min(prev_area, curr_area)

            if dist_px <= max_dist and area_ratio <= config.track_area_ratio_max:
                area_cost    = abs(math.log(curr_area / prev_area))
                dist_cost    = dist_px / max(max_dist, 1e-6)
                cost_matrix[i, j] = dist_cost + config.track_area_log_weight * area_cost

    finite_mask = np.isfinite(cost_matrix)
    valid_rows  = np.where(finite_mask.any(axis=1))[0]
    valid_cols  = np.where(finite_mask.any(axis=0))[0]
    if len(valid_rows) == 0 or len(valid_cols) == 0:
        return []

    large_penalty = 1e9
    reduced = np.where(np.isfinite(cost_matrix[np.ix_(valid_rows, valid_cols)]),
                       cost_matrix[np.ix_(valid_rows, valid_cols)], large_penalty)

    bad_rows = np.all(reduced >= large_penalty, axis=1)
    bad_cols = np.all(reduced >= large_penalty, axis=0)
    if bad_rows.any() or bad_cols.any():
        keep_r = np.where(~bad_rows)[0]
        keep_c = np.where(~bad_cols)[0]
        if len(keep_r) == 0 or len(keep_c) == 0:
            return []
        reduced    = reduced   [np.ix_(keep_r, keep_c)]
        valid_rows = valid_rows[keep_r]
        valid_cols = valid_cols[keep_c]

    row_ind, col_ind = linear_sum_assignment(reduced)

    matches = []
    for r, c in zip(row_ind, col_ind):
        orig_i = valid_rows[r]
        orig_j = valid_cols[c]
        link_cost = cost_matrix[orig_i, orig_j]
        if not np.isfinite(link_cost):
            continue
        pr = prev_cluster.iloc[orig_i]
        cr = curr_cluster.iloc[orig_j]
        dist_px = math.hypot(
            float(cr["centroid_x_px"]) - float(pr["centroid_x_px"]),
            float(cr["centroid_y_px"]) - float(pr["centroid_y_px"]),
        )
        matches.append({
            "prev_index":        int(pr["global_index"]),
            "curr_index":        int(cr["global_index"]),
            "link_cost":         float(link_cost),
            "link_distance_px":  float(dist_px),
            "dbscan_cluster_id":   int(pr.get("dbscan_cluster_id", -1)),
            "dbscan_cluster_size": int(pr.get("dbscan_cluster_size", 0)),
            "dbscan_is_crowded":   bool(pr.get("dbscan_is_crowded", False)),
        })
    return matches


def assign_track_ids_hybrid_dbscan(
    timed_df: pd.DataFrame,
    config: PipelineConfig,
    valid_only: bool = False,
) -> pd.DataFrame:
    if timed_df.empty:
        return pd.DataFrame()

    df = timed_df.copy().reset_index(drop=True)
    df["global_index"] = np.arange(len(df), dtype=int)
    df = df.sort_values(["t", "nucleus_id"]).reset_index(drop=True)

    df["track_id"]                = -1
    df["track_link_distance_px"]  = np.nan
    df["track_link_cost"]         = np.nan
    df["track_dbscan_cluster_id"] = -1
    df["track_dbscan_cluster_size"] = 0
    df["track_dbscan_is_crowded"] = False
    df["track_link_method"]       = "unassigned"

    time_values = sorted(df["t"].unique())
    if not time_values:
        return df

    next_track_id = 1
    for idx in df.index[df["t"] == time_values[0]]:
        df.at[idx, "track_id"]          = next_track_id
        df.at[idx, "track_link_method"] = "seed"
        next_track_id += 1

    for t_prev, t_curr in zip(time_values[:-1], time_values[1:]):
        prev_df = df[df["t"] == t_prev].reset_index()
        curr_df = df[df["t"] == t_curr].reset_index()

        prev_clustered, curr_clustered, pooled = build_dbscan_candidate_clusters(
            prev_df, curr_df, config)

        all_matches: List[dict] = []
        for cluster_id in sorted(pooled["dbscan_cluster_id"].unique()):
            pc = prev_clustered[prev_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            cc = curr_clustered[curr_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            all_matches.extend(match_cluster_with_assignment(pc, cc, config))

        matched_curr_global: set = set()

        # FIX: loop body was previously unreachable due to a misindented `continue`.
        for match in sorted(all_matches, key=lambda x: x["link_cost"]):
            prev_global = int(match["prev_index"])
            curr_global = int(match["curr_index"])

            if curr_global in matched_curr_global:
                continue

            track_id = int(df.at[prev_global, "track_id"])
            if track_id == -1:
                track_id = next_track_id
                df.at[prev_global, "track_id"]          = track_id
                df.at[prev_global, "track_link_method"] = "backfilled_seed"
                next_track_id += 1

            matched_curr_global.add(curr_global)
            df.at[curr_global, "track_id"]                  = track_id
            df.at[curr_global, "track_link_distance_px"]    = float(match["link_distance_px"])
            df.at[curr_global, "track_link_cost"]           = float(match["link_cost"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(match["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(match["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(match["dbscan_is_crowded"])
            df.at[curr_global, "track_link_method"]         = "hybrid_dbscan"

        for _, row in curr_clustered.iterrows():
            curr_global = int(row["index"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(row["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(row["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(row["dbscan_is_crowded"])
            if df.at[curr_global, "track_id"] == -1:
                df.at[curr_global, "track_id"]          = next_track_id
                df.at[curr_global, "track_link_method"] = "new_track_unmatched"
                next_track_id += 1

    df.to_pickle(config.track_dir / "tracked_nuclei.pkl")
    return df


def summarize_tracking_debug(tracked_df: pd.DataFrame) -> pd.DataFrame:
    if tracked_df.empty:
        return pd.DataFrame()
    track_len = tracked_df.groupby("track_id").size()
    return pd.DataFrame([{
        "n_rows":                int(len(tracked_df)),
        "n_tracks":              int(tracked_df["track_id"].nunique()),
        "median_track_length":   float(track_len.median()),
        "mean_track_length":     float(track_len.mean()),
        "fraction_crowded_links":float(tracked_df["track_dbscan_is_crowded"].fillna(False).mean()),
        "fraction_new_tracks":   float((tracked_df["track_link_method"] == "new_track_unmatched").mean()),
    }])


In [ ]:
def build_cumulative_halo_masks(nucleus_mask: np.ndarray, config: PipelineConfig) -> Dict[str, np.ndarray]:
    dist_outside = ndi.distance_transform_edt(~nucleus_mask)
    masks = {"nucleus_mask": nucleus_mask.astype(bool)}

    for i in range(1, config.n_halos + 1):
        masks[f"halo_{i}_cum"] = dist_outside <= i * config.halo_step_px

    for i in range(1, config.n_halos + 1):
        prev = nucleus_mask if i == 1 else masks[f"halo_{i-1}_cum"]
        masks[f"ring_{i}"] = masks[f"halo_{i}_cum"] & ~prev

    masks["nucleus_eroded"] = (
        ndi.binary_erosion(nucleus_mask, structure=morphology.disk(config.erosion_px))
        if config.erosion_px > 0 else nucleus_mask.copy()
    )
    masks["cytoplasm_mask"] = masks[f"halo_{config.n_halos}_cum"] & ~nucleus_mask
    return masks


def _mask_stats(intensity_image: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    area_px = int(mask.sum())
    if area_px == 0:
        return {"area_px": 0, "intden": 0.0, "mean_intensity": np.nan}
    vals = intensity_image[mask]
    return {"area_px": area_px, "intden": float(vals.sum()), "mean_intensity": float(vals.mean())}


def measure_fiji_equivalent_halos(
    intensity_image: np.ndarray,
    nucleus_mask: np.ndarray,
    config: PipelineConfig,
) -> Dict[str, float]:
    masks = build_cumulative_halo_masks(nucleus_mask, config)
    out: Dict[str, float] = {}

    for key in ("nucleus_mask", "nucleus_eroded", "cytoplasm_mask"):
        prefix = key.replace("_mask", "").replace("_cum", "")
        for k, v in _mask_stats(intensity_image, masks[key]).items():
            out[f"{prefix}_{k}"] = v

    for i in range(1, config.n_halos + 1):
        for k, v in _mask_stats(intensity_image, masks[f"halo_{i}_cum"]).items():
            out[f"halo_{i}_cum_{k}"] = v
        for k, v in _mask_stats(intensity_image, masks[f"ring_{i}"]).items():
            out[f"ring_{i}_{k}"] = v

    nm = out.get("nucleus_mean_intensity", np.nan)
    cm = out.get("cytoplasm_mean_intensity", np.nan)
    out["nc_ratio"]          = float(nm / cm)          if np.isfinite(nm) and np.isfinite(cm) and cm != 0         else np.nan
    out["nc_ratio_fraction"] = float(nm / (nm + cm))   if np.isfinite(nm) and np.isfinite(cm) and (nm + cm) != 0  else np.nan
    return out


def recover_nucleus_mask_for_row(row, config):
    """Mask of the selected section (t, z, label) from the saved instance hyperstack."""
    lab=tiff.memmap(str(config.nucleus_instance_hyperstack_path),mode='r')[int(row.t),int(row.z)]
    label=int(row.label)
    if not (lab==label).any(): raise ValueError('Instance missing for measured row')
    return lab==label


def _process_single_tracked_nucleus(row, img_5d: np.ndarray, config: PipelineConfig) -> dict:
    nucleus_mask  = recover_nucleus_mask_for_row(row, config)
    nuclear_plane = get_nuclear_plane(img_5d, int(row.t), int(row.z), config)
    halo_metrics  = measure_fiji_equivalent_halos(nuclear_plane, nucleus_mask, config)
    rec = row._asdict()
    rec.update(halo_metrics)
    rec["nucleus_area_um2"]   = rec.get("nucleus_area_px", 0) * config.pixel_size_um ** 2
    rec["cytoplasm_area_um2"] = rec.get("cytoplasm_area_px", 0) * config.pixel_size_um ** 2
    return rec


def run_halo_analysis_for_tracked_nuclei(
    tracked_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    if tracked_df.empty:
        halo_df = pd.DataFrame()
        halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
        return halo_df
    rows = Parallel(n_jobs=max(1, int(config.n_workers)), backend="threading", batch_size=1)(
        delayed(_process_single_tracked_nucleus)(row, img_5d, config)
        for row in tracked_df.itertuples(index=False)
    )
    halo_df = pd.DataFrame(rows)
    halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
    return halo_df


In [ ]:
# =============================================================================
# 19. Radial sweep (droplet-bounded)  [v16]
# =============================================================================
# Reuses v15 artefacts:
#   - recover_nucleus_mask_from_plane(...) for the per-nucleus mask (Section 18)
#   - droplet_instance_hyperstack.tif (TZYX) written by segmentation (Section 27)
# Iterates the SAME tracked_df the halo analysis consumes, so every row already
# carries a persistent track_id == schema Nucleus_ID.

_DROPLET_STACK_CACHE: Dict[str, np.ndarray] = {}


def load_droplet_instance_stack(config: PipelineConfig) -> np.ndarray:
    """Load (and cache) the TZYX droplet instance hyperstack from segmentation."""
    key = str(config.droplet_instance_hyperstack_path)
    if key not in _DROPLET_STACK_CACHE:
        if not config.droplet_instance_hyperstack_path.exists():
            raise FileNotFoundError(
                f"Droplet instance hyperstack not found: "
                f"{config.droplet_instance_hyperstack_path}. Run segmentation "
                f"(Section 27) first, or point cfg at an existing run."
            )
        _DROPLET_STACK_CACHE[key] = tiff.imread(config.droplet_instance_hyperstack_path)
    return _DROPLET_STACK_CACHE[key]


def assign_nucleus_to_droplet(nucleus_mask: np.ndarray,
                              droplet_labeled_2d: np.ndarray) -> int:
    """Droplet label with greatest overlap with this nucleus (0 if no overlap)."""
    overlapping = droplet_labeled_2d[nucleus_mask]
    overlapping = overlapping[overlapping > 0]
    if overlapping.size == 0:
        return 0
    labels, counts = np.unique(overlapping, return_counts=True)
    return int(labels[np.argmax(counts)])


def sample_ray_to_droplet_edge(
    image_2d, droplet_mask, nucleus_mask, center_y, center_x, theta,
    step_size=1.0, exclude_nucleus=True, max_steps=4000,
):
    """Sample one radial ray from the centroid outward until it leaves the droplet.
    All distances in pixels. Nearest-pixel sampling (see Section 19 notes)."""
    ys, xs, vals, dists, inside_flags = [], [], [], [], []
    for step in range(max_steps):
        r = step * step_size
        y = center_y + r * np.sin(theta)
        x = center_x + r * np.cos(theta)
        yi = int(round(y)); xi = int(round(x))
        if yi < 0 or yi >= image_2d.shape[0] or xi < 0 or xi >= image_2d.shape[1]:
            break
        if not droplet_mask[yi, xi]:
            break
        is_nucleus = bool(nucleus_mask[yi, xi])
        if exclude_nucleus and is_nucleus:
            continue
        ys.append(yi); xs.append(xi); vals.append(image_2d[yi, xi])
        dists.append(r); inside_flags.append(is_nucleus)
    return {
        "y": np.asarray(ys, dtype=int),
        "x": np.asarray(xs, dtype=int),
        "values": np.asarray(vals, dtype=float),
        "distance_px": np.asarray(dists, dtype=float),
        "inside_nucleus": np.asarray(inside_flags, dtype=bool),
    }


def radial_profile_for_tracked_nucleus(
    radial_plane, nucleus_mask, droplet_mask, track_id, droplet_id, t, z,
    center_y, center_x, n_angles=180, step_size=1.0,
    exclude_nucleus=True, max_steps=4000,
) -> pd.DataFrame:
    """Long-form radial profile for one nucleus: one row per sampled ray point.
    distance_norm is the per-ray fraction of the distance to the droplet edge
    (schema Rho_Normalized)."""
    rows = []
    thetas = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    for theta_idx, theta in enumerate(thetas):
        ray = sample_ray_to_droplet_edge(
            radial_plane, droplet_mask, nucleus_mask, center_y, center_x, theta,
            step_size=step_size, exclude_nucleus=exclude_nucleus, max_steps=max_steps)
        if ray["distance_px"].size == 0:
            continue
        ray_len = float(ray["distance_px"][-1])
        norm = (np.zeros_like(ray["distance_px"]) if np.isclose(ray_len, 0)
                else ray["distance_px"] / ray_len)
        for i in range(ray["distance_px"].size):
            rows.append({
                "track_id": int(track_id), "t": int(t), "z": int(z),
                "droplet_id": int(droplet_id),
                "theta_index": theta_idx, "theta_rad": float(theta),
                "y": int(ray["y"][i]), "x": int(ray["x"][i]),
                "distance_px": float(ray["distance_px"][i]),
                "distance_norm": float(norm[i]),
                "intensity": float(ray["values"][i]),
                "ray_length_px": ray_len,
                "inside_nucleus": bool(ray["inside_nucleus"][i]),
            })
    return pd.DataFrame(rows)


def _process_single_tracked_nucleus_radial(row, img_5d, droplet_stack,
                                           config: PipelineConfig) -> pd.DataFrame:
    t, z = int(row.t), int(row.z)
    cx, cy = float(row.centroid_x_px), float(row.centroid_y_px)
    nucleus_mask = recover_nucleus_mask_for_row(row, config)
    droplet_labeled_2d = droplet_stack[t, z]
    droplet_id = assign_nucleus_to_droplet(nucleus_mask, droplet_labeled_2d)
    if droplet_id == 0:
        return pd.DataFrame()   # no assigned droplet -> no bounded sweep
    droplet_mask = droplet_labeled_2d == droplet_id
    radial_plane = img_5d[t, z, config.radial_channel_index]
    return radial_profile_for_tracked_nucleus(
        radial_plane, nucleus_mask, droplet_mask,
        track_id=int(row.track_id), droplet_id=droplet_id, t=t, z=z,
        center_y=cy, center_x=cx,
        n_angles=config.radial_n_angles,
        step_size=config.radial_step_size_px,
        exclude_nucleus=config.radial_exclude_nucleus,
        max_steps=config.radial_max_ray_steps)


def run_radial_sweep_for_tracked_nuclei(
    tracked_df: pd.DataFrame, img_5d: np.ndarray, config: PipelineConfig,
) -> pd.DataFrame:
    """Radial sweep over every tracked nucleus at its selected best-Z.
    Returns a long-form DataFrame (one row per ray sample point) and pickles it."""
    out_path = config.analysis_dir / "radial_sweep.pkl"
    if tracked_df.empty:
        radial_df = pd.DataFrame()
        radial_df.to_pickle(out_path)
        return radial_df

    droplet_stack = load_droplet_instance_stack(config)
    assert droplet_stack.shape[0] == img_5d.shape[0], "Droplet stack T mismatch vs image"
    assert droplet_stack.shape[1] == img_5d.shape[1], "Droplet stack Z mismatch vs image"
    assert droplet_stack.shape[-2:] == img_5d.shape[-2:], "Droplet stack Y/X mismatch vs image"

    dfs = Parallel(n_jobs=max(1, int(config.n_workers)), backend="threading", batch_size=1)(
        delayed(_process_single_tracked_nucleus_radial)(row, img_5d, droplet_stack, config)
        for row in tracked_df.itertuples(index=False)
    )
    dfs = [d for d in dfs if not d.empty]
    radial_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    radial_df.to_pickle(out_path)
    return radial_df


In [ ]:
def _save_portable_table(df: pd.DataFrame, out_stem: Path) -> dict:
    """Save a table as CSV (human-readable) + pickle (lossless, fast).
    Parquet removed — requires pyarrow/fastparquet which are not available."""
    out_stem.parent.mkdir(parents=True, exist_ok=True)
    saved: dict = {}
    if df is None:
        return saved
    csv_path = out_stem.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    saved["csv"] = str(csv_path)
    pkl_path = out_stem.with_suffix(".pkl")
    df.to_pickle(pkl_path)
    saved["pickle"] = str(pkl_path)
    return saved


## 10. Droplet population library (inlined)

Verbatim copies of `nuclear_scaling.droplet_population` and `nuclear_scaling.droplet_geometry`
(Nuclear_Scaling_Core, as of 2026-11-04), inlined so that all analysis is contained in this
notebook. After each cell the functions are collected into a namespace (`dpop`, `dg`) so the
calls read exactly as before. The only edit is in `droplet_geometry`: its relative import of
constants from `droplet_population` now reads them from the first cell. The two modules have no
name collisions with each other (apart from an identical helper) or with this notebook.

In [ ]:
# §10 INLINED LIBRARY: nuclear_scaling.droplet_population (Nuclear_Scaling_Core, verbatim as of 2026-11-04).
# Collected into the namespace `dpop` at the end of this cell.
"""
droplet_population -- the selected analysis population for the Vulcan pipeline.

This module turns per-component measurements (one row per segmented nuclear
component on one Z plane) into three explicit, mutually exclusive tables:

    accepted     passes every enabled rule against a resolved parent droplet
    excluded     fails at least one definite rule; the reason is recorded
    unresolved   cannot be judged (no parent, ambiguous parent, ...);
                 NOT a confirmed artifact and NOT part of the accepted set

It never touches images, masks or the pipeline's baseline tables: everything
here is numpy/pandas on tables that `droplet_geometry` produced. Baseline
tables are filtered on the way out with `select_population`; they are never
rewritten.

What is approved, and what is not
---------------------------------
* APPROVED (user review, 2026-10): whole-droplet exclusion when two fitted
  droplet circles overlap by MORE THAN 5 % of the smaller full circle.
    - both circles must be observed sphere-inlier fits; a sphere-predicted
      circle can never trigger the rule
    - un-eroded circles, analytic intersection, no clipping to the image
    - strict comparison: overlap > tolerance
    - a hit on any evaluated Z plane flags BOTH droplet IDs on every Z plane
      of that timepoint; identity is never carried across timepoints
* EXISTING GATE (training-like thresholds, formulas preserved):
  raw-NLS enrichment >= 1.20 and solidity >= 0.80.
* EXPLORATORY (kept visible, never promoted silently):
    - the 85 % containment figure. It is used only to decide whether a
      component's parent droplet is resolved. A component below it is routed
      to `unresolved` (reason `partial_parent`), never called an artifact.
    - removal of exactly-one-pixel components (off by default).
  Probability and NPC-contrast cut-offs are deliberately absent: neither
  separated the reviewed artifacts from the reviewed real nuclei.

Known gaps this module does NOT close
-------------------------------------
* Z-linking (`group_nuclei_across_z`) is used as-is. A linked ID is not
  guaranteed to be one biological nucleus; branch flags are reported.
* Droplets holding more than one nucleus are not excluded.
"""


import hashlib
import json
import os
import re
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable, Mapping, Optional

import numpy as np
import pandas as pd

# ── Vocabulary ──────────────────────────────────────────────────────────────
OBSERVED = "observed_inlier"       # circle fitted on a sphere-inlier plane
PREDICTED = "sphere_predicted"     # circle implied by the sphere, not observed

STATUS_ACCEPTED = "accepted"
STATUS_EXCLUDED = "excluded"
STATUS_UNRESOLVED = "unresolved"
STATUSES = (STATUS_ACCEPTED, STATUS_EXCLUDED, STATUS_UNRESOLVED)

# Parent-assignment states written by droplet_geometry.assign_components.
GEOM_ASSIGNED = "assigned"
GEOM_PARTIAL = "partial_parent"
GEOM_MISSING = "missing_parent"
GEOM_AMBIGUOUS = "ambiguous_parent"
GEOM_NO_TIMEPOINT = "no_timepoint_geometry"
GEOMETRY_STATUSES = (GEOM_ASSIGNED, GEOM_PARTIAL, GEOM_MISSING, GEOM_AMBIGUOUS,
                     GEOM_NO_TIMEPOINT)

# Definite exclusion reasons.
REASON_OVERLAP = "overlapping_droplet_geometry"
REASON_ENRICHMENT = "enrichment"
REASON_SOLIDITY = "solidity"
REASON_ONE_PIXEL = "one_pixel_component"
# Reasons a component cannot be judged (in addition to the GEOM_* states).
REASON_NO_REFERENCE = "unassessable_cytoplasm"

COMPONENT_KEYS = ["t", "z", "label"]
ID_KEYS = ["t", "nucleus_3d_id"]

PAIR_COLUMNS = ["t", "z", "parent_a", "parent_b", "overlap_fraction",
                "observed_pair", "source_a", "source_b"]
FLAG_COLUMNS = ["t", "parent", "max_observed_overlap", "evidence_planes",
                "evidence_z", "partners", "reason"]

POPULATIONS = ("baseline", "accepted", "excluded", "unresolved",
               "accepted_plus_unresolved")
_SEP = " + "


# ── Settings ────────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class PopulationSettings:
    """Every rule that decides the selected population, in one place.

    Deliberately separate from the notebook's PipelineConfig: that dataclass
    is hashed into the run ID, so adding a field there would point the
    notebook at a new, empty run directory and force neural inference again.
    """

    # APPROVED rule.
    overlap_tolerance: float = 0.05
    # EXISTING GATE thresholds (training-like).
    min_enrichment: float = 1.20
    min_solidity: float = 0.80
    # EXPLORATORY. "unresolved": a component whose best parent holds less than
    # the containment fraction used by assign_components stays unresolved.
    # "evaluate": judge it against that best parent like an assigned one.
    partial_parent_policy: str = "unresolved"
    # EXPLORATORY, off by default: drop components of exactly one pixel.
    exclude_one_pixel_components: bool = False

    STATUS_OF_FIELD = {
        "overlap_tolerance": "APPROVED",
        "min_enrichment": "existing gate",
        "min_solidity": "existing gate",
        "partial_parent_policy": "EXPLORATORY",
        "exclude_one_pixel_components": "EXPLORATORY",
    }

    def __post_init__(self):
        if not 0.0 < float(self.overlap_tolerance) < 1.0:
            raise ValueError("overlap_tolerance must lie strictly between 0 and 1")
        if self.partial_parent_policy not in ("unresolved", "evaluate"):
            raise ValueError("partial_parent_policy must be 'unresolved' or 'evaluate'")
        if not np.isfinite([self.min_enrichment, self.min_solidity]).all():
            raise ValueError("gate thresholds must be finite")

    def to_dict(self) -> dict:
        return asdict(self)

    def describe(self) -> pd.DataFrame:
        """Table of each rule, its value, and whether it is approved."""
        values = self.to_dict()
        return pd.DataFrame(
            [dict(setting=k, value=values[k], standing=v)
             for k, v in self.STATUS_OF_FIELD.items()])


# ── Circle overlap ──────────────────────────────────────────────────────────
def circle_overlap_fraction(x1, y1, r1, x2, y2, r2) -> float:
    """Intersection area / area of the smaller FULL circle (analytic).

    Circles are taken un-eroded and are not clipped to the image. Returns 0
    for disjoint or externally tangent circles and 1 when one circle lies
    inside the other. Reference implementation; `droplet_overlap_pairs`
    evaluates the same expression in bulk.
    """
    if not np.isfinite([x1, y1, r1, x2, y2, r2]).all() or min(r1, r2) <= 0:
        raise ValueError("Circles must have finite coordinates and positive radii")
    d = float(np.hypot(x1 - x2, y1 - y2))
    if d >= r1 + r2:
        return 0.0
    if d <= abs(r1 - r2):
        return 1.0
    alpha = np.arccos(np.clip((d * d + r1 * r1 - r2 * r2) / (2 * d * r1), -1, 1))
    beta = np.arccos(np.clip((d * d + r2 * r2 - r1 * r1) / (2 * d * r2), -1, 1))
    term = max(0.0, (-d + r1 + r2) * (d + r1 - r2) * (d - r1 + r2) * (d + r1 + r2))
    lens = r1 * r1 * alpha + r2 * r2 * beta - 0.5 * np.sqrt(term)
    return float(np.clip(lens / (np.pi * min(r1, r2) ** 2), 0, 1))


def _pairwise_overlap(cx: np.ndarray, cy: np.ndarray, r: np.ndarray):
    """Vectorised circle_overlap_fraction for every pair i < j on one plane."""
    i, j = np.triu_indices(len(r), k=1)
    d = np.hypot(cx[i] - cx[j], cy[i] - cy[j])
    r1, r2 = r[i], r[j]
    fraction = np.zeros(len(d), dtype=float)
    contained = d <= np.abs(r1 - r2)
    lens = ~contained & (d < r1 + r2)
    fraction[contained] = 1.0
    if lens.any():
        dd, a, b = d[lens], r1[lens], r2[lens]
        alpha = np.arccos(np.clip((dd * dd + a * a - b * b) / (2 * dd * a), -1, 1))
        beta = np.arccos(np.clip((dd * dd + b * b - a * a) / (2 * dd * b), -1, 1))
        term = np.maximum(0.0, (-dd + a + b) * (dd + a - b) * (dd - a + b) * (dd + a + b))
        area = a * a * alpha + b * b * beta - 0.5 * np.sqrt(term)
        fraction[lens] = np.clip(area / (np.pi * np.minimum(a, b) ** 2), 0, 1)
    return i, j, fraction


def droplet_overlap_pairs(circles: pd.DataFrame) -> pd.DataFrame:
    """Every pair of circles on the same (t, z) plane with non-zero overlap.

    `circles` needs t, z, parent, cx, cy, radius_px, source. Pairs involving a
    sphere-predicted circle are reported with observed_pair=False so they can
    be inspected, but they never trigger exclusion.
    """
    need = {"t", "z", "parent", "cx", "cy", "radius_px", "source"}
    missing = need - set(circles.columns)
    if missing:
        raise KeyError(f"circles is missing columns: {sorted(missing)}")
    if circles.empty:
        return pd.DataFrame(columns=PAIR_COLUMNS)
    if circles.duplicated(["t", "z", "parent"]).any():
        raise ValueError("Duplicate circle keys (t, z, parent)")
    numeric = circles[["cx", "cy", "radius_px"]].to_numpy(dtype=float)
    if not np.isfinite(numeric).all() or (numeric[:, 2] <= 0).any():
        raise ValueError("Circles must have finite coordinates and positive radii")
    unknown = set(circles.source) - {OBSERVED, PREDICTED}
    if unknown:
        raise ValueError(f"Unknown circle source(s): {sorted(unknown)}")

    frames = []
    for (t, z), plane in circles.groupby(["t", "z"], sort=True):
        if len(plane) < 2:
            continue
        plane = plane.sort_values("parent")
        parent = plane.parent.to_numpy()
        source = plane.source.to_numpy()
        i, j, fraction = _pairwise_overlap(
            plane.cx.to_numpy(float), plane.cy.to_numpy(float),
            plane.radius_px.to_numpy(float))
        keep = fraction > 0
        if not keep.any():
            continue
        i, j = i[keep], j[keep]
        frames.append(pd.DataFrame({
            "t": int(t), "z": int(z),
            "parent_a": parent[i].astype(int), "parent_b": parent[j].astype(int),
            "overlap_fraction": fraction[keep],
            "observed_pair": (source[i] == OBSERVED) & (source[j] == OBSERVED),
            "source_a": source[i], "source_b": source[j]}))
    if not frames:
        return pd.DataFrame(columns=PAIR_COLUMNS)
    return pd.concat(frames, ignore_index=True)[PAIR_COLUMNS]


def overlap_hits(pairs: pd.DataFrame, tolerance: float) -> pd.DataFrame:
    """Pairs that trigger the rule: both observed, overlap STRICTLY above tolerance."""
    if not 0.0 < float(tolerance) < 1.0:
        raise ValueError("tolerance must lie strictly between 0 and 1")
    if pairs.empty:
        return pairs.iloc[0:0].copy()
    observed = pairs.observed_pair.astype(bool)
    return pairs[observed & (pairs.overlap_fraction.astype(float) > float(tolerance))].copy()


def flag_overlapping_droplets(pairs: pd.DataFrame, tolerance: float = 0.05) -> pd.DataFrame:
    """One row per flagged (t, parent). Both members of every hit are flagged.

    A flag belongs to the fitted inventory identity within one timepoint. It
    applies to that droplet on every Z plane of that timepoint and says
    nothing about any other timepoint.
    """
    hits = overlap_hits(pairs, tolerance)
    if hits.empty:
        return pd.DataFrame(columns=FLAG_COLUMNS)
    long = pd.concat([
        hits[["t", "z", "parent_a", "parent_b", "overlap_fraction"]].rename(
            columns={"parent_a": "parent", "parent_b": "partner"}),
        hits[["t", "z", "parent_b", "parent_a", "overlap_fraction"]].rename(
            columns={"parent_b": "parent", "parent_a": "partner"}),
    ], ignore_index=True)
    rows = []
    for (t, parent), g in long.groupby(["t", "parent"], sort=True):
        rows.append(dict(
            t=int(t), parent=int(parent),
            max_observed_overlap=float(g.overlap_fraction.max()),
            evidence_planes=int(g.z.nunique()),
            evidence_z=";".join(str(int(z)) for z in sorted(g.z.unique())),
            partners=";".join(str(int(p)) for p in sorted(g.partner.unique())),
            reason=REASON_OVERLAP))
    return pd.DataFrame(rows, columns=FLAG_COLUMNS)


# ── Component decisions ─────────────────────────────────────────────────────
_DECISION_INPUTS = ["t", "z", "label", "nucleus_3d_id", "area_pixels",
                    "geometry_status", "parent", "enrichment", "solidity",
                    "reference_ok"]


def decide_components(components: pd.DataFrame, flags: pd.DataFrame,
                      settings: PopulationSettings = PopulationSettings()) -> pd.DataFrame:
    """Add status and explicit reasons to each component. Input is not modified.

    Precedence: any definite failure -> excluded; otherwise anything that
    cannot be judged -> unresolved; otherwise accepted. Enrichment and the
    overlap rule need a resolved parent droplet, so they are only applied
    where the parent is resolved. Solidity is a property of the mask alone and
    is applied everywhere. A component is never moved to a different parent.
    """
    missing = set(_DECISION_INPUTS) - set(components.columns)
    if missing:
        raise KeyError(f"components is missing columns: {sorted(missing)}")
    if components.duplicated(COMPONENT_KEYS).any():
        raise ValueError("Duplicate component keys (t, z, label)")
    unknown = set(components.geometry_status) - set(GEOMETRY_STATUSES)
    if unknown:
        raise ValueError(f"Unknown geometry_status value(s): {sorted(unknown)}")

    d = components.copy()
    n = len(d)
    status_geom = d.geometry_status.to_numpy()
    resolved = status_geom == GEOM_ASSIGNED
    if settings.partial_parent_policy == "evaluate":
        resolved = resolved | (status_geom == GEOM_PARTIAL)

    flagged = set()
    if flags is not None and len(flags):
        flagged = set(zip(flags.t.astype(int), flags.parent.astype(int)))
    parent = d.parent.fillna(0).astype(int).to_numpy()
    t = d.t.astype(int).to_numpy()
    parent_flagged = np.fromiter(
        ((int(a), int(b)) in flagged for a, b in zip(t, parent)), bool, count=n)
    parent_flagged &= parent > 0

    solidity = pd.to_numeric(d.solidity, errors="coerce").to_numpy(float)
    enrichment = pd.to_numeric(d.enrichment, errors="coerce").to_numpy(float)
    reference_ok = d.reference_ok.map(lambda v: v is True or v == 1).to_numpy(bool)
    area = pd.to_numeric(d.area_pixels, errors="coerce").to_numpy(float)

    # Same tests as measure_segment_gate, applied to the stored measurements.
    fail_solidity = ~np.isfinite(solidity) | (solidity < settings.min_solidity)
    fail_enrichment = resolved & reference_ok & (enrichment < settings.min_enrichment)
    fail_overlap = resolved & parent_flagged
    fail_one_pixel = (area == 1) if settings.exclude_one_pixel_components else np.zeros(n, bool)
    no_reference = resolved & ~reference_ok

    exclusion, unresolved, no_overlap = [], [], []
    for k in range(n):
        ex = []
        if fail_enrichment[k]:
            ex.append(REASON_ENRICHMENT)
        if fail_solidity[k]:
            ex.append(REASON_SOLIDITY)
        if fail_one_pixel[k]:
            ex.append(REASON_ONE_PIXEL)
        no_overlap.append(bool(ex))
        if fail_overlap[k]:
            ex.append(REASON_OVERLAP)
        un = []
        if not resolved[k]:
            un.append(str(status_geom[k]))
        elif no_reference[k]:
            un.append(REASON_NO_REFERENCE)
        exclusion.append(_SEP.join(ex))
        unresolved.append(_SEP.join(un))

    has_exclusion = np.array([bool(x) for x in exclusion], bool)
    has_unknown = np.array([bool(x) for x in unresolved], bool)
    status = np.where(has_exclusion, STATUS_EXCLUDED,
                      np.where(has_unknown, STATUS_UNRESOLVED, STATUS_ACCEPTED))
    without_overlap = np.where(np.array(no_overlap, bool), STATUS_EXCLUDED,
                               np.where(has_unknown, STATUS_UNRESOLVED, STATUS_ACCEPTED))

    d["assignment_resolved"] = resolved
    d["parent_flagged"] = parent_flagged
    d["exclusion_reasons"] = exclusion
    d["unresolved_reasons"] = unresolved
    d["status"] = status
    d["reason"] = np.where(has_exclusion, d.exclusion_reasons,
                           np.where(has_unknown, d.unresolved_reasons, STATUS_ACCEPTED))
    d["status_without_overlap"] = without_overlap
    d["removed_by_overlap"] = (without_overlap == STATUS_ACCEPTED) & (status != STATUS_ACCEPTED)
    return d


def summarize_linked_ids(decisions: pd.DataFrame) -> pd.DataFrame:
    """One row per (t, linked nuclear ID).

    An ID is accepted when at least one of its components is accepted. With no
    accepted component it is unresolved if any component could not be judged,
    and excluded only when every component definitely failed. `lost_to_overlap`
    separates losing a whole ID to the overlap rule from losing some of its
    components (`partially_removed`).
    """
    columns = ID_KEYS + [
        "review_id", "status", "reasons", "n_components", "n_accepted",
        "n_excluded", "n_unresolved", "n_planes", "n_accepted_planes",
        "components_removed_by_overlap", "lost_to_overlap", "partially_removed",
        "has_overlap_flagged_component", "multiple_instances_same_z",
        "max_area_pixels_all", "max_area_pixels_accepted"]
    if decisions.empty:
        return pd.DataFrame(columns=columns)
    rows = []
    for (t, nid), g in decisions.groupby(ID_KEYS, sort=True):
        accepted = g[g.status == STATUS_ACCEPTED]
        n_acc = len(accepted)
        n_exc = int((g.status == STATUS_EXCLUDED).sum())
        n_unr = int((g.status == STATUS_UNRESOLVED).sum())
        status = (STATUS_ACCEPTED if n_acc else
                  STATUS_UNRESOLVED if n_unr else STATUS_EXCLUDED)
        reasons = sorted({part for text in g.loc[g.status != STATUS_ACCEPTED, "reason"]
                          for part in str(text).split(_SEP) if part})
        rows.append(dict(
            t=int(t), nucleus_3d_id=int(nid), review_id="T%d_N%d" % (t, nid),
            status=status, reasons=_SEP.join(reasons),
            n_components=len(g), n_accepted=n_acc, n_excluded=n_exc, n_unresolved=n_unr,
            n_planes=int(g.z.nunique()), n_accepted_planes=int(accepted.z.nunique()),
            components_removed_by_overlap=int(g.removed_by_overlap.sum()),
            lost_to_overlap=bool((g.status_without_overlap == STATUS_ACCEPTED).any() and not n_acc),
            partially_removed=bool(n_acc and n_acc < len(g)),
            has_overlap_flagged_component=bool(g.parent_flagged.any()),
            multiple_instances_same_z=bool((g.groupby("z").size() > 1).any()),
            max_area_pixels_all=float(g.area_pixels.max()),
            max_area_pixels_accepted=float(accepted.area_pixels.max()) if n_acc else np.nan))
    return pd.DataFrame(rows, columns=columns)


def overlap_impact(decisions: pd.DataFrame, pairs: pd.DataFrame, flags: pd.DataFrame,
                   tolerance: float) -> pd.DataFrame:
    """Per-timepoint effect of the overlap rule on components and linked IDs.

    "baseline" here means the population with every other rule applied and the
    overlap rule switched off. Counts are linked IDs, not independently
    confirmed biological nuclei.
    """
    hits = overlap_hits(pairs, tolerance)
    rows = []
    for t, g in decisions.groupby("t", sort=True):
        before = g.status_without_overlap == STATUS_ACCEPTED
        after = g.status == STATUS_ACCEPTED
        ids = (g.assign(_before=before, _after=after)
               .groupby("nucleus_3d_id")[["_before", "_after"]].any())
        hits_t = hits[hits.t == t] if len(hits) else hits
        rows.append(dict(
            t=int(t),
            droplets_flagged=int((flags.t == t).sum()) if len(flags) else 0,
            overlapping_observed_pairs=int(len(hits_t[["parent_a", "parent_b"]].drop_duplicates())),
            baseline_components=int(before.sum()),
            remaining_components=int(after.sum()),
            components_removed=int(g.removed_by_overlap.sum()),
            baseline_linked_ids=int(ids._before.sum()),
            remaining_linked_ids=int(ids._after.sum()),
            linked_ids_lost=int((ids._before & ~ids._after).sum()),
            unresolved_assignments=int(g.geometry_status.isin(
                [GEOM_MISSING, GEOM_AMBIGUOUS, GEOM_PARTIAL, GEOM_NO_TIMEPOINT]).sum()),
            unknown_gate_decisions=int(g.geometry_status.isin(
                [GEOM_MISSING, GEOM_AMBIGUOUS, GEOM_NO_TIMEPOINT]).sum())))
    return pd.DataFrame(rows)


def status_summary(decisions: pd.DataFrame, linked_ids: pd.DataFrame) -> pd.DataFrame:
    """Per-timepoint counts of components and linked IDs in each status."""
    rows = []
    for t in sorted(set(decisions.t.astype(int))):
        c = decisions[decisions.t == t]
        i = linked_ids[linked_ids.t == t]
        row = dict(t=t, components=len(c), linked_ids=len(i))
        for s in STATUSES:
            row[f"components_{s}"] = int((c.status == s).sum())
        for s in STATUSES:
            row[f"linked_ids_{s}"] = int((i.status == s).sum())
        row["linked_ids_partially_removed"] = int(i.partially_removed.sum())
        row["linked_ids_branched"] = int(i.multiple_instances_same_z.sum())
        rows.append(row)
    return pd.DataFrame(rows)


def reason_counts(decisions: pd.DataFrame) -> pd.DataFrame:
    """Component counts per timepoint, status and individual reason."""
    d = decisions[decisions.status != STATUS_ACCEPTED]
    if d.empty:
        return pd.DataFrame(columns=["t", "status", "reason", "components"])
    long = d.assign(reason=d.reason.str.split(_SEP, regex=False)).explode("reason")
    return (long.groupby(["t", "status", "reason"]).size()
            .rename("components").reset_index())


# ── Building, tagging, saving ───────────────────────────────────────────────
def _canonical(obj) -> str:
    return json.dumps(obj, sort_keys=True, default=str)


def population_tag(settings: PopulationSettings, upstream: Optional[Mapping] = None) -> str:
    """Short stable name for one (rules, upstream measurements) combination."""
    payload = dict(settings=settings.to_dict(), upstream=dict(upstream or {}))
    return "pop-" + hashlib.sha256(_canonical(payload).encode()).hexdigest()[:10]


def build_population(components: pd.DataFrame, circles: pd.DataFrame,
                     settings: PopulationSettings = PopulationSettings(),
                     upstream: Optional[Mapping] = None) -> dict:
    """Apply every enabled rule and return the population as a dict of tables.

    Keys: components (all, with status and reasons), accepted, excluded,
    unresolved, linked_ids, circles, pairs, flags, impact, summary, reasons,
    settings, upstream, tag. The inputs are not modified.
    """
    pairs = droplet_overlap_pairs(circles)
    flags = flag_overlapping_droplets(pairs, settings.overlap_tolerance)
    decisions = decide_components(components, flags, settings)
    linked = summarize_linked_ids(decisions)
    return dict(
        components=decisions,
        accepted=decisions[decisions.status == STATUS_ACCEPTED].copy(),
        excluded=decisions[decisions.status == STATUS_EXCLUDED].copy(),
        unresolved=decisions[decisions.status == STATUS_UNRESOLVED].copy(),
        linked_ids=linked, circles=circles.copy(), pairs=pairs, flags=flags,
        impact=overlap_impact(decisions, pairs, flags, settings.overlap_tolerance),
        summary=status_summary(decisions, linked),
        reasons=reason_counts(decisions),
        settings=settings.to_dict(), upstream=dict(upstream or {}),
        tag=population_tag(settings, upstream))


_TABLES = ("components", "accepted", "excluded", "unresolved", "linked_ids",
           "circles", "pairs", "flags", "impact", "summary", "reasons")
_FILE_OF = {"components": "component_decisions", "accepted": "accepted_components",
            "excluded": "excluded_components", "unresolved": "unresolved_components",
            "linked_ids": "linked_id_decisions", "circles": "droplet_circles",
            "pairs": "overlapping_circle_pairs", "flags": "flagged_droplets",
            "impact": "overlap_impact_by_t", "summary": "status_summary_by_t",
            "reasons": "reason_counts"}


def _atomic_write(path: Path, writer) -> None:
    tmp = path.with_name(path.name + ".tmp")
    writer(tmp)
    os.replace(tmp, path)


def save_population(population: Mapping, root: Path) -> Path:
    """Write the population under root/<tag>/ and return that directory.

    Only this directory is written. The pipeline's baseline pickles
    (plane_objects, grouped_z_objects, best_z_nuclei, tracked_nuclei, ...) and
    every mask file are never opened for writing. Re-saving the same tag
    replaces the files in place.
    """
    out = Path(root) / str(population["tag"])
    out.mkdir(parents=True, exist_ok=True)
    for key in _TABLES:
        table = population[key]
        stem = _FILE_OF[key]
        _atomic_write(out / f"{stem}.pkl", lambda p, table=table: table.to_pickle(p))
        _atomic_write(out / f"{stem}.csv", lambda p, table=table: table.to_csv(p, index=False))
    manifest = dict(
        tag=population["tag"], settings=population["settings"],
        upstream=population["upstream"],
        saved_at=datetime.now().isoformat(timespec="seconds"),
        rows={key: int(len(population[key])) for key in _TABLES},
        definition=dict(
            overlap_metric="analytic intersection / smaller full un-eroded circle area; no image clipping",
            overlap_trigger="both circles observed_inlier, any evaluated plane, strict > tolerance",
            overlap_propagation="both droplet identities across Z within the timepoint; never across timepoints",
            unresolved="not judged; neither accepted nor a confirmed artifact",
            not_enforced=["Z-linking unchanged", "multi-nucleus droplets not excluded"]))
    _atomic_write(out / "population_manifest.json",
                  lambda p: p.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf8"))
    return out


def list_saved_populations(root: Path) -> pd.DataFrame:
    """Populations saved under `root`, newest first (tag, time, rules, row counts)."""
    rows = []
    for manifest_path in sorted(Path(root).glob("pop-*/population_manifest.json")):
        m = json.loads(manifest_path.read_text(encoding="utf8"))
        rows.append(dict(tag=m["tag"], saved_at=m["saved_at"], directory=manifest_path.parent,
                         **{f"rule_{k}": v for k, v in m["settings"].items()},
                         accepted=m["rows"]["accepted"], excluded=m["rows"]["excluded"],
                         unresolved=m["rows"]["unresolved"]))
    table = pd.DataFrame(rows)
    return table.sort_values("saved_at", ascending=False).reset_index(drop=True) if len(table) else table


def load_population(directory: Path) -> dict:
    """Reload a population written by save_population."""
    directory = Path(directory)
    manifest = json.loads((directory / "population_manifest.json").read_text(encoding="utf8"))
    population = {key: pd.read_pickle(directory / f"{_FILE_OF[key]}.pkl") for key in _TABLES}
    population.update(settings=manifest["settings"], upstream=manifest["upstream"],
                      tag=manifest["tag"], output_dir=directory)
    return population


# ── Using the population downstream ─────────────────────────────────────────
def _wanted_statuses(which: str) -> tuple:
    if which not in POPULATIONS:
        raise ValueError(f"which must be one of {POPULATIONS}, not {which!r}")
    return {"accepted": (STATUS_ACCEPTED,), "excluded": (STATUS_EXCLUDED,),
            "unresolved": (STATUS_UNRESOLVED,),
            "accepted_plus_unresolved": (STATUS_ACCEPTED, STATUS_UNRESOLVED),
            "baseline": STATUSES}[which]


def select_population(table: pd.DataFrame, population: Mapping, which: str = "accepted",
                      key_lookup: Optional[pd.DataFrame] = None,
                      on_missing: str = "raise") -> pd.DataFrame:
    """Return the rows of a baseline table that belong to one population.

    The table is matched on its own component, (t, z, label). A best-Z row is
    therefore kept or dropped on the decision for the component it actually
    measures; no other Z plane is substituted. Tables without `label` (the
    radial sweep) are matched through `key_lookup`, a table carrying
    t, track_id, z, label (tracked_df).

    which = "baseline" returns an unfiltered copy. "accepted_plus_unresolved"
    exists so that including unjudged components is always a visible choice.
    Rows with no decision (a timepoint the stage was not run on) raise unless
    on_missing="drop". The input table is never modified.
    """
    wanted = _wanted_statuses(which)
    if on_missing not in ("raise", "drop"):
        raise ValueError("on_missing must be 'raise' or 'drop'")
    if which == "baseline":
        return table.copy()
    if table.empty:
        return table.copy()
    decisions = population["components"][COMPONENT_KEYS + ["status"]]

    if set(COMPONENT_KEYS) <= set(table.columns):
        merged = table.merge(decisions, on=COMPONENT_KEYS, how="left", validate="many_to_one")
        status = merged["status"].to_numpy(object)
    elif {"t", "track_id"} <= set(table.columns):
        if key_lookup is None:
            raise ValueError("This table has no (t, z, label); pass key_lookup=tracked_df")
        keys = ["t", "z", "track_id"] if "z" in table.columns else ["t", "track_id"]
        lookup = key_lookup[list(dict.fromkeys(keys + COMPONENT_KEYS))].merge(
            decisions, on=COMPONENT_KEYS, how="left", validate="many_to_one")
        # A key that maps to several components is kept only if they all agree.
        per_key = lookup.groupby(keys)["status"].agg(
            lambda s: s.iloc[0] if s.notna().all() and s.nunique() == 1 else
            (np.nan if s.isna().any() else "mixed")).rename("status").reset_index()
        merged = table.merge(per_key, on=keys, how="left", validate="many_to_one")
        status = merged["status"].to_numpy(object)
    else:
        raise KeyError("Table needs (t, z, label) or (t, track_id) to be matched to the population")

    # Positional masks: a left merge keeps the table's row order, and the
    # table's own index may not be unique.
    undecided = pd.isna(status)
    if undecided.any() and on_missing == "raise":
        times = sorted(set(table["t"].to_numpy()[undecided].astype(int)))
        raise ValueError(
            f"{int(undecided.sum())} row(s) have no population decision (timepoints {times}). "
            "Run the population stage on those timepoints, or pass on_missing='drop' "
            "to leave them out explicitly.")
    keep = np.isin(status, wanted)
    out = table[keep].copy()
    out["population_status"] = status[keep]
    return out


def population_as_gate(population: Mapping, pixel_size_um: float, which: str = "accepted",
                       legacy_gate: Optional[Mapping] = None) -> dict:
    """Present a population in the dict shape the section-38 audits expect.

    `gate_pass` is membership of the requested population. Enrichment, solidity
    and parent come from the fitted-droplet measurement. If the original
    model-droplet gate is supplied, its plane-local `parent_droplet` is carried
    over for the envelope audit, which draws the model droplet mask; the fitted
    parent stays available as `fitted_parent`.
    """
    wanted = _wanted_statuses(which)
    # The audits merge this table with the linked object table on (t, z, label)
    # and take nucleus_3d_id from there, exactly as with the original gate.
    d = population["components"].drop(columns="nucleus_3d_id")
    d["gate_pass"] = d.status.isin(wanted).to_numpy(bool)
    d["gate_reason"] = d.reason
    d["area_um2"] = d.area_pixels.astype(float) * float(pixel_size_um) ** 2
    d["fitted_parent"] = d.parent
    d["parent_droplet"] = d.parent
    if legacy_gate is not None:
        legacy = legacy_gate["all_instances"][COMPONENT_KEYS + ["parent_droplet"]]
        d = d.drop(columns="parent_droplet").merge(
            legacy, on=COMPONENT_KEYS, how="left", validate="one_to_one")
        if d.parent_droplet.isna().any():
            raise ValueError("Original gate does not cover every population component")
        d["parent_droplet"] = d.parent_droplet.astype(int)
    settings = dict(population["settings"])
    return dict(all_instances=d, accepted=d[d.gate_pass].copy(), rejected=d[~d.gate_pass].copy(),
                settings=settings, population=which, tag=population["tag"],
                output_dir=population.get("output_dir"))


_REVIEW_ID = re.compile(r"^T(\d+)_N(\d+)$")


def reviewed_outcomes(population: Mapping, review_labels: Mapping[str, str]) -> pd.DataFrame:
    """Where each manually reviewed linked ID ended up.

    `review_labels` maps "T<t>_N<linked id>" to a label such as "artifact" or
    "real nucleus". Linked IDs belong to one segmentation run: labels from
    another run are meaningless here. This is a report only and never feeds
    back into the decisions.
    """
    ids = population["linked_ids"].set_index("review_id")
    rows = []
    for review_id, label in review_labels.items():
        if not _REVIEW_ID.match(str(review_id)):
            raise ValueError(f"Review ID {review_id!r} is not of the form T<t>_N<id>")
        if review_id in ids.index:
            r = ids.loc[review_id]
            rows.append(dict(review_id=review_id, review_label=label, found=True,
                             status=r.status, reasons=r.reasons,
                             n_components=int(r.n_components), n_accepted=int(r.n_accepted),
                             n_excluded=int(r.n_excluded), n_unresolved=int(r.n_unresolved),
                             lost_to_overlap=bool(r.lost_to_overlap)))
        else:
            rows.append(dict(review_id=review_id, review_label=label, found=False,
                             status="not_in_population", reasons="", n_components=0,
                             n_accepted=0, n_excluded=0, n_unresolved=0, lost_to_overlap=False))
    return pd.DataFrame(rows, columns=["review_id", "review_label", "found", "status", "reasons",
                                       "n_components", "n_accepted", "n_excluded",
                                       "n_unresolved", "lost_to_overlap"])


def compare_tolerances(components: pd.DataFrame, circles: pd.DataFrame,
                       tolerances: Iterable[float],
                       settings: PopulationSettings = PopulationSettings()) -> pd.DataFrame:
    """Overlap impact at several tolerances. Diagnostic only; saves nothing."""
    pairs = droplet_overlap_pairs(circles)
    frames = []
    for tolerance in sorted(set(map(float, tolerances))):
        trial = PopulationSettings(**{**settings.to_dict(), "overlap_tolerance": tolerance})
        flags = flag_overlapping_droplets(pairs, tolerance)
        decisions = decide_components(components, flags, trial)
        impact = overlap_impact(decisions, pairs, flags, tolerance)
        impact.insert(0, "tolerance", tolerance)
        frames.append(impact)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


dpop = SimpleNamespace(**{_k: globals()[_k] for _k in ('COMPONENT_KEYS', 'FLAG_COLUMNS', 'GEOMETRY_STATUSES', 'GEOM_AMBIGUOUS', 'GEOM_ASSIGNED', 'GEOM_MISSING', 'GEOM_NO_TIMEPOINT', 'GEOM_PARTIAL', 'ID_KEYS', 'OBSERVED', 'PAIR_COLUMNS', 'POPULATIONS', 'PREDICTED', 'PopulationSettings', 'REASON_ENRICHMENT', 'REASON_NO_REFERENCE', 'REASON_ONE_PIXEL', 'REASON_OVERLAP', 'REASON_SOLIDITY', 'STATUSES', 'STATUS_ACCEPTED', 'STATUS_EXCLUDED', 'STATUS_UNRESOLVED', '_DECISION_INPUTS', '_FILE_OF', '_REVIEW_ID', '_SEP', '_TABLES', '_atomic_write', '_canonical', '_pairwise_overlap', '_wanted_statuses', 'build_population', 'circle_overlap_fraction', 'compare_tolerances', 'decide_components', 'droplet_overlap_pairs', 'flag_overlapping_droplets', 'list_saved_populations', 'load_population', 'overlap_hits', 'overlap_impact', 'population_as_gate', 'population_tag', 'reason_counts', 'reviewed_outcomes', 'save_population', 'select_population', 'status_summary', 'summarize_linked_ids')})
print('dpop: 50 names from inlined droplet_population')


In [ ]:
# §10 INLINED LIBRARY: nuclear_scaling.droplet_geometry (Nuclear_Scaling_Core, verbatim as of 2026-11-04).
# Collected into the namespace `dg` at the end of this cell.
"""
droplet_geometry -- training-style droplet geometry for the inference pipeline.

Inference builds droplets by thresholding the model's droplet head and taking
connected components. That is not how training defined a droplet, and on real
data it produced merged parents of ~13 million pixels. This module instead
reuses the training notebook's own geometry (NPC watershed inventory, radial
wall extraction, RANSAC circle fits, robust sphere profile, `p2_circle`
policy) on the raw image, without touching any nuclear mask.

Three steps, each resumable:

    compute_geometry      droplet geometry for every timepoint, cached per
                          timepoint with code / configuration / input provenance
    assign_components     reassign each saved nuclear component to a fitted
                          droplet support and re-measure enrichment / solidity
    (droplet_population)  apply the rules and build the population tables

Nothing here runs neural inference, retrains anything, or opens a mask, image
or baseline table for writing. Images and masks are memory-mapped read-only.

Two different Z scales appear and must not be merged:
    geometry_z_step_um   empirical axial scale of the droplet sphere fit (2.18)
    cfg.z_step_um        acquisition Z spacing (2.0); not used here at all
"""


import ast
import copy
import hashlib
import itertools
import json
import os
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Iterable, Mapping, Optional, Sequence

import numpy as np
import pandas as pd

# inlined: these constants are defined by the droplet_population cell above
_FROM_DROPLET_POPULATION = (GEOM_AMBIGUOUS, GEOM_ASSIGNED, GEOM_MISSING,
                                 GEOM_NO_TIMEPOINT, GEOM_PARTIAL, OBSERVED, PREDICTED)

# The training functions that make up the geometry workflow, in the order the
# reviewed audit hashed them. Changing this tuple changes the code hash and so
# invalidates every geometry cache; do it deliberately.
GEOMETRY_FUNCTION_NAMES = (
    "extract_plane", "clip_histogram", "_circularity", "detect_droplets_npc_watershed",
    "_circle_from_3", "_fit_circle_kasa", "fit_circle_ransac", "smooth_npc_plane",
    "wall_points_on_smoothed", "fit_droplet_circle_on_smoothed", "_sphere_lsq",
    "fit_sphere_profile", "fit_sphere_ransac", "compute_droplet_geometry",
    "sphere_radius_px", "predicted_circle", "p2_circle")

GEOMETRY_MANIFEST = "geometry_signature.json"
COMPONENT_SCHEMA_VERSION = 1

CIRCLE_COLUMNS = ["t", "z", "parent", "cx", "cy", "radius_px", "support_pixels", "source"]
COMPONENT_COLUMNS = [
    "t", "z", "label", "nucleus_3d_id", "area_pixels", "parent_candidates",
    "candidate_parents", "parent", "parent_source", "parent_overlap_fraction",
    "geometry_status", "geometry_gate_pass", "contained_gate_pass", "enrichment",
    "solidity", "nucleus_mean", "cytoplasm_mean", "ring_pixels", "reference_ok",
    "gate_reason"]


# ── Enrichment / solidity gate (one canonical copy of the training formula) ──
@dataclass(frozen=True)
class ArtifactGateSettings:
    min_enrichment: float = 1.20
    min_solidity: float = 0.80
    ring_gap_um: float = 0.5
    ring_width_um: float = 2.0
    min_ring_pixels: int = 50


def measure_segment_gate(image, candidate, support, raw_foreground, pixel_size_um,
                         settings=ArtifactGateSettings(), solidity=None):
    """Raw-NLS enrichment and solidity of one candidate nucleus.

    Training formula, unchanged. All masks share one crop that is padded by at
    least the outer ring radius. The reference ring is the band between
    `ring_gap_um` and `ring_gap_um + ring_width_um` outside the candidate,
    restricted to the droplet support and with raw Otsu foreground removed.
    Returns (measurements, ring mask). `gate_pass` is False when the reference
    ring is too small to measure; that is "unassessable", not an artifact.
    """
    from skimage import measure, morphology

    gap = max(1, int(np.ceil(settings.ring_gap_um / pixel_size_um)))
    outer = gap + max(1, int(np.ceil(settings.ring_width_um / pixel_size_um)))
    ring = (morphology.binary_dilation(candidate, morphology.disk(outer)) &
            ~morphology.binary_dilation(candidate, morphology.disk(gap)) &
            support & ~raw_foreground)
    nr = int(ring.sum())
    inside = float(np.mean(image[candidate], dtype=np.float64)) if candidate.any() else np.nan
    outside = float(np.mean(image[ring], dtype=np.float64)) if nr else np.nan
    valid = nr >= settings.min_ring_pixels and np.isfinite(outside) and outside > 0 and np.isfinite(inside)
    enrichment = inside / outside if valid else np.nan
    if solidity is None:
        regions = measure.regionprops(candidate.astype(np.uint8))
        solidity = float(regions[0].solidity) if regions else np.nan
    reasons = []
    if not valid:
        reasons.append("unassessable_cytoplasm")
    elif enrichment < settings.min_enrichment:
        reasons.append("enrichment")
    if not np.isfinite(solidity) or solidity < settings.min_solidity:
        reasons.append("solidity")
    return dict(enrichment=enrichment, solidity=solidity, nucleus_mean=inside,
                cytoplasm_mean=outside, ring_pixels=nr, reference_ok=bool(valid),
                gate_pass=not reasons,
                gate_reason="keep" if not reasons else " + ".join(reasons)), ring


# ── Loading the training geometry ───────────────────────────────────────────
@dataclass
class TrainingGeometry:
    """The training notebook's geometry functions plus the settings they read."""
    namespace: dict
    config: SimpleNamespace
    settings: dict
    code_sha256: str
    notebook: Optional[Path] = None


def load_training_geometry(training_notebook, pixel_size_um: float,
                           npc_channel_index: int) -> TrainingGeometry:
    """Lift the named geometry functions and declared defaults out of the notebook.

    The notebook is parsed, never run: only the function definitions listed in
    GEOMETRY_FUNCTION_NAMES are compiled, and only literal defaults named in
    PipelineConfig._GEOM_FIELDS (plus erosion_px) are read. No training cell,
    setup cell or pipeline object is executed. Pixel size and NPC channel come
    from the inference configuration, not from the training notebook.
    """
    from scipy import ndimage
    from skimage import filters, measure, morphology, segmentation
    from skimage.morphology import h_maxima

    training_notebook = Path(training_notebook)
    source = training_notebook.read_text(encoding="utf8")
    functions, config = {}, None
    for cell in json.loads(source)["cells"]:
        if cell["cell_type"] != "code":
            continue
        text = "".join(cell["source"])
        try:
            tree = ast.parse(text)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.FunctionDef):
                functions[node.name] = ast.get_source_segment(text, node)
            if isinstance(node, ast.ClassDef) and node.name == "PipelineConfig":
                config = node
    absent = [name for name in GEOMETRY_FUNCTION_NAMES if name not in functions]
    if absent:
        raise ValueError(f"{training_notebook.name} does not define: {absent}")
    if config is None:
        raise ValueError(f"{training_notebook.name} has no PipelineConfig class")

    fields = {n.target.id: n.value for n in config.body if isinstance(n, ast.AnnAssign)}
    geom_fields = next(
        (ast.literal_eval(n.value) for n in config.body
         if isinstance(n, ast.Assign)
         and any(isinstance(t, ast.Name) and t.id == "_GEOM_FIELDS" for t in n.targets)),
        None)
    if geom_fields is None:
        raise ValueError("Training PipelineConfig declares no _GEOM_FIELDS")
    settings = {k: ast.literal_eval(fields[k]) for k in tuple(geom_fields) + ("erosion_px",)}
    settings.update(pixel_size_um=float(pixel_size_um), npc_channel_idx=int(npc_channel_index))

    gc = SimpleNamespace(**settings)
    gc.min_droplet_area_px = lambda: gc.min_droplet_area_um2 / gc.pixel_size_um ** 2
    selected = "\n\n".join(functions[name] for name in GEOMETRY_FUNCTION_NAMES)
    namespace = dict(np=np, filters=filters, morphology=morphology, measure=measure,
                     segmentation=segmentation, ndimage=ndimage, h_maxima=h_maxima,
                     copy=copy, itertools=itertools, cfg=gc)
    exec(compile(selected, str(training_notebook) + ":geometry-only", "exec"), namespace)
    return TrainingGeometry(namespace=namespace, config=gc, settings=settings,
                            code_sha256=hashlib.sha256(selected.encode()).hexdigest(),
                            notebook=training_notebook)


# ── Provenance ──────────────────────────────────────────────────────────────
def _canonical(obj) -> str:
    return json.dumps(obj, sort_keys=True, default=str)


def _file_identity(path) -> list:
    path = Path(path)
    st = path.stat()
    return [str(path.resolve()), st.st_size, st.st_mtime_ns]


def geometry_signature(training: TrainingGeometry, image_path) -> dict:
    """What a cached geometry depends on: settings, function source, input file.

    Same layout as the reviewed section-38g audit, so its caches stay usable.
    """
    return dict(settings=training.settings, code_sha256=training.code_sha256,
                image=_file_identity(image_path))


def signature_hash(signature: Mapping) -> str:
    return hashlib.sha256(_canonical(signature).encode()).hexdigest()[:12]


def _same_signature(a: Mapping, b: Mapping) -> bool:
    return json.loads(_canonical(a)) == json.loads(_canonical(b))


def _atomic_pickle(obj, path: Path) -> None:
    tmp = path.with_name(path.name + ".tmp")
    pd.to_pickle(obj, tmp)
    os.replace(tmp, path)


def _prepare_cache_dir(cache_dir: Path, signature: Mapping) -> None:
    """Create the cache, or verify that an existing one has matching provenance."""
    manifest = cache_dir / GEOMETRY_MANIFEST
    if manifest.exists():
        if not _same_signature(json.loads(manifest.read_text(encoding="utf8")), signature):
            raise ValueError(
                f"Geometry cache provenance mismatch in {cache_dir}. The training geometry "
                "code, its settings, or the input image changed. Use a fresh cache directory.")
        return
    if cache_dir.exists() and any(cache_dir.iterdir()):
        raise ValueError(f"{cache_dir} holds files but no {GEOMETRY_MANIFEST}; use a fresh directory")
    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest.write_text(json.dumps(signature, indent=2, default=str), encoding="utf8")


def find_geometry_caches(search_root, signature: Mapping) -> pd.DataFrame:
    """Earlier geometry caches under `search_root` and whether they match.

    Looks for geometry_signature.json up to two folders deep (this is where the
    section-38g audit left its `ransac_droplet_<timestamp>` folders). Read-only.
    """
    rows = []
    root = Path(search_root)
    if root.exists():
        found = sorted(set(root.glob(f"*/{GEOMETRY_MANIFEST}")) |
                       set(root.glob(f"*/*/{GEOMETRY_MANIFEST}")))
        for manifest in found:
            try:
                same = _same_signature(json.loads(manifest.read_text(encoding="utf8")), signature)
            except (OSError, ValueError):
                same = False
            times = sorted(int(p.stem.split("_t")[-1]) for p in manifest.parent.glob("geometry_t*.pkl"))
            rows.append(dict(cache_dir=manifest.parent, provenance_matches=same, timepoints=times))
    return pd.DataFrame(rows, columns=["cache_dir", "provenance_matches", "timepoints"])


# ── Step 1: geometry for every timepoint ────────────────────────────────────
def _geometry_for_timepoint(hs, t: int, training: TrainingGeometry, progress: bool):
    ns, gc = training.namespace, training.config
    plane = ns["extract_plane"](hs, t, gc.inventory_ref_z, gc.npc_channel_idx)
    inventory = ns["detect_droplets_npc_watershed"](plane, gc, compact=True)
    if not inventory:
        raise RuntimeError("T%d: no droplet inventory; refusing to fall back to model droplets" % t)
    geometry = ns["compute_droplet_geometry"](hs, t, inventory, gc, progress=progress)
    return geometry, len(inventory)


def _geometry_worker(job: dict) -> dict:
    """Compute and cache one timepoint. Module-level so a spawned process can run it."""
    import tifffile

    started = time.perf_counter()
    t = int(job["t"])
    try:
        training = load_training_geometry(job["training_notebook"], job["pixel_size_um"],
                                          job["npc_channel_index"])
        if training.code_sha256 != job["code_sha256"]:
            raise RuntimeError("Training geometry source changed while the stage was running")
        hs = tifffile.memmap(job["image_path"], mode="r")
        geometry, n_inventory = _geometry_for_timepoint(hs, t, training, progress=False)
        _atomic_pickle(dict(geometry=geometry, n_inventory=n_inventory), Path(job["cache_file"]))
        return dict(t=t, error=None, seconds=time.perf_counter() - started)
    except Exception as exc:  # reported to the parent, which decides what to do
        return dict(t=t, error=f"{type(exc).__name__}: {exc}", seconds=time.perf_counter() - started)


def compute_geometry(image_path, training_notebook=None, *, pixel_size_um: float,
                     npc_channel_index: int, cache_dir, timepoints: Optional[Iterable[int]] = None,
                     reuse_cache_dirs: Sequence = (), n_workers: int = 1,
                     on_error: str = "raise", progress: bool = True,
                     training: Optional[TrainingGeometry] = None) -> dict:
    """Training-style droplet geometry for every requested timepoint.

    timepoints=None means every timepoint in the image. Each finished timepoint
    is written to `cache_dir/geometry_tNNN.pkl`; an interrupted run resumes
    from the files already there. The cache carries a provenance signature
    (geometry settings, source of the geometry functions, image path/size/mtime)
    and is refused if it does not match.

    reuse_cache_dirs: older caches (for example the T0-T2 audit folders). A
    timepoint missing from `cache_dir` is copied from the first of these whose
    provenance matches; a mismatched one is ignored with a message.

    n_workers > 1 computes timepoints in separate spawned processes (never
    forked: the notebook kernel has TensorFlow loaded). Each worker holds one
    timepoint's smoothed NPC planes in memory. Per-plane progress is only
    printed when n_workers == 1.

    on_error="raise" stops at the first timepoint that fails, after the others
    in flight have been cached. on_error="record" finishes the rest and lists
    the failure in the coverage table; that timepoint's components then come
    out as unresolved (`no_timepoint_geometry`), never as artifacts.
    """
    import tifffile

    if on_error not in ("raise", "record"):
        raise ValueError("on_error must be 'raise' or 'record'")
    if training is None:
        if training_notebook is None:
            raise ValueError("Give training_notebook (or a loaded TrainingGeometry)")
        training = load_training_geometry(training_notebook, pixel_size_um, npc_channel_index)
    elif int(n_workers) > 1 and training.notebook is None:
        raise ValueError("n_workers > 1 needs a TrainingGeometry loaded from a notebook file")
    gc = training.config
    hs = tifffile.memmap(str(image_path), mode="r")
    if hs.ndim != 5:
        raise ValueError(f"Expected a TZCYX hyperstack, got shape {hs.shape}")
    if not 0 <= gc.inventory_ref_z < hs.shape[1]:
        raise ValueError("Training inventory_ref_z lies outside the stack")
    ts = sorted(set(range(hs.shape[0]) if timepoints is None else map(int, timepoints)))
    if not ts or ts[0] < 0 or ts[-1] >= hs.shape[0]:
        raise ValueError(f"Invalid timepoints {ts} for an image with {hs.shape[0]} timepoints")

    signature = geometry_signature(training, image_path)
    cache_dir = Path(cache_dir)
    _prepare_cache_dir(cache_dir, signature)
    if progress:
        print(f"Geometry cache (resumable): {cache_dir}", flush=True)

    donors = []
    for other in reuse_cache_dirs:
        other = Path(other)
        manifest = other / GEOMETRY_MANIFEST
        if other.resolve() == cache_dir.resolve():
            continue
        if manifest.exists() and _same_signature(json.loads(manifest.read_text(encoding="utf8")), signature):
            donors.append(other)
        elif progress:
            print(f"Not reusing {other}: provenance missing or different", flush=True)

    files = {t: cache_dir / ("geometry_t%03d.pkl" % t) for t in ts}
    origin, seconds, errors = {}, {}, {}
    todo = []
    for t in ts:
        if files[t].exists():
            origin[t] = "cache"
            continue
        donor = next((d / files[t].name for d in donors if (d / files[t].name).exists()), None)
        if donor is not None:
            _atomic_pickle(pd.read_pickle(donor), files[t])
            origin[t] = f"reused from {donor.parent.name}"
            continue
        todo.append(t)
    if progress:
        print(f"Timepoints: {len(ts)} requested, {len(ts) - len(todo)} already cached, "
              f"{len(todo)} to compute", flush=True)

    def finished(t, error, took):
        seconds[t] = took
        if error is None:
            origin[t] = "computed"
        else:
            origin[t], errors[t] = "failed", error
        if progress:
            done = sum(1 for x in todo if x in seconds)
            mean = float(np.mean([seconds[x] for x in todo if x in seconds]))
            left = (len(todo) - done) * mean / max(1, min(int(n_workers), len(todo)))
            state = "done" if error is None else f"FAILED ({error})"
            print(f"T{t} geometry {state} in {took / 60:.1f} min | {done}/{len(todo)} computed, "
                  f"about {left / 60:.0f} min left", flush=True)

    if todo and int(n_workers) > 1:
        import multiprocessing
        from concurrent.futures import ProcessPoolExecutor, as_completed

        jobs = [dict(t=t, training_notebook=str(training.notebook), pixel_size_um=float(pixel_size_um),
                     npc_channel_index=int(npc_channel_index), code_sha256=training.code_sha256,
                     image_path=str(image_path), cache_file=str(files[t])) for t in todo]
        context = multiprocessing.get_context("spawn")
        with ProcessPoolExecutor(max_workers=min(int(n_workers), len(todo)), mp_context=context) as pool:
            for future in as_completed([pool.submit(_geometry_worker, job) for job in jobs]):
                result = future.result()
                finished(result["t"], result["error"], result["seconds"])
    else:
        for t in todo:
            started = time.perf_counter()
            try:
                geometry, n_inventory = _geometry_for_timepoint(hs, t, training, progress)
                _atomic_pickle(dict(geometry=geometry, n_inventory=n_inventory), files[t])
                finished(t, None, time.perf_counter() - started)
            except Exception as exc:
                finished(t, f"{type(exc).__name__}: {exc}", time.perf_counter() - started)
                if on_error == "raise":
                    raise
    if errors and on_error == "raise":
        raise RuntimeError("Geometry failed for timepoint(s): " +
                           "; ".join(f"T{t}: {e}" for t, e in sorted(errors.items())))

    geometry, coverage = {}, []
    for t in ts:
        if t in errors:
            coverage.append(dict(t=t, status="failed", inventory=0, validated_geometry=0,
                                 origin=origin[t], minutes=seconds.get(t, np.nan) / 60,
                                 error=errors[t]))
            continue
        saved = pd.read_pickle(files[t])
        geometry[t] = saved["geometry"]
        coverage.append(dict(t=t, status="ok", inventory=int(saved["n_inventory"]),
                             validated_geometry=len(saved["geometry"]), origin=origin[t],
                             minutes=seconds.get(t, np.nan) / 60, error=""))
    return dict(geometry=geometry, coverage=pd.DataFrame(coverage), signature=signature,
                signature_hash=signature_hash(signature), cache_dir=cache_dir,
                settings=dict(training.settings), training=training,
                image_path=Path(image_path), timepoints=ts)


# ── Step 2: reassign components to fitted supports ──────────────────────────
def circle_crop(cx, cy, radius, shape):
    """Bounding box of a disc clipped to the image, and the disc mask inside it."""
    h, w = shape
    a, b = max(0, int(np.floor(cy - radius))), max(0, int(np.floor(cx - radius)))
    c, d = min(h, int(np.ceil(cy + radius)) + 1), min(w, int(np.ceil(cx + radius)) + 1)
    a, b, c, d = min(a, h), min(b, w), max(c, 0), max(d, 0)
    yy, xx = np.ogrid[a:c, b:d]
    return (a, b, c, d), (xx - cx) ** 2 + (yy - cy) ** 2 <= radius ** 2


def plane_circles(geometry_t: Mapping, z: int, shape, training: TrainingGeometry) -> list:
    """Droplet circles on one plane under the training `p2_circle` policy.

    A valid observed circle is used where the plane is a sphere inlier;
    otherwise the sphere prediction. `source` records which, so a predicted
    circle is never presented as an observed boundary. `r` is the un-eroded
    radius; `mask` is the support eroded by the training `erosion_px`.
    """
    gc = training.config
    circles = []
    for did, g in geometry_t.items():
        selected = training.namespace["p2_circle"](g, z, gc)
        if selected is None:
            continue
        kind, circ = selected
        if kind not in ("fitted", "predicted"):
            raise ValueError("Unsupported training p2_circle contract")
        if circ[2] <= gc.erosion_px:
            continue
        cx, cy, r = map(float, circ)
        box, mask = circle_crop(cx, cy, r - gc.erosion_px, shape)
        if mask.any():
            circles.append(dict(parent=int(did) + 1, cx=cx, cy=cy, r=r,
                                source=OBSERVED if kind == "fitted" else PREDICTED,
                                box=box, mask=mask))
    return circles


def included_planes(segmentation_index: pd.DataFrame, min_z: int) -> set:
    """(t, z) planes that segmentation committed and that are at or above min_z."""
    included = segmentation_index["included"].astype(str).str.lower().isin(["true", "1"])
    planes = segmentation_index.loc[included & (segmentation_index.z >= int(min_z)), ["t", "z"]]
    return set(map(tuple, planes.astype(int).to_numpy()))


def _assign_timepoint(hs, labels, group_t: pd.DataFrame, geometry_t: Optional[Mapping],
                      training: TrainingGeometry, nuclear_channel_index: int,
                      pixel_size_um: float, params: ArtifactGateSettings,
                      min_parent_overlap: float, progress: bool):
    from skimage import filters, measure

    gc = training.config
    t = int(group_t.t.iloc[0])
    rows, circle_rows = [], []
    gap = max(1, int(np.ceil(params.ring_gap_um / pixel_size_um)))
    pad = gap + max(1, int(np.ceil(params.ring_width_um / pixel_size_um)))
    for z, group in group_t.groupby("z", sort=True):
        z = int(z)
        lab = np.asarray(labels[t, z])
        regions = {r.label: r for r in measure.regionprops(lab)}
        circles, assignments, support = [], {}, {}
        if geometry_t is not None:
            raw = np.asarray(hs[t, z, nuclear_channel_index])
            circles = plane_circles(geometry_t, z, lab.shape, training)
            for circle in circles:
                did = circle["parent"]
                a, b, c, d = circle["box"]
                mask = circle["mask"]
                ids, counts = np.unique(lab[a:c, b:d][mask], return_counts=True)
                for label, count in zip(ids, counts):
                    if label:
                        assignments.setdefault(int(label), []).append((int(count), did))
                values = raw[a:c, b:d][mask]
                threshold = (float(filters.threshold_otsu(values))
                             if values.max() > values.min() else np.nan)
                support[did] = (circle, threshold)
                circle_rows.append(dict(t=t, z=z, parent=did, cx=circle["cx"], cy=circle["cy"],
                                        radius_px=circle["r"], support_pixels=int(mask.sum()),
                                        source=circle["source"]))
        for row in group.itertuples():
            label = int(row.label)
            region = regions.get(label)
            if region is None or int(region.area) != int(row.area_px):
                raise ValueError(
                    f"T{t} Z{z} label {label}: the object table disagrees with the saved nuclear "
                    "masks. They must come from the same segmentation run.")
            candidates = sorted(assignments.get(label, []), reverse=True)
            eligible = [(n, did) for n, did in candidates if n / region.area >= min_parent_overlap]
            rec = dict(
                t=t, z=z, label=label, nucleus_3d_id=int(row.nucleus_3d_id),
                area_pixels=int(region.area), parent_candidates=len(candidates),
                candidate_parents=";".join("%d:%.4f" % (did, n / region.area) for n, did in candidates),
                parent=0, parent_source="none", parent_overlap_fraction=0.0,
                geometry_status=GEOM_MISSING if geometry_t is not None else GEOM_NO_TIMEPOINT,
                geometry_gate_pass=None, contained_gate_pass=None, enrichment=np.nan,
                solidity=float(region.solidity), nucleus_mean=np.nan, cytoplasm_mean=np.nan,
                ring_pixels=0, reference_ok=False, gate_reason="unassessable geometry")
            # More than one circle holding the component is ambiguous, as is an
            # exact tie for the largest share. The component is then left
            # unassigned; it is never handed to one of the candidates.
            tied = not eligible and len(candidates) > 1 and candidates[0][0] == candidates[1][0]
            if candidates and len(eligible) <= 1 and not tied:
                n, did = eligible[0] if eligible else candidates[0]
                circle, threshold = support[did]
                overlap = n / region.area
                a, b, c, d = region.bbox
                a, b = max(0, a - pad), max(0, b - pad)
                c, d = min(lab.shape[0], c + pad), min(lab.shape[1], d + pad)
                yy, xx = np.ogrid[a:c, b:d]
                inside = ((xx - circle["cx"]) ** 2 + (yy - circle["cy"]) ** 2
                          <= (circle["r"] - gc.erosion_px) ** 2)
                crop = raw[a:c, b:d]
                candidate = lab[a:c, b:d] == label
                foreground = ((crop > threshold) & inside if np.isfinite(threshold)
                              else np.zeros_like(inside))
                metric, _ = measure_segment_gate(crop, candidate, inside, foreground,
                                                 pixel_size_um, params, float(region.solidity))
                gate_pass = bool(metric.pop("gate_pass"))
                rec.update(metric)
                rec.update(parent=did, parent_source=circle["source"],
                           parent_overlap_fraction=overlap,
                           geometry_status=GEOM_ASSIGNED if overlap >= min_parent_overlap else GEOM_PARTIAL,
                           geometry_gate_pass=gate_pass,
                           contained_gate_pass=bool(gate_pass and overlap >= min_parent_overlap))
            elif len(eligible) > 1 or tied:
                rec["geometry_status"] = GEOM_AMBIGUOUS
            rows.append(rec)
        if progress:
            print("  T%d Z%d: %d fitted/predicted supports, %d nuclear components"
                  % (t, z, len(circles), len(group)), flush=True)
    return (pd.DataFrame(rows, columns=COMPONENT_COLUMNS),
            pd.DataFrame(circle_rows, columns=CIRCLE_COLUMNS))


def _concat(frames, columns) -> pd.DataFrame:
    frames = [f for f in frames if len(f)]
    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True)[columns]


def assign_components(image_path, labels_path, components: pd.DataFrame, geometry: Mapping, *,
                      nuclear_channel_index: int, pixel_size_um: float,
                      gate_settings: ArtifactGateSettings = ArtifactGateSettings(),
                      min_parent_overlap: float = 0.85, valid_planes: Optional[set] = None,
                      cache_dir=None, progress: bool = True) -> dict:
    """Reassign saved nuclear components to fitted droplet supports and re-measure them.

    `components` is the pipeline's linked object table (grouped_z_df): one row
    per saved nuclear instance with t, z, label, area_px and nucleus_3d_id. It
    is read, never changed. `geometry` is the result of compute_geometry.

    For each component the fitted (eroded) circles covering it are counted:
        assigned          one circle holds >= min_parent_overlap of its pixels
        partial_parent    its best circle holds less than that
        ambiguous_parent  two or more circles hold >= min_parent_overlap, or
                          the best share is an exact tie
        missing_parent    no fitted circle touches it
    `min_parent_overlap` (0.85) is an exploratory figure: it decides whether a
    parent counts as resolved, it is not a validated biological cut-off.

    Circles are evaluated on the planes that carry at least one nuclear
    component, which is exactly the set the reviewed overlap diagnostic used.

    Per-timepoint results are cached under `cache_dir` and reused only when the
    geometry signature, mask file, gate settings and that timepoint's object
    rows are all unchanged.
    """
    import tifffile

    if not 0 < min_parent_overlap <= 1:
        raise ValueError("min_parent_overlap must be in (0, 1]")
    if pixel_size_um <= 0:
        raise ValueError("pixel_size_um must be positive")
    need = {"t", "z", "label", "area_px", "nucleus_3d_id"}
    if need - set(components.columns):
        raise KeyError(f"components is missing columns: {sorted(need - set(components.columns))}")
    if components.duplicated(["t", "z", "label"]).any():
        raise ValueError("Duplicate component keys (t, z, label)")
    hs = tifffile.memmap(str(image_path), mode="r")
    labels = tifffile.memmap(str(labels_path), mode="r")
    if hs.shape[:2] + hs.shape[-2:] != labels.shape:
        raise ValueError(f"Raw image {hs.shape} and nuclear masks {labels.shape} do not match")
    training = geometry["training"]
    wanted = set(geometry["timepoints"])
    present = set(components.t.astype(int))
    if not present <= wanted:
        raise ValueError(f"No geometry was requested for timepoint(s) {sorted(present - wanted)}. "
                         "Run compute_geometry on them, or pass only the rows you want judged.")
    if valid_planes is not None:
        planes = set(map(tuple, components[["t", "z"]].drop_duplicates().astype(int).to_numpy()))
        bad = sorted(planes - set(valid_planes))
        if bad:
            raise ValueError(f"Components lie on planes segmentation did not include: {bad[:5]}")

    table = components[["t", "z", "label", "area_px", "nucleus_3d_id"]].sort_values(["t", "z", "label"]).reset_index(drop=True)
    shared = dict(schema=COMPONENT_SCHEMA_VERSION, geometry=geometry["signature_hash"],
                  labels=_file_identity(labels_path), gate=asdict(gate_settings),
                  min_parent_overlap=float(min_parent_overlap),
                  nuclear_channel_index=int(nuclear_channel_index),
                  pixel_size_um=float(pixel_size_um))
    if cache_dir is not None:
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        if progress:
            print(f"Component cache (resumable): {cache_dir}", flush=True)

    frames, circle_frames, hashes = [], [], {}
    for t, group_t in table.groupby("t", sort=True):
        t = int(t)
        rows_hash = hashlib.sha256(pd.util.hash_pandas_object(group_t, index=False).values.tobytes()).hexdigest()
        signature = dict(shared, t=t, rows=rows_hash,
                         has_geometry=t in geometry["geometry"])
        hashes[t] = signature_hash(signature)
        cache_file = cache_dir / ("components_t%03d.pkl" % t) if cache_dir is not None else None
        if cache_file is not None and cache_file.exists():
            saved = pd.read_pickle(cache_file)
            if _same_signature(saved.get("signature", {}), signature):
                frames.append(saved["components"])
                circle_frames.append(saved["circles"])
                if progress:
                    print(f"T{t}: loaded {len(saved['components'])} cached component rows", flush=True)
                continue
            if progress:
                print(f"T{t}: cached components have different provenance; recomputing", flush=True)
        started = time.perf_counter()
        comp, circ = _assign_timepoint(hs, labels, group_t, geometry["geometry"].get(t), training,
                                       int(nuclear_channel_index), float(pixel_size_um),
                                       gate_settings, float(min_parent_overlap), progress)
        if cache_file is not None:
            _atomic_pickle(dict(signature=signature, components=comp, circles=circ), cache_file)
        frames.append(comp)
        circle_frames.append(circ)
        if progress:
            print(f"T{t}: {len(comp)} components against {circ.parent.nunique() if len(circ) else 0} "
                  f"droplets in {(time.perf_counter() - started) / 60:.1f} min", flush=True)

    result = _concat(frames, COMPONENT_COLUMNS)
    circles = _concat(circle_frames, CIRCLE_COLUMNS)
    # Everything the population inherits from the measurement steps. It goes into
    # the population tag, so two populations measured differently never share a folder.
    upstream = dict(geometry=geometry["signature_hash"],
                    components=signature_hash(dict(shared, per_t=hashes)),
                    timepoints=sorted(present),
                    gate_measurement=asdict(gate_settings),
                    assignment_min_containment=float(min_parent_overlap),
                    geometry_settings=dict(geometry["settings"]),
                    image=str(Path(image_path)), nuclear_masks=str(Path(labels_path)))
    return dict(components=result, circles=circles, upstream=upstream, cache_dir=cache_dir)


dg = SimpleNamespace(**{_k: globals()[_k] for _k in ('ArtifactGateSettings', 'CIRCLE_COLUMNS', 'COMPONENT_COLUMNS', 'COMPONENT_SCHEMA_VERSION', 'GEOMETRY_FUNCTION_NAMES', 'GEOMETRY_MANIFEST', 'TrainingGeometry', '_FROM_DROPLET_POPULATION', '_assign_timepoint', '_atomic_pickle', '_canonical', '_concat', '_file_identity', '_geometry_for_timepoint', '_geometry_worker', '_prepare_cache_dir', '_same_signature', 'assign_components', 'circle_crop', 'compute_geometry', 'find_geometry_caches', 'geometry_signature', 'included_planes', 'load_training_geometry', 'measure_segment_gate', 'plane_circles', 'signature_hash')})
print('dg: 27 names from inlined droplet_geometry')


## 11. Growth curve and phase velocities (definitions)

In [ ]:
# §11 GROWTH CURVE AND PHASE VELOCITIES (definitions).
# Median logistic A(t) = Amax / (1 + exp(-k (t - tm))), fit in log area with a near-L1 loss:
# the curve is the median nucleus area at each time (log is monotone) and robust to the low tail.
# Chosen over Gompertz / saturating exponential / power law / linear by AIC and leave-one-frame-out
# CV on the gold population (2026-11-04). Phase boundaries are the curvature extremes of the
# logistic: acceleration ends at tm − 1.317/k, deceleration starts at tm + 1.317/k.
from scipy.optimize import least_squares

LOGISTIC_P0 = (420.0, 0.08, 20.0)
LOGISTIC_BOUNDS = ([50, 1e-3, -60], [3000, 1, 120])
PHASE_NAMES = ("acceleration", "near-linear", "deceleration")


def logi(p, t):
    p = np.asarray(p, float)
    return p[..., 0, None] / (1 + np.exp(-p[..., 1, None] * (np.asarray(t, float) - p[..., 2, None]))) \
        if p.ndim > 1 else p[0] / (1 + np.exp(-p[1] * (np.asarray(t, float) - p[2])))


def fit_median_logistic(t, y, p0=LOGISTIC_P0):
    t, y = np.asarray(t, float), np.asarray(y, float)
    return least_squares(lambda p: np.log(logi(p, t)) - np.log(y), p0, bounds=LOGISTIC_BOUNDS,
                         loss="soft_l1", f_scale=0.02).x


def logistic_summary(p) -> dict:
    Amax, k, tm = p
    return {"Amax": Amax, "k": k, "tm": tm, "peak_rate": k * Amax / 4,
            "t_acc_end": tm - 1.317 / k, "t_dec_start": tm + 1.317 / k,
            "A(30)": float(logi(p, 30.)), "A(55)": float(logi(p, 55.))}


def boot_logistic(t, y, p, n=500, seed=0) -> np.ndarray:
    """Resample nuclei (one replicate). For several replicates use hier_boot_logistic."""
    t, y = np.asarray(t, float), np.asarray(y, float)
    rng = np.random.default_rng(seed)
    return np.array([fit_median_logistic(t[i], y[i], p)
                     for i in (rng.integers(0, len(t), len(t)) for _ in range(n))])


def hier_boot_logistic(reps: Dict[str, Tuple[np.ndarray, np.ndarray]], p0, n=500, seed=0) -> np.ndarray:
    """Two-level bootstrap: resample replicates, then nuclei within each chosen replicate."""
    rng, names, out = np.random.default_rng(seed), list(reps), []
    for _ in range(n):
        ts, ys = [], []
        for r in rng.choice(names, len(names), replace=True):
            t, y = reps[r]; i = rng.integers(0, len(t), len(t)); ts.append(t[i]); ys.append(y[i])
        out.append(fit_median_logistic(np.concatenate(ts), np.concatenate(ys), p0))
    return np.array(out)


def phase_windows(ref_p, t_obs=(6.0, 59.0), min_len=6.0) -> Dict[str, Optional[Tuple[float, float]]]:
    """Phase windows from a reference fit, clipped to the observed window; a phase shorter than
    one frame interval inside the window is reported as not observed (None)."""
    s = logistic_summary(ref_p)
    edges = [t_obs[0], s["t_acc_end"], s["t_dec_start"], t_obs[1]]
    out = {}
    for name, a, b in zip(PHASE_NAMES, edges[:-1], edges[1:]):
        a, b = max(a, t_obs[0]), min(b, t_obs[1])
        out[name] = (a, b) if b - a >= min_len else None
    return out


def phase_velocity_table(P: np.ndarray, windows: dict) -> pd.DataFrame:
    """Per parameter set (rows of P): logistic summary + mean absolute (µm²/min) and relative
    (1/min) velocity over each phase window."""
    rows = []
    for p in np.atleast_2d(P):
        s = logistic_summary(p)
        for ph, win in windows.items():
            if win:
                a, b = win; Aa, Ab = float(logi(p, a)), float(logi(p, b))
                s[f"abs vel {ph}"] = (Ab - Aa) / (b - a)
                s[f"rel vel {ph}"] = (np.log(Ab) - np.log(Aa)) / (b - a)
        rows.append(s)
    return pd.DataFrame(rows)


def frame_medians(t, y, frame_min: float, rng=None) -> Tuple[np.ndarray, np.ndarray]:
    """Median time and area per acquisition frame (optionally resampled within frames)."""
    t, y = np.asarray(t, float), np.asarray(y, float)
    f = np.floor(t / frame_min).astype(int)
    if rng is not None:
        idx = np.concatenate([rng.choice(np.flatnonzero(f == g), (f == g).sum()) for g in np.unique(f)])
        t, y, f = t[idx], y[idx], f[idx]
    m = pd.DataFrame({"f": f, "t": t, "y": y}).groupby("f").agg(t=("t", "median"), y=("y", "median"))
    return m.t.to_numpy(), m.y.to_numpy()


def model_free_phase_velocities(t, y, windows: dict, frame_min: float, n=500, seed=7) -> Dict[str, np.ndarray]:
    """Slope of frame medians (area and log area) inside each phase, bootstrapped within frames."""
    rng, out = np.random.default_rng(seed), {}
    for ph, win in windows.items():
        if not win:
            continue
        a, b = win; vals = []
        for _ in range(n):
            ft, fy = frame_medians(t, y, frame_min, rng)
            inside = (ft >= a - frame_min / 2) & (ft <= b + frame_min / 2)
            vals.append((np.polyfit(ft[inside], fy[inside], 1)[0], np.polyfit(ft[inside], np.log(fy[inside]), 1)[0])
                        if inside.sum() >= 2 else (np.nan, np.nan))
        out[ph] = np.array(vals)
    return out


def ci_text(s) -> str:
    s = pd.Series(s).dropna()
    return f"{s.median():.3g} [{s.quantile(.025):.3g}, {s.quantile(.975):.3g}]" if len(s) else "n/a"


def equivalence_margin_um2(area_um2: float, diameter_margin_um: float = 0.5) -> float:
    """Area margin equivalent to ±diameter_margin_um at the given area (circle)."""
    return np.pi / 2 * 2 * np.sqrt(area_um2 / np.pi) * diameter_margin_um


## 12. QC plotting helpers

In [ ]:
def plot_nucleus_mask_overlay_qc(
    img_5d: np.ndarray,
    nucleus_mask_4d: np.ndarray,
    config: PipelineConfig,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Overlay raw nuclear channel with the binary nucleus mask."""
    T, Z, C, Y, X = img_5d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()

    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, t_idx in zip(axes, timepoints):
        z_idx = (Z // 2 if z_mode == "middle"
                 else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1))))
        raw = img_5d[t_idx, z_idx, config.nuclear_channel_index].astype(np.float32)
        if raw.max() > 0:
            raw /= raw.max()
        ax.imshow(raw, cmap="gray")
        ax.imshow(nucleus_mask_4d[t_idx, z_idx].astype(bool), alpha=0.35)
        ax.set_title(f"t={t_idx}, z={z_idx}")
        ax.axis("off")

    for ax in axes[len(timepoints):]:
        ax.axis("off")

    fig.suptitle("QC: Nuclear channel + nucleus mask overlay", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_nucleus_mask_qc_summary(
    nucleus_mask_4d: np.ndarray,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Mask snapshots with contours + total nuclear area over time."""
    from skimage.measure import find_contours
    T, Z, Y, X = nucleus_mask_4d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()

    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig = plt.figure(figsize=(4 * n_cols, 4 * (n_rows + 1)))
    gs  = fig.add_gridspec(n_rows + 1, n_cols)

    panel_axes = [fig.add_subplot(gs[r, c]) for r in range(n_rows) for c in range(n_cols)]

    for ax, t_idx in zip(panel_axes, timepoints):
        z_idx = (Z // 2 if z_mode == "middle"
                 else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1))))
        mask_plane = nucleus_mask_4d[t_idx, z_idx].astype(bool)
        labeled    = measure.label(mask_plane)
        ax.imshow(mask_plane, cmap="gray")
        for contour in find_contours(mask_plane.astype(float), level=0.5):
            ax.plot(contour[:, 1], contour[:, 0], linewidth=0.5)
        ax.set_title(f"t={t_idx}, z={z_idx}\nobjects={labeled.max()}")
        ax.axis("off")

    for ax in panel_axes[len(timepoints):]:
        ax.axis("off")

    area_ax = fig.add_subplot(gs[n_rows, :])
    area_ax.plot(np.arange(T), nucleus_mask_4d.reshape(T, -1).sum(axis=1), marker="o")
    area_ax.set_xlabel("Time index")
    area_ax.set_ylabel("Total nucleus mask area (px)")
    area_ax.set_title("QC: Total segmented nuclear area over time")
    plt.tight_layout()
    plt.show()


def plot_nucleus_count_over_time(nucleus_mask_4d: np.ndarray) -> None:
    """Count connected nucleus objects across all Z slices per time point."""
    T, Z = nucleus_mask_4d.shape[:2]
    counts = [
        sum(measure.label(nucleus_mask_4d[t, z].astype(bool)).max() for z in range(Z))
        for t in range(T)
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(T), counts, marker="o")
    ax.set_xlabel("Time index")
    ax.set_ylabel("Connected nucleus objects")
    ax.set_title("QC: Connected nucleus object count over time")
    plt.tight_layout()
    plt.show()


def plot_nucleus_probability_qc(class_prob_map: np.ndarray, raw_plane: np.ndarray,
                                title: str = "") -> None:
    """Show raw plane, nucleus probability map, and its histogram."""
    nucleus_prob = class_prob_map[..., 2]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw_plane, cmap="gray"); axes[0].set_title("Raw"); axes[0].axis("off")
    im = axes[1].imshow(nucleus_prob, cmap="inferno")
    axes[1].set_title("Nucleus probability"); plt.colorbar(im, ax=axes[1])
    axes[2].hist(nucleus_prob.ravel(), bins=100); axes[2].set_title("Probability distribution")
    fig.suptitle(title); plt.tight_layout(); plt.show()


def plot_nc_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot."); return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nc_ratio_fraction"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("N / (N + C)")
    ax.set_title("N/C ratio fraction by track")
    plt.tight_layout(); plt.show()


def plot_area_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot."); return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nucleus_area_um2"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear area (µm²)")
    ax.set_title("Largest nuclear cross-sectional area by track")
    plt.tight_layout(); plt.show()


def plot_largest_cross_sectional_area_vs_time(
    df: pd.DataFrame,
    config: PipelineConfig,
) -> None:
    """
    Scatter all individual nuclear areas plus population mean vs true acquisition time.

    Works with timed_df, tracked_df, or halo_df.  Requires 'true_time_min'.
    """
    if df.empty:
        print("No data to plot."); return
    plot_df = df.copy()
    if "nucleus_area_um2" not in plot_df.columns:
        if "area_px" not in plot_df.columns:
            raise ValueError("DataFrame must contain 'nucleus_area_um2' or 'area_px'.")
        plot_df["nucleus_area_um2"] = plot_df["area_px"] * config.pixel_size_um ** 2
    if "true_time_min" not in plot_df.columns:
        raise ValueError("DataFrame must contain 'true_time_min'.")

    plot_df = plot_df.sort_values("true_time_min")
    mean_df = (plot_df.groupby("true_time_min", as_index=False)["nucleus_area_um2"]
               .mean().rename(columns={"nucleus_area_um2": "mean_area"}))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["true_time_min"], plot_df["nucleus_area_um2"], alpha=0.35, s=20,
               label="Individual nuclei")
    ax.plot(mean_df["true_time_min"], mean_df["mean_area"], linewidth=2, label="Population mean")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear cross-sectional area (µm²)")
    ax.set_title("Nuclear area vs time")
    ax.legend(); plt.tight_layout(); plt.show()


## 12b. Stage flags

In [ ]:
# ── Stage execution flags ───────────────────────────────────────────────
# Segmentation (§13) has no flag: it reads its checkpoint and decides fresh / resume /
# complete. Every stage below reloads its saved table when its flag is False.
SAVE_PROBABILITY_TIFF = True     # required by the reproducible baseline (§13)

RUN_OBJECT_EXTRACTION = True
RUN_POPULATION        = True     # §15 droplet population (cached by signature, quick on reruns)
RUN_SELECTION         = True
RUN_TIMING            = True     # cheap; also re-applies cfg.time_zero_offset_min
RUN_TRACKING          = True
RUN_HALO_ANALYSIS     = True
RUN_RADIAL_SWEEP      = True
RUN_GROWTH_ANALYSIS   = True
RUN_DB_SCHEMA_EXPORT  = True


## 12c. Initialise runtime and load the image

In [ ]:
configure_tensorflow_for_gpu(cfg)

print(f"Model path:      {cfg.model_path}")
print(f"Input image:     {cfg.input_image_path}")
print(f"Output dir:      {cfg.output_dir}")

assert cfg.model_path.exists(),       f"Missing model: {cfg.model_path}"
assert cfg.input_image_path.exists(), f"Missing image: {cfg.input_image_path}"

save_config(cfg, cfg.output_dir / "debug_pipeline_config.json")
print("Configuration saved.")

save_config(cfg, cfg.output_dir / "pipeline_config.json")


In [ ]:
t0 = time.perf_counter()
img_5d = load_image_5d(cfg.input_image_path)
print(f"Shape: {img_5d.shape}  |  dtype: {img_5d.dtype}  |  load time: {time.perf_counter() - t0:.2f}s")


## 13. Segmentation (or reuse of a finished segmentation)

In [ ]:
# §13 SEGMENTATION. State is decided from the on-disk checkpoint before any model load.
_T, _Z = img_5d.shape[0], img_5d.shape[1]
_Y, _X = img_5d.shape[-2], img_5d.shape[-1]

with segmentation_timer('Check saved segmentation state'):
    seg_state = assess_segmentation_state(cfg, (_T, _Z, _Y, _X),
                                          save_probability_tiff=SAVE_PROBABILITY_TIFF)
print(f"[segmentation state: {seg_state.mode}]")
print(seg_state.reason)

if seg_state.mode == "complete":
    seg_index_df = pd.read_pickle(cfg.segmentation_index_path)
    print("Segmentation complete — no model load.")
elif seg_state.mode == "inconsistent":
    raise RuntimeError(seg_state.reason)
elif cfg.segmentation_source_run is not None:
    raise RuntimeError("The source segmentation run is not complete; it is never written to from here. "
                       "Finish it in its own notebook, or set SEGMENTATION_SOURCE_RUN = None.")
elif seg_state.mode in ("fresh", "resume"):
    if seg_state.todo_t and globals().get('_LOADED_SEGMENTATION_MODEL_SHA') != cfg.model_sha256:
        with segmentation_timer('Load segmentation model'):
            model = load_unet_model(cfg.model_path)
        _LOADED_SEGMENTATION_MODEL_SHA = cfg.model_sha256
    t0 = time.perf_counter()
    seg_index_df = run_segmentation_for_all_planes(
        img_5d=img_5d, model=(model if seg_state.todo_t else None),
        config=cfg, state=seg_state, save_probability_tiff=SAVE_PROBABILITY_TIFF)
    print(f"Segmentation finished in {time.perf_counter() - t0:.2f}s")
else:
    raise RuntimeError(f"unhandled seg_state.mode={seg_state.mode!r}")

print(seg_index_df.shape)


In [ ]:
# -- Reset segmentation state (this run only) -------------------------------
# Uncomment ONE call, run it, then re-run §13. Refused when the segmentation is read from a
# source run: a source run is never modified from this notebook.
#   scope="all"     wipe checkpoint, _resume/, hyperstacks, per-plane masks, seg tables and the
#                   downstream object tables of this run.
#   scope="resume"  wipe only the checkpoint + _resume/ (forces a full recompute).
#
# reset_segmentation(cfg, scope="all")
# reset_segmentation(cfg, scope="resume")
print("Reset cell — edit to uncomment a reset_segmentation(...) call.")


## 14. Per-plane objects

In [ ]:
# §14 PER-PLANE OBJECTS: one row per nuclear instance (t, z, label) with area, centroid, bbox.
require_segmentation(cfg)
objects_path = cfg.obj_dir / "plane_objects.pkl"
if RUN_OBJECT_EXTRACTION or not objects_path.exists():
    objects_df = extract_objects_from_saved_masks(seg_index_df, cfg)
else:
    objects_df = pd.read_pickle(objects_path)
print(objects_df.shape)
display(objects_df.head())


## 15. Droplet population
Fitted droplet geometry, parent assignment, enrichment and solidity gate, and the approved 5 %
droplet-overlap exclusion, per component. Only components with status `accepted` go on to §16.

In [ ]:
# §15a DROPLET POPULATION SETTINGS. Not PipelineConfig fields: they are recorded in the
# population tag (hash) instead, so changing them does not move the run directory.
POP_SETTINGS = dpop.PopulationSettings(
    overlap_tolerance=0.05,                # approved: strict >, both circles observed
    min_enrichment=1.20,                   # existing gate
    min_solidity=0.80,                     # existing gate
    partial_parent_policy="unresolved",
    exclude_one_pixel_components=False,
)
GATE_MEASUREMENT = dg.ArtifactGateSettings(
    min_enrichment=POP_SETTINGS.min_enrichment, min_solidity=POP_SETTINGS.min_solidity,
    ring_gap_um=0.5, ring_width_um=2.0, min_ring_pixels=50)
ASSIGNMENT_MIN_CONTAINMENT = 0.85          # share of a component one circle must hold
POP_TIMEPOINTS = tuple(range(cfg.sel_min_timepoint, img_5d.shape[0]))

# Droplet geometry functions are read from the training notebook's definitions (parsed, never run).
TRAINING_NOTEBOOK = (Path.home() / "Projects/Nuclear_Scaling/Code_Repository/Python"
                     / "Model_Training" / "vulcan_training_2_5_3.ipynb")
GEOMETRY_WORKERS = 1        # >1 uses spawned processes, which cannot import notebook-defined functions
GEOMETRY_ON_ERROR = "raise"
assert GEOMETRY_WORKERS == 1, "Inlined library: keep GEOMETRY_WORKERS = 1"
assert TRAINING_NOTEBOOK.exists(), f"Training notebook not found: {TRAINING_NOTEBOOK}"
POP_ROOT = cfg.analysis_dir / "selected_population"
display(POP_SETTINGS.describe())
print("Timepoints:", POP_TIMEPOINTS)


In [ ]:
# §15b Droplet geometry (training-style sphere fits), all population timepoints, cached.
training_geometry = dg.load_training_geometry(TRAINING_NOTEBOOK, cfg.pixel_size_um, cfg.npc_channel_index)
_geometry_signature = dg.geometry_signature(training_geometry, cfg.input_image_path)
GEOMETRY_CACHE_DIR = POP_ROOT / "geometry" / ("sig-" + dg.signature_hash(_geometry_signature))
# Caches from earlier runs on the same image are reused when their provenance matches.
_older = [cfg.seg_dir.parent / "analysis" / "selected_population" / "geometry"]
_older_caches = pd.concat([dg.find_geometry_caches(p, _geometry_signature) for p in _older if p.exists()]
                          or [pd.DataFrame(columns=["cache_dir", "provenance_matches"])])
DROPLET_GEOMETRY = dg.compute_geometry(
    cfg.input_image_path, pixel_size_um=cfg.pixel_size_um, npc_channel_index=cfg.npc_channel_index,
    training=training_geometry, timepoints=POP_TIMEPOINTS, cache_dir=GEOMETRY_CACHE_DIR,
    reuse_cache_dirs=list(_older_caches.loc[_older_caches.provenance_matches.astype(bool), "cache_dir"]),
    n_workers=GEOMETRY_WORKERS, on_error=GEOMETRY_ON_ERROR)
display(DROPLET_GEOMETRY["coverage"])


In [ ]:
# §15c Parent assignment + enrichment/solidity on the fitted supports, per component.
# v19: there is no Z-linking. The library still carries a `nucleus_3d_id` column through its
# tables; it is filled with a unique id per component (not used by any decision).
pop_objects = objects_df[objects_df.t.isin(POP_TIMEPOINTS)].copy()
pop_objects["nucleus_3d_id"] = np.arange(1, len(pop_objects) + 1)
COMPONENT_MEASUREMENTS = dg.assign_components(
    cfg.input_image_path, cfg.nucleus_instance_hyperstack_path, pop_objects, DROPLET_GEOMETRY,
    nuclear_channel_index=cfg.nuclear_channel_index, pixel_size_um=cfg.pixel_size_um,
    gate_settings=GATE_MEASUREMENT, min_parent_overlap=ASSIGNMENT_MIN_CONTAINMENT,
    valid_planes=dg.included_planes(pd.read_pickle(cfg.segmentation_index_path), cfg.focus_min_z),
    cache_dir=POP_ROOT / "components" / ("geom-" + DROPLET_GEOMETRY["signature_hash"]))
display(pd.crosstab(COMPONENT_MEASUREMENTS["components"].t,
                    COMPONENT_MEASUREMENTS["components"].geometry_status, margins=True))


In [ ]:
# §15d Decisions for every component (accepted / excluded / unresolved), saved by tag.
POPULATION = dpop.build_population(
    COMPONENT_MEASUREMENTS["components"], COMPONENT_MEASUREMENTS["circles"],
    POP_SETTINGS, upstream=COMPONENT_MEASUREMENTS["upstream"])
POPULATION["output_dir"] = dpop.save_population(POPULATION, POP_ROOT / "populations")
print("Population", POPULATION["tag"], "->", POPULATION["output_dir"])
_c = POPULATION["components"]
display(pd.crosstab(_c.t, _c.status, margins=True))
print("Components per reason (a component with two reasons is counted under each):")
display(POPULATION["reasons"].pivot_table(index=["status", "reason"], columns="t",
                                           values="components", fill_value=0))


## 16. Single-nucleus selection
Edge → local area maximum (no Z-linking) → area floor → single-nucleus rule. Every candidate keeps
its rule flags in `sections_all.pkl`; `reason` is the first rule that removed it.

In [ ]:
# §16 SINGLE-NUCLEUS SELECTION (run). Inputs: per-plane objects (§14) and the droplet population
# decisions (§15). Writes every candidate with its rule flags, and the selected nuclei.
sections_path, nuclei_path = cfg.obj_dir / "sections_all.pkl", cfg.obj_dir / "nuclei_selected.pkl"
if RUN_SELECTION or not nuclei_path.exists():
    with segmentation_timer("Single-nucleus selection"):
        sections_df, nuclei_df = select_single_nuclei(
            objects_df, POPULATION["components"], *img_5d.shape[-2:], cfg)
    sections_df.to_pickle(sections_path); nuclei_df.to_pickle(nuclei_path)
else:
    sections_df, nuclei_df = pd.read_pickle(sections_path), pd.read_pickle(nuclei_path)

SELECTION_COUNTS = selection_rule_counts(sections_df)
print("Candidates per timepoint, by the FIRST rule that removed them:")
display(SELECTION_COUNTS)
print(f"Selected single nuclei: {len(nuclei_df)}")


In [ ]:
# QC, informational (re-run freely; changes nothing): what each rule removed, by area, and a
# gallery of examples per rule (NLS at the section's own plane; cyan = the section's mask).
_reasons = [r for r in ["edge", "not local max", "below area floor", "cluster: bilobed",
                        "cluster: touching", "selected"] if r in set(sections_df.reason)]
fig, ax = plt.subplots(figsize=(10, 4))
for r in _reasons:
    s = sections_df[sections_df.reason == r]
    ax.hist(s.area_um2, bins=np.linspace(0, 650, 66), histtype="step", lw=1.8, label=f"{r} (n={len(s)})")
ax.set(xlabel="section area (µm²)", ylabel="sections", title="Area of sections by selection outcome")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

GALLERY_PER_RULE, PAD = 6, 40
_inst = tiff.memmap(str(cfg.nucleus_instance_hyperstack_path), mode="r")
_H, _W = img_5d.shape[-2:]
fig, axs = plt.subplots(len(_reasons), GALLERY_PER_RULE, figsize=(2.6 * GALLERY_PER_RULE, 2.6 * len(_reasons)))
axs = np.atleast_2d(axs)
for row, r in zip(axs, _reasons):
    s = sections_df[sections_df.reason == r]
    s = s.sample(min(len(s), GALLERY_PER_RULE), random_state=0) if len(s) else s
    for ax in row: ax.axis("off")
    for ax, (_, q) in zip(row, s.iterrows()):
        y0, x0 = max(0, int(q.bbox_min_row) - PAD), max(0, int(q.bbox_min_col) - PAD)
        y1, x1 = min(_H, int(q.bbox_max_row) + PAD), min(_W, int(q.bbox_max_col) + PAD)
        im = np.asarray(img_5d[int(q.t), int(q.z), cfg.nuclear_channel_index, y0:y1, x0:x1], np.float32)
        ax.imshow(im, cmap="gray", vmin=np.percentile(im, 1), vmax=np.percentile(im, 99.5))
        m = np.asarray(_inst[int(q.t), int(q.z), y0:y1, x0:x1]) == int(q.label)
        for cc in measure.find_contours(m.astype(float), .5): ax.plot(cc[:, 1], cc[:, 0], c="cyan", lw=.8)
        ax.set_title(f"{r}\nt{int(q.t)} z{int(q.z)} {q.area_um2:.0f} µm²", fontsize=7)
plt.tight_layout(); plt.show()


## 17. True acquisition time

In [ ]:
# §17 TRUE ACQUISITION TIME for the selected nuclei (applies cfg.time_zero_offset_min).
timed_path = cfg.track_dir / "nuclei_timed.pkl"
if RUN_TIMING or not timed_path.exists():
    timed_df = add_tile_timing_metadata(nuclei_df, img_5d.shape[-2:], cfg, out_path=timed_path)
else:
    timed_df = pd.read_pickle(timed_path)
print(f"time offset: {cfg.time_zero_offset_min} min | frame interval: {cfg.frame_interval_min} min")
display(timed_df.groupby("t").true_time_min.agg(["min", "max", "count"]))


## 18. Tracking and halos

In [ ]:
# §18a TRACKING of the selected nuclei across timepoints (hybrid DBSCAN tracker).
tracked_path = cfg.track_dir / "tracked_nuclei.pkl"
if RUN_TRACKING or not tracked_path.exists():
    tracked_df = assign_track_ids_hybrid_dbscan(timed_df, cfg)
else:
    tracked_df = pd.read_pickle(tracked_path)
display(summarize_tracking_debug(tracked_df))


In [ ]:
# §18b HALOS around each selected nucleus at its selected section.
halo_path = cfg.analysis_dir / "halo_analysis.pkl"
if RUN_HALO_ANALYSIS or not halo_path.exists():
    halo_df = run_halo_analysis_for_tracked_nuclei(tracked_df, img_5d, cfg)
else:
    halo_df = pd.read_pickle(halo_path)
print(halo_df.shape)


## 19. Radial sweep

In [ ]:
# §19 RADIAL SWEEP (droplet-bounded) from each selected nucleus.
radial_sweep_path = cfg.analysis_dir / "radial_sweep.pkl"
if RUN_RADIAL_SWEEP or not radial_sweep_path.exists():
    radial_df = run_radial_sweep_for_tracked_nuclei(tracked_df, img_5d, cfg)
else:
    radial_df = pd.read_pickle(radial_sweep_path)
print(radial_df.shape)


## 20. Export the single-nucleus table

In [ ]:
# §20 EXPORT of the single-nucleus table for this replicate: what §21 and §25 read.
# Provenance travels with the table (model, config, selection rules, rule counts, time offset).
SINGLE_NUCLEI_DIR = cfg.exports_dir / "single_nuclei"
SINGLE_NUCLEI_DIR.mkdir(parents=True, exist_ok=True)
_cols = ["t", "z", "label", "nucleus_id", "track_id", "parent", "centroid_x_px", "centroid_y_px",
         "bbox_min_row", "bbox_min_col", "bbox_max_row", "bbox_max_col", "area_px", "area_um2",
         "solidity", "tile_index", "true_time_min"]
single_nuclei_df = tracked_df[[c for c in _cols if c in tracked_df.columns]].copy()
_save_portable_table(single_nuclei_df, SINGLE_NUCLEI_DIR / "single_nuclei")
_save_portable_table(SELECTION_COUNTS.reset_index(), SINGLE_NUCLEI_DIR / "selection_rule_counts")
(SINGLE_NUCLEI_DIR / "provenance.json").write_text(json.dumps(dict(
    created=datetime.now().isoformat(timespec="seconds"), run_id=run.run_id,
    pipeline_revision=cfg.pipeline_revision, model=cfg.model_name, model_sha256=cfg.model_sha256,
    input_image=str(cfg.input_image_path), segmentation_read_from=str(cfg.seg_dir.parent),
    time_zero_offset_min=cfg.time_zero_offset_min, population_tag=POPULATION["tag"],
    selection={k: getattr(cfg, k) for k in ("sel_min_timepoint", "sel_edge_px", "sel_window_planes",
                                            "sel_min_area_um2", "sel_min_solidity", "sel_touch_px",
                                            "focus_min_z")},
    n_nuclei=int(len(single_nuclei_df))), indent=2, default=str))
print(f"{len(single_nuclei_df)} nuclei -> {SINGLE_NUCLEI_DIR}")


## 21. Growth curve, phases and velocities (this replicate)
Median logistic in log area with a nuclei bootstrap. The acceleration phase usually ends before
imaging starts (≈ 8 min) and is then reported as not observed.

In [ ]:
# §21a GROWTH CURVE for this replicate: median logistic + bootstrap (nuclei resampled).
# Within one replicate the bootstrap is over nuclei; replicate-level uncertainty is in §25.
N_BOOT = 500
T_OBS = (float(np.floor(timed_df.true_time_min.min())), float(np.ceil(timed_df.true_time_min.max())))
GROWTH_DIR = cfg.exports_dir / "growth"; GROWTH_DIR.mkdir(parents=True, exist_ok=True)
gx = single_nuclei_df.true_time_min.to_numpy(float)
gy = single_nuclei_df.area_um2.to_numpy(float)
if RUN_GROWTH_ANALYSIS or not (GROWTH_DIR / "logistic_bootstrap.npy").exists():
    GROWTH_P = fit_median_logistic(gx, gy)
    GROWTH_BOOTS = boot_logistic(gx, gy, GROWTH_P, n=N_BOOT)
    np.save(GROWTH_DIR / "logistic_bootstrap.npy", GROWTH_BOOTS)
    np.save(GROWTH_DIR / "logistic_fit.npy", GROWTH_P)
else:
    GROWTH_P = np.load(GROWTH_DIR / "logistic_fit.npy"); GROWTH_BOOTS = np.load(GROWTH_DIR / "logistic_bootstrap.npy")

GROWTH_PHASES = phase_windows(GROWTH_P, t_obs=T_OBS, min_len=cfg.frame_interval_min)
GROWTH_TABLE = phase_velocity_table(GROWTH_BOOTS, GROWTH_PHASES)
print("observed window:", T_OBS, "| phase windows (min):",
      {k: (None if v is None else tuple(round(x, 1) for x in v)) for k, v in GROWTH_PHASES.items()})
growth_summary = pd.Series({k: ci_text(v) for k, v in GROWTH_TABLE.items()}, name="median [95% CI]")
display(growth_summary.to_frame())
growth_summary.to_csv(GROWTH_DIR / "logistic_summary.csv")


In [ ]:
# §21b FIGURES: area with the median logistic and its 95 % band; absolute and relative growth
# velocity with phase shading; model-free frame-median velocities as a check on the curve shape.
tt = np.linspace(*T_OBS, 300)
curves = logi(GROWTH_BOOTS, tt)
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(gx, gy, s=10, c="teal", alpha=.35, label=f"single nuclei (n={len(gx)})")
ax.fill_between(tt, *np.quantile(curves, [.025, .975], axis=0), color="teal", alpha=.2)
ax.plot(tt, logi(GROWTH_P, tt), c="teal", lw=2.5, label="median logistic (95% band)")
_s = logistic_summary(GROWTH_P)
for tb, lab in ((_s["t_acc_end"], "acceleration ends"), (_s["tm"], "fastest growth (tm)"),
                (_s["t_dec_start"], "deceleration starts")):
    if T_OBS[0] <= tb <= T_OBS[1]:
        ax.axvline(tb, c="0.4", ls="--", lw=1)
        ax.text(tb, ax.get_ylim()[1] * .97, f" {lab}\n {tb:.0f} min", fontsize=8, va="top")
ax.set(xlabel="Acquisition time (minutes)", ylabel="Maximum nuclear cross-sectional area (µm²)",
       title="Median logistic growth — selected single nuclei")
ax.grid(alpha=.2); ax.legend(loc="lower right"); plt.tight_layout()
fig.savefig(GROWTH_DIR / "area_logistic.png", dpi=300); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(16, 5))
ft, fy = frame_medians(gx, gy, cfg.frame_interval_min)
mid = (ft[1:] + ft[:-1]) / 2
for ax, g, fm, ttl, yl in ((axs[0], np.gradient(curves, tt, axis=1), np.diff(fy) / np.diff(ft),
                            "absolute growth velocity", "dA/dt (µm²/min)"),
                           (axs[1], np.gradient(np.log(curves), tt, axis=1), np.diff(np.log(fy)) / np.diff(ft),
                            "relative growth velocity", "d lnA/dt (1/min)")):
    for (ph, win), shade in zip(GROWTH_PHASES.items(), ("#e8f0ff", "#fff3e0", "#eef7ee")):
        if win: ax.axvspan(*win, color=shade, zorder=0)
    ax.fill_between(tt, *np.quantile(g, [.025, .975], axis=0), color="teal", alpha=.2)
    ax.plot(tt, np.median(g, axis=0), c="teal", lw=2, label="fit")
    ax.plot(mid, fm, "o", c="teal", ms=5, label="frame medians")
    ax.set(title=ttl, xlabel="Acquisition time (minutes)", ylabel=yl); ax.grid(alpha=.2)
axs[0].legend(); plt.tight_layout(); fig.savefig(GROWTH_DIR / "velocity.png", dpi=300); plt.show()


In [ ]:
# §21c PHASE VELOCITIES: fitted (from the bootstrap) and model-free (frame medians, bootstrapped
# within frames). If the two disagree in a phase, the logistic misdescribes that phase there.
_free = model_free_phase_velocities(gx, gy, GROWTH_PHASES, cfg.frame_interval_min)
rows = []
for ph, win in GROWTH_PHASES.items():
    if win is None:
        rows.append(dict(phase=ph, window="not observed")); continue
    rows.append({"phase": ph, "window": f"{win[0]:.0f}–{win[1]:.0f} min",
                 "abs vel fit (µm²/min)": ci_text(GROWTH_TABLE[f"abs vel {ph}"]),
                 "abs vel model-free": ci_text(_free[ph][:, 0]),
                 "rel vel fit (1/min)": ci_text(GROWTH_TABLE[f"rel vel {ph}"]),
                 "rel vel model-free": ci_text(_free[ph][:, 1])})
PHASE_VELOCITIES = pd.DataFrame(rows).set_index("phase")
display(PHASE_VELOCITIES)
PHASE_VELOCITIES.to_csv(GROWTH_DIR / "phase_velocities.csv")


## 22. Figures — membrane and envelope plot set

Everything below runs off the selected single nuclei (§16): `halo_sel`, `timed_sel`, `tracked_sel`
and `radial_sel` are the §18–19 tables, bound in the next cell. Figures are written to
`<run>/exports/figures_population/figures/`.

In [ ]:
# §22 Bind the tables the figure sections plot. In v19 the tables ARE the selected population
# (single nuclei, §16), so no further population filter is applied here.
for _name, _path in [("halo_df", cfg.analysis_dir / "halo_analysis.pkl"),
                     ("timed_df", cfg.track_dir / "nuclei_timed.pkl"),
                     ("tracked_df", cfg.track_dir / "tracked_nuclei.pkl"),
                     ("radial_df", cfg.analysis_dir / "radial_sweep.pkl")]:
    if _name not in globals():
        globals()[_name] = pd.read_pickle(_path); print(f"Loaded {_name} from disk.")
halo_sel, timed_sel, tracked_sel, radial_sel = halo_df, timed_df, tracked_df, radial_df
ANALYSIS_POPULATION = "single_nuclei_v19"
POP_LABEL = f"single nuclei (v19 selection), population {POPULATION['tag'] if 'POPULATION' in globals() else ''}"
POP_EXPORT_DIR = cfg.exports_dir / "figures_population"
POP_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Figures use:", POP_LABEL, "| output:", POP_EXPORT_DIR)
print({k: len(v) for k, v in dict(halo=halo_sel, timed=timed_sel, tracked=tracked_sel, radial=radial_sel).items()})


In [ ]:
# ── Section 36 parameters ─────────────────────────────────────────────────
# Peak search is capped at this fraction of the nucleus->wall gap. See 36b for
# why: the membrane channel also images the droplet interface, which outshines
# the NE. Raise toward 1.0 only after checking frac_at_search_edge.
ENVELOPE_SEARCH_RHO_MAX = 0.6

# Perinuclear shell used for the asymmetry metrics, in microns outward from the
# nuclear mask edge.
ASYMMETRY_BAND_UM = (0.0, 3.0)

# Width of the acquisition-time bins for the box plots, in minutes.
# 2x3 tiles at 1 min/tile => 6 min per frame, so 6.0 == one frame per box.
AREA_BIN_MIN = 6.0


In [ ]:
from typing import Sequence  # not in the Section 1 import block

SAVE_FIGURES = False
FIG_FORMATS = ("png", "pdf")
FIG_DPI = 300


def figures_dir(config) -> Path:
    # Follows the population bound in the previous cell (POP_EXPORT_DIR), so a
    # selected population's figures never overwrite the baseline's.
    d = POP_EXPORT_DIR / "figures"
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_figure(fig, name: str, config, formats=FIG_FORMATS, dpi: int = FIG_DPI) -> List[Path]:
    """Write a figure to <population folder>/figures/<name>.<ext> for each format."""
    if not SAVE_FIGURES:
        return []
    out = []
    d = figures_dir(config)
    for ext in formats:
        p = d / f"{name}.{ext}"
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        out.append(p)
    return out


def resolve_area_um2(df: pd.DataFrame, config, out_col: str = "nucleus_area_um2") -> pd.DataFrame:
    """Guarantee an area column in um^2, recomputed from pixels when possible.

    Recomputing from *_area_px means the figure always reflects the live
    pixel_size_um rather than whatever calibration was in force when the
    table was pickled.
    """
    df = df.copy()
    px2 = config.pixel_size_um ** 2
    if "nucleus_area_px" in df.columns:
        df[out_col] = df["nucleus_area_px"] * px2
    elif "area_px" in df.columns:
        df[out_col] = df["area_px"] * px2
    elif out_col in df.columns:
        pass
    else:
        raise ValueError("need one of nucleus_area_px / area_px / " + out_col)
    return df


def attach_true_time(df: pd.DataFrame, tracked_df: pd.DataFrame) -> pd.DataFrame:
    """Merge true_time_min onto a table keyed by (t, track_id).

    radial_df carries only (t, track_id, ...) -- acquisition time lives on
    tracked_df / timed_df.
    """
    if "true_time_min" in df.columns:
        return df
    if tracked_df is None or "true_time_min" not in tracked_df.columns:
        out = df.copy()
        out["true_time_min"] = out["t"].astype(float)
        return out
    lut = (tracked_df[["t", "track_id", "true_time_min"]]
           .drop_duplicates(subset=["t", "track_id"]))
    return df.merge(lut, on=["t", "track_id"], how="left")


def _time_bins(time_min: pd.Series, bin_min: float) -> Tuple[pd.Series, np.ndarray]:
    """Left edge of each observation's bin, plus the sorted unique edges."""
    lo = np.floor(np.asarray(time_min, dtype=float) / bin_min) * bin_min
    return pd.Series(lo, index=time_min.index), np.unique(lo[np.isfinite(lo)])


def _close_polar(theta: np.ndarray, r: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Repeat the first point at theta+2pi so a polar curve closes."""
    theta = np.asarray(theta, dtype=float)
    r = np.asarray(r, dtype=float)
    if theta.size == 0:
        return theta, r
    return np.append(theta, theta[0] + 2 * np.pi), np.append(r, r[0])


def _orient_polar(ax) -> None:
    """Put a polar axis in the same frame as the displayed image.

    The sweep casts rays as y = cy + r*sin(theta), x = cx + r*cos(theta). Image
    y increases DOWNWARD, so theta increases clockwise on screen and theta=0
    points right (+x). Matplotlib's polar default is zero at east but
    counter-clockwise, which mirrors the nucleus about the horizontal axis --
    fine for a shape summary, wrong the moment you compare a rose against a
    raw crop or against phi_deg from the asymmetry table.
    """
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(-1)


def _t_colors(all_t: Sequence) -> Dict:
    cmap = plt.get_cmap("viridis")
    n = max(len(all_t) - 1, 1)
    return {t: cmap(i / n) for i, t in enumerate(sorted(all_t))}


### 36a. Radius coordinate — `rho_wall`

`radial_profile_for_tracked_nucleus` writes `distance_norm = distance_px /
ray_length_px`, which is a fraction of the **centroid**-to-wall distance. Since
`radial_exclude_nucleus=True` drops interior samples, the smallest value on a
ray is `R_nucleus / R_droplet` — around 0.17 on this dataset, not 0. Plotting it
on an axis labelled *nucleus edge → droplet edge* therefore mislabels the origin,
and worse, the same `distance_norm` value means a different physical distance
from the envelope for a small nucleus than for a large one, so binning it pools
non-comparable points as nuclei grow.

`add_wall_normalised_radius` recomputes the schema's `Rho_Normalized` per ray —
0 at the nuclear surface, 1 at the wall — and also adds `r_um` and
`r_from_edge_um` for work in absolute microns. It does not modify
`radial_sweep.pkl`; it returns an augmented copy.

In [ ]:
def add_wall_normalised_radius(radial_df: pd.DataFrame, config,
                               out_col: str = "rho_wall") -> pd.DataFrame:
    """Add a radius coordinate that runs 0 at the nuclear surface -> 1 at the wall.

    `distance_norm` as written by the sweep is distance_px / ray_length_px, i.e.
    a fraction of the CENTROID-to-wall distance. Because the sweep skips samples
    inside the nucleus, the smallest value on a ray is R_nucleus / R_droplet
    (~0.2-0.5), not 0 -- so a plot with `distance_norm` on the x axis labelled
    "nucleus edge -> droplet edge" is mis-labelled, and the effective bin width
    differs between a big nucleus and a small one.

    This recomputes the schema's Rho_Normalized per ray:

        rho_wall = (r - r_edge) / (r_wall - r_edge)

    r_edge is the last sample inside the nucleus when the sweep kept interior
    points, otherwise the first retained sample (which sits just outside the
    mask boundary). Also adds r_um and r_from_edge_um for absolute-distance work.
    """
    if radial_df is None or radial_df.empty:
        return radial_df

    df = radial_df.copy()
    keys = ["t", "track_id", "theta_index"]
    g = df.groupby(keys)["distance_px"]

    r_first = g.transform("min")
    if "inside_nucleus" in df.columns and bool(df["inside_nucleus"].any()):
        inside_d = df["distance_px"].where(df["inside_nucleus"])
        r_edge = inside_d.groupby([df[k] for k in keys]).transform("max")
        r_edge = r_edge.fillna(r_first)
    else:
        r_edge = r_first

    r_wall = (df["ray_length_px"] if "ray_length_px" in df.columns
              else g.transform("max"))

    gap = (r_wall - r_edge).replace(0, np.nan)
    df[out_col] = ((df["distance_px"] - r_edge) / gap).clip(0.0, 1.0)
    df["r_um"] = df["distance_px"] * config.pixel_size_um
    df["r_edge_um"] = r_edge * config.pixel_size_um
    df["r_from_edge_um"] = df["r_um"] - df["r_edge_um"]
    df["theta_deg"] = np.degrees(df["theta_rad"]) % 360.0
    return df


### 36b. Rose plots — envelope radius by angle

For each ray, the radius at which the NE channel peaks — a mask-independent
estimate of nuclear shape, and the one that shows whether growth is isotropic.

**The peak search is capped** at `rho_wall <= search_rho_max` (default 0.6).
The radial channel is Ch0, which images the droplet oil–water interface as well
as the envelope, and on this rig the interface is usually the brightest thing on
the ray. An uncapped `argmax` therefore returns the droplet boundary for any
nucleus whose NE signal is weaker than its own droplet edge, and the rose plot
becomes a picture of the droplet — which is what the earlier scratch cell was
showing when its peak-radius histogram sat near the droplet radius instead of
the expected nuclear radius.

Watch `frac_at_search_edge` in the printout. If it's high, the cap rather than
the data is picking the peak and you should look at a radial profile directly
before trusting the rose.

Polar axes use `_orient_polar`: zero at east, clockwise. The sweep casts rays
with `y = cy + r·sin θ` and image y increases downward, so θ runs clockwise on
screen — matplotlib's counter-clockwise default silently mirrors the nucleus.

In [ ]:
def compute_envelope_radius_by_angle(
    radial_df: pd.DataFrame,
    config,
    search_rho_max: float = 0.6,
    verbose: bool = True,
) -> pd.DataFrame:
    """Per (t, track_id, ray): the radius at which the NE channel peaks.

    The search is restricted to rho_wall <= search_rho_max. The radial channel
    is Ch0, which images the droplet oil-water interface as well as the nuclear
    envelope, and the interface is usually the brightest thing on the ray -- an
    unrestricted argmax returns the droplet wall for any nucleus whose NE signal
    is weaker than its own droplet boundary, which turns the rose plot into a
    picture of the droplet. Pass search_rho_max=1.0 for the unrestricted
    behaviour, and watch `frac_at_search_edge` in the printout: a high value
    means the cap is doing the deciding, not the data.
    """
    cols = ["t", "track_id", "theta_index", "theta_rad", "peak_r_um",
            "peak_intensity", "rho_at_peak", "at_search_edge"]
    if radial_df is None or radial_df.empty:
        return pd.DataFrame(columns=cols)

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    search = df[df["rho_wall"] <= float(search_rho_max)]
    if search.empty:
        return pd.DataFrame(columns=cols)

    idx = search.groupby(["t", "track_id", "theta_index"])["intensity"].idxmax()
    peaks = search.loc[idx, ["t", "track_id", "theta_index", "theta_rad",
                             "distance_px", "intensity", "rho_wall"]].copy()
    peaks = peaks.rename(columns={"intensity": "peak_intensity",
                                  "rho_wall": "rho_at_peak"})
    peaks["peak_r_um"] = peaks["distance_px"] * config.pixel_size_um
    peaks["at_search_edge"] = peaks["rho_at_peak"] >= 0.98 * float(search_rho_max)
    peaks = peaks.drop(columns="distance_px").reset_index(drop=True)[cols]

    if verbose:
        frac = float(peaks["at_search_edge"].mean())
        print(f"envelope peaks: {len(peaks)} rays, "
              f"search_rho_max={search_rho_max}, "
              f"frac_at_search_edge={frac:.3f}")
        print(peaks.groupby("t")["peak_r_um"].median().round(2).to_string())
    return peaks


def plot_rose_individual(peak_df: pd.DataFrame, tracked_df: pd.DataFrame, config,
                         n: int = 5, selection: str = "longest",
                         share_r: bool = True, save_as: Optional[str] = "rose_individual"):
    """Polar envelope-shape plot for n example nuclei, one curve per timepoint."""
    if peak_df is None or peak_df.empty:
        print("No data to plot."); return []

    if selection == "longest":
        counts = (tracked_df.groupby("track_id")["t"].nunique()
                  .sort_values(ascending=False))
        track_ids = [tid for tid in counts.index if tid in set(peak_df["track_id"])][:n]
    else:
        track_ids = sorted(peak_df["track_id"].unique())[:n]
    if not track_ids:
        print("No tracks in common between peak_df and tracked_df."); return []

    n_cols = min(5, len(track_ids))
    n_rows = math.ceil(len(track_ids) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.6 * n_cols, 3.9 * n_rows),
                             subplot_kw={"projection": "polar"}, squeeze=False)
    axes = axes.ravel()
    colors = _t_colors(peak_df["t"].unique())
    r_max = float(peak_df[peak_df.track_id.isin(track_ids)]["peak_r_um"].quantile(0.99))

    for ax, tid in zip(axes, track_ids):
        d = peak_df[peak_df.track_id == tid]
        for t in sorted(d["t"].unique()):
            dt = d[d.t == t].sort_values("theta_index")
            th, r = _close_polar(dt.theta_rad.to_numpy(), dt.peak_r_um.to_numpy())
            ax.plot(th, r, color=colors[t], lw=1.2, label=f"t={t}")
        ax.set_title(f"track {tid}", fontsize=10)
        _orient_polar(ax)
        ax.tick_params(labelsize=7)
        if share_r:
            ax.set_ylim(0, r_max)

    for ax in axes[len(track_ids):]:
        ax.axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right", fontsize=7,
               title="frame", title_fontsize=7)
    fig.suptitle("Envelope radius by angle (\u00b5m) \u2014 individual nuclei", fontsize=13)
    fig.tight_layout(rect=(0, 0, 0.90, 1))
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return track_ids


def plot_rose_pooled(peak_df: pd.DataFrame, config, show_iqr: bool = True,
                     min_rays: int = 5, save_as: Optional[str] = "rose_pooled"):
    """Population-pooled rose: radius = median peak_r_um over nuclei per (t, theta)."""
    if peak_df is None or peak_df.empty:
        print("No data to plot."); return

    pooled = (peak_df.groupby(["t", "theta_index"])
              .agg(theta_rad=("theta_rad", "first"),
                   med=("peak_r_um", "median"),
                   q1=("peak_r_um", lambda s: s.quantile(0.25)),
                   q3=("peak_r_um", lambda s: s.quantile(0.75)),
                   n=("peak_r_um", "size"))
              .reset_index())
    pooled = pooled[pooled["n"] >= min_rays]
    if pooled.empty:
        print(f"No (t, theta) bin reached min_rays={min_rays}."); return

    fig, ax = plt.subplots(figsize=(6.4, 6.4), subplot_kw={"projection": "polar"})
    colors = _t_colors(pooled["t"].unique())
    for t in sorted(pooled["t"].unique()):
        dt = pooled[pooled.t == t].sort_values("theta_index")
        th, r = _close_polar(dt.theta_rad.to_numpy(), dt["med"].to_numpy())
        ax.plot(th, r, color=colors[t], lw=1.6, label=f"t={t}")
        if show_iqr:
            _, lo = _close_polar(dt.theta_rad.to_numpy(), dt["q1"].to_numpy())
            th2, hi = _close_polar(dt.theta_rad.to_numpy(), dt["q3"].to_numpy())
            ax.fill_between(th2, lo, hi, color=colors[t], alpha=0.12, lw=0)
    _orient_polar(ax)
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.05, 1.05), title="frame")
    ax.set_title("Envelope radius by angle (\u00b5m)\nmedian \u00b1 IQR, pooled over nuclei",
                 fontsize=12, pad=18)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return pooled


### 36c. Membrane intensity by angle and distance

The θ × r intensity map, one panel per frame: the raw evidence behind both the
rose and the asymmetry index. A complete envelope is a continuous bright band at
r ≈ 0 spanning all 360°; a partial one is a band with gaps; directional delivery
is a band that is brighter over a contiguous arc.

`radius_col="rho_wall"` spans the full gap and shows the droplet wall as a
second bright band at r = 1. Because the wall usually outshines the envelope, it
compresses the NE contrast — pass `radius_col="r_from_edge_um"` with
`r_max_um≈8` to crop to the perinuclear zone, which is the more readable view
for membrane delivery.

Empty (θ, r) cells are drawn in grey rather than interpolated, and the function
warns when more than 5 % of the grid is empty — that means `rbins` is finer than
the ray sampling supports.

In [ ]:
def build_angle_distance_grid(radial_df: pd.DataFrame, config, rbins: int = 60,
                              radius_col: str = "rho_wall",
                              r_max_um: Optional[float] = None,
                              agg: str = "mean") -> Dict:
    """theta x radius intensity grids, one per timepoint.

    radius_col='rho_wall'        -> 0 at the nuclear surface, 1 at the droplet wall
    radius_col='r_from_edge_um'  -> absolute microns outward from the nuclear surface
    """
    if radial_df is None or radial_df.empty:
        return {}
    df = radial_df
    if radius_col not in df.columns:
        df = add_wall_normalised_radius(df, config)

    if radius_col == "rho_wall":
        edges = np.linspace(0.0, 1.0, rbins + 1)
    else:
        hi = float(r_max_um if r_max_um is not None
                   else np.nanpercentile(df[radius_col], 99))
        edges = np.linspace(0.0, hi, rbins + 1)

    d = df.copy()
    d["rbin"] = pd.cut(d[radius_col], edges, labels=False, include_lowest=True)
    d = d[d["rbin"].notna()]

    n_ang = int(config.radial_n_angles)
    grids = {}
    for t, sub in d.groupby("t"):
        grids[int(t)] = (sub.pivot_table(index="theta_index", columns="rbin",
                                         values="intensity", aggfunc=agg)
                         .reindex(index=range(n_ang), columns=range(rbins))
                         .to_numpy())
    return {"grids": grids, "edges": edges, "radius_col": radius_col}


def plot_membrane_angle_distance(radial_df: pd.DataFrame, config,
                                 track_id: Optional[int] = None,
                                 rbins: int = 60,
                                 radius_col: str = "rho_wall",
                                 r_max_um: Optional[float] = None,
                                 clip_pct: Tuple[float, float] = (2, 98),
                                 cmap: str = "inferno",
                                 max_cols: int = 5,
                                 save_as: Optional[str] = None):
    """Membrane intensity as a function of angle and distance, one panel per frame.

    track_id=None pools every nucleus in absolute lab angle. Pooling is only
    meaningful if asymmetry shares a direction across nuclei -- check panel (c)
    of the asymmetry figure first; if the population direction is random, the
    pooled map washes out and the single-nucleus view is the informative one.
    """
    if radial_df is None or radial_df.empty:
        print("No data to plot."); return

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    if track_id is None:
        span = df.groupby("track_id")["t"].nunique().sort_values(ascending=False)
        label = "pooled over all nuclei"
        sub = df
    else:
        sub = df[df.track_id == int(track_id)]
        label = f"track {int(track_id)}"
        if sub.empty:
            print(f"track {track_id} not present in radial_df."); return

    built = build_angle_distance_grid(sub, config, rbins=rbins,
                                      radius_col=radius_col, r_max_um=r_max_um)
    grids, edges = built["grids"], built["edges"]
    if not grids:
        print("No data to plot."); return

    stacked = np.concatenate([g.ravel() for g in grids.values()])
    allv = stacked[np.isfinite(stacked)]
    if allv.size == 0:
        print("Grid is entirely empty."); return
    vmin, vmax = np.percentile(allv, clip_pct)

    empty_frac = 1.0 - allv.size / stacked.size
    if empty_frac > 0.05:
        print(f"WARNING: {empty_frac:.1%} of (theta, r) cells have no sample. "
              f"Lower rbins (currently {rbins}) or raise config.radial_step_size_px "
              f"resolution -- empty cells are drawn in grey, not interpolated.")

    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad("0.88")

    ts = sorted(grids)
    ncol = min(max_cols, len(ts))
    nrow = math.ceil(len(ts) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow),
                             squeeze=False, constrained_layout=True)
    xlabel = ("r (nuclear surface \u2192 droplet wall)" if radius_col == "rho_wall"
              else "distance from nuclear surface (\u00b5m)")
    for ax, t in zip(axes.ravel(), ts):
        im = ax.imshow(np.ma.masked_invalid(grids[t]), aspect="auto", origin="lower",
                       extent=[edges[0], edges[-1], 0, 360],
                       vmin=vmin, vmax=vmax, cmap=cmap_obj)
        ax.set_title(f"t={t}", fontsize=9)
        ax.set_xlabel(xlabel, fontsize=7)
        ax.set_ylabel("\u03b8 (deg)", fontsize=7)
        ax.set_yticks([0, 90, 180, 270, 360])
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[len(ts):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.75, label="membrane intensity (a.u.)")
    fig.suptitle(f"Membrane intensity by angle and distance \u2014 {label}", fontsize=12)
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return built


### 36d. Membrane asymmetry vs time

`I(θ)` is the mean membrane intensity in a shell just outside the nuclear
surface (default 0–3 µm from the mask edge). Fitting the first Fourier harmonic

$$I(\theta) \approx A_0 + A_1\cos(\theta - \phi)$$

gives the headline metric **`asym_index` = A₁/A₀** — 0 for a uniform shell,
rising as the signal becomes one-sided — and **`phi_deg`**, the direction of the
bright side. `cv` and `lobe_ratio` are reported alongside because a single
harmonic is blind to two-lobed asymmetry; if `cv` climbs while `asym_index` does
not, the shell is patchy rather than polarised.

**Panel (c) is the control to read first.** It asks whether every nucleus is
bright on the *same* side of the image, via the circular resultant `R` of the
population's `phi` values against a Rayleigh threshold. Stochastic
dynein-mediated delivery should be strong per nucleus and randomly oriented
across them, giving R near zero. R above the threshold means the bright side has
a preferred lab-frame direction — which is the signature of an illumination
gradient, a tilted coverslip, or a systematic z-offset, not of biology. Rule
that out before interpreting panel (a).

A₁/A₀ > 1 is possible and means the harmonic fit implies negative intensity
somewhere — read it as "essentially all the signal is on one side".

In [ ]:
def _first_harmonic(theta: np.ndarray, values: np.ndarray) -> Tuple[float, float, float]:
    """Fit v(theta) ~ a0 + a1*cos(theta - phi); return (a0, a1, phi).

    phi is the angle of maximum signal. Uniform theta sampling assumed.
    """
    theta = np.asarray(theta, dtype=float)
    v = np.asarray(values, dtype=float)
    ok = np.isfinite(v) & np.isfinite(theta)
    if ok.sum() < 8:
        return np.nan, np.nan, np.nan
    theta, v = theta[ok], v[ok]
    a0 = float(v.mean())
    c1 = np.mean(v * np.exp(-1j * theta))
    return a0, float(2 * np.abs(c1)), float(-np.angle(c1))


def compute_membrane_asymmetry(radial_df: pd.DataFrame, config,
                               tracked_df: Optional[pd.DataFrame] = None,
                               band: Tuple[float, float] = (0.0, 3.0),
                               band_units: str = "um",
                               min_rays: int = 30) -> pd.DataFrame:
    """Per (t, track_id) asymmetry of the perinuclear membrane signal.

    The signal is I(theta) = mean intensity in a shell just outside the nuclear
    surface -- by default 0-3 microns from the mask edge, which is where
    delivered membrane accumulates. Set band_units='rho' to use a fraction of
    the nucleus-to-wall gap instead (scale-free but droplet-size dependent).

    Columns
    -------
    a0              mean perinuclear intensity around the nucleus
    a1              amplitude of the first Fourier harmonic
    asym_index      a1 / a0, the headline number: 0 = uniform shell,
                    ~0.3+ = clearly one-sided
    phi_deg         direction of the bright side, degrees (image frame, CCW from +x)
    cv              std/mean across angle -- catches multi-lobed asymmetry that
                    a single harmonic misses
    lobe_ratio      brightest decile of angles / dimmest decile
    n_rays          rays contributing
    """
    empty = pd.DataFrame(columns=["t", "track_id", "a0", "a1", "asym_index",
                                  "phi_rad", "phi_deg", "cv", "lobe_ratio",
                                  "n_rays", "true_time_min"])
    if radial_df is None or radial_df.empty:
        return empty

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    lo, hi = float(band[0]), float(band[1])
    col = "r_from_edge_um" if band_units == "um" else "rho_wall"
    shell = df[(df[col] >= lo) & (df[col] <= hi)]
    if shell.empty:
        print(f"No samples in band {band} ({band_units}).")
        return empty

    per_ray = (shell.groupby(["t", "track_id", "theta_index"])
               .agg(theta_rad=("theta_rad", "first"), I=("intensity", "mean"))
               .reset_index())

    rows = []
    for (t, tid), d in per_ray.groupby(["t", "track_id"]):
        if len(d) < min_rays:
            continue
        d = d.sort_values("theta_index")
        th, I = d.theta_rad.to_numpy(), d.I.to_numpy()
        a0, a1, phi = _first_harmonic(th, I)
        k = max(1, len(I) // 10)
        srt = np.sort(I)
        dim = float(srt[:k].mean())
        rows.append({
            "t": int(t), "track_id": int(tid),
            "a0": a0, "a1": a1,
            "asym_index": (a1 / a0) if (np.isfinite(a0) and a0 > 0) else np.nan,
            "phi_rad": phi, "phi_deg": float(np.degrees(phi) % 360.0),
            "cv": float(np.std(I) / a0) if a0 else np.nan,
            "lobe_ratio": float(srt[-k:].mean() / dim) if dim > 0 else np.nan,
            "n_rays": int(len(I)),
        })

    asym_df = pd.DataFrame(rows)
    if asym_df.empty:
        return empty
    return attach_true_time(asym_df, tracked_df).sort_values(["t", "track_id"])


def _circular_stats(phi_rad: np.ndarray) -> Tuple[float, float, float]:
    """Mean direction, resultant length R, and Rayleigh p for a set of angles."""
    phi = np.asarray(phi_rad, dtype=float)
    phi = phi[np.isfinite(phi)]
    n = phi.size
    if n == 0:
        return np.nan, np.nan, np.nan
    C = np.mean(np.exp(1j * phi))
    R = float(np.abs(C))
    p = float(np.exp(-n * R ** 2))          # Rayleigh test, large-n approximation
    return float(np.angle(C)), R, p


def plot_membrane_asymmetry_vs_time(asym_df: pd.DataFrame, config,
                                    metric: str = "asym_index",
                                    bin_min: float = 6.0,
                                    show_tracks: bool = True,
                                    save_as: Optional[str] = "membrane_asymmetry_vs_time"):
    """Three panels: asymmetry vs time, its distribution per time bin, and whether
    the bright side points the same way across the population.

    Panel (c) is the control that matters. If the population resultant R sits
    above the Rayleigh line, every nucleus is bright on the same side of the
    image -- which is what an illumination gradient or a coverslip tilt looks
    like, not what stochastic dynein-driven delivery looks like. Biological
    asymmetry should be strong per nucleus and randomly oriented across them.
    """
    if asym_df is None or asym_df.empty:
        print("No data to plot."); return

    d = asym_df.dropna(subset=[metric, "true_time_min"]).copy()
    if d.empty:
        print("No finite values to plot."); return
    d["bin_lo"], edges = _time_bins(d["true_time_min"], bin_min)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    # (a) per-track trajectories + population median/IQR
    ax = axes[0]
    if show_tracks:
        for tid, dt in d.groupby("track_id"):
            if len(dt) < 2:
                continue
            dt = dt.sort_values("true_time_min")
            ax.plot(dt.true_time_min, dt[metric], color="0.7", lw=0.6, alpha=0.6, zorder=1)
    g = d.groupby("true_time_min")[metric]
    med, q1, q3 = g.median(), g.quantile(0.25), g.quantile(0.75)
    ax.fill_between(med.index, q1, q3, alpha=0.25, color="#4C72B0", lw=0, zorder=2)
    ax.plot(med.index, med, lw=2.2, color="#4C72B0", zorder=3, label="median \u00b1 IQR")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel({"asym_index": "asymmetry index  $A_1/A_0$",
                   "cv": "CV of perinuclear intensity",
                   "lobe_ratio": "bright decile / dim decile"}.get(metric, metric))
    ax.set_title("(a) Membrane asymmetry vs time")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    # (b) distribution per time bin
    ax = axes[1]
    groups = [d.loc[d.bin_lo == lo, metric].to_numpy() for lo in edges]
    centers = edges + bin_min / 2
    ax.boxplot(groups, positions=centers, widths=bin_min * 0.7,
               showfliers=False, patch_artist=True,
               boxprops=dict(facecolor="#DD8452", alpha=0.55, lw=0.8),
               medianprops=dict(color="k", lw=1.4),
               whiskerprops=dict(lw=0.8), capprops=dict(lw=0.8))
    ax.set_xlim(edges[0] - bin_min * 0.7, edges[-1] + bin_min * 1.7)
    ax.set_xticks(centers)
    ax.set_xticklabels([f"{lo:g}\u2013{lo + bin_min:g}" for lo in edges],
                       rotation=45, ha="right", fontsize=8)
    ax.set_xlabel(f"Time bin (min, width {bin_min:g})")
    ax.set_ylabel("asymmetry index")
    ax.set_title(f"(b) Distribution per {bin_min:g}-min bin")
    ax.spines[["top", "right"]].set_visible(False)

    # (c) directional consistency across the population
    ax = axes[2]
    stats = []
    for lo in edges:
        sub = d[d.bin_lo == lo]
        mu, R, p = _circular_stats(sub["phi_rad"].to_numpy())
        stats.append({"bin_lo": lo, "n": len(sub), "mean_dir_deg": np.degrees(mu) % 360,
                      "R": R, "rayleigh_p": p})
    stats = pd.DataFrame(stats)
    ax.plot(stats.bin_lo + bin_min / 2, stats.R, marker="o", lw=1.8, color="#55A868",
            label="population resultant $R$")
    n_ref = stats["n"].replace(0, np.nan)
    ax.plot(stats.bin_lo + bin_min / 2, np.sqrt(-np.log(0.05) / n_ref),
            ls="--", color="k", lw=1, label="Rayleigh p=0.05")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time bin (min)")
    ax.set_ylabel("directional concentration $R$")
    ax.set_title("(c) Do all nuclei point the same way?")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Perinuclear membrane asymmetry", fontsize=13)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return stats


def plot_asymmetry_direction_rose(asym_df: pd.DataFrame, config, nbins: int = 24,
                                  save_as: Optional[str] = "asymmetry_direction_rose"):
    """Polar histogram of the bright-side direction, one panel per frame."""
    if asym_df is None or asym_df.empty:
        print("No data to plot."); return
    ts = sorted(asym_df["t"].unique())
    ncol = min(5, len(ts))
    nrow = math.ceil(len(ts) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.2 * nrow),
                             subplot_kw={"projection": "polar"}, squeeze=False)
    edges = np.linspace(0, 2 * np.pi, nbins + 1)
    for ax, t in zip(axes.ravel(), ts):
        phi = asym_df.loc[asym_df.t == t, "phi_rad"].dropna().to_numpy() % (2 * np.pi)
        counts, _ = np.histogram(phi, bins=edges)
        ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge",
               color="#8172B3", alpha=0.8, edgecolor="w", lw=0.4)
        mu, R, p = _circular_stats(asym_df.loc[asym_df.t == t, "phi_rad"].to_numpy())
        if np.isfinite(R) and counts.max() > 0:
            ax.plot([mu, mu], [0, counts.max()], color="k", lw=1.6)
        ax.set_title(f"t={t}  n={len(phi)}\nR={R:.2f}, p={p:.3g}", fontsize=8)
        _orient_polar(ax)
        ax.tick_params(labelsize=6)
        ax.set_yticklabels([])
    for ax in axes.ravel()[len(ts):]:
        ax.axis("off")
    fig.suptitle("Direction of the membrane-bright side (image frame)", fontsize=12)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()


### 36e. Cross-sectional area by time interval

Box plot of nuclear cross-sectional area in fixed acquisition-time bins.

On bin width: `true_time_min = t · (tile_rows · tile_cols · minutes_per_tile) +
tile_offset_min`. With the current 2 × 3 tiling at 1 min/tile that is **6 minutes
per frame**, so a 6-minute bin is exactly one frame with its six tile offsets
folded in. Binning narrower than 6 min splits a frame by tile — that is a
stage-position axis, not a time axis, and any structure it shows is tile order.
Wider bins (12, 18) pool frames and are legitimate.

Pass `halo_df` for repaired, halo-measured areas, or `timed_df` for every
detection before tracking. Area is always recomputed from the pixel count with
the live `cfg.pixel_size_um`, so a stale pickle from a different calibration
can't leak into the figure.

In [ ]:
def plot_area_boxplot_binned(df: pd.DataFrame, config, bin_min: float = 6.0,
                             time_col: str = "true_time_min",
                             area_col: Optional[str] = None,
                             show_points: bool = True,
                             show_median_trend: bool = True,
                             show_n: bool = True,
                             ylim: Optional[Tuple[float, float]] = None,
                             title: Optional[str] = None,
                             save_as: Optional[str] = "area_boxplot_6min"):
    """Nuclear cross-sectional area, boxed into fixed-width acquisition-time bins.

    Note on bin width: true_time_min = t * (tile_rows * tile_cols *
    minutes_per_tile) + tile_offset_min. With the current 2x3 tiling at 1 min
    per tile that is 6 minutes per frame, so a 6-minute bin is exactly one
    frame and the tile offsets fall inside it. Narrower bins split a frame by
    tile, which is a stage-position axis, not a biology axis.
    """
    if df is None or df.empty:
        print("No data to plot."); return

    d = df.copy()
    if area_col is None:
        d = resolve_area_um2(d, config)
        area_col = "nucleus_area_um2"
    if time_col not in d.columns:
        raise ValueError(f"DataFrame must contain '{time_col}'.")
    d = d.dropna(subset=[area_col, time_col])
    if d.empty:
        print("No finite values to plot."); return

    d["bin_lo"], edges = _time_bins(d[time_col], bin_min)
    groups = [d.loc[d.bin_lo == lo, area_col].to_numpy() for lo in edges]
    centers = edges + bin_min / 2

    fig, ax = plt.subplots(figsize=(max(7.5, 0.85 * len(edges) + 3), 5))

    if show_points:
        rng = np.random.default_rng(0)
        for lo, vals in zip(edges, groups):
            if vals.size == 0:
                continue
            x = lo + bin_min / 2 + rng.uniform(-bin_min * 0.22, bin_min * 0.22, vals.size)
            ax.scatter(x, vals, s=7, alpha=0.25, color="0.35", lw=0, zorder=1)

    ax.boxplot(groups, positions=centers, widths=bin_min * 0.7,
               showfliers=False, patch_artist=True, zorder=2,
               boxprops=dict(facecolor="#4C72B0", alpha=0.45, lw=0.9),
               medianprops=dict(color="k", lw=1.6),
               whiskerprops=dict(lw=0.9), capprops=dict(lw=0.9))

    if show_median_trend:
        meds = [np.median(v) if v.size else np.nan for v in groups]
        ax.plot(centers, meds, color="#C44E52", lw=1.8, marker="o", ms=4,
                zorder=3, label="median")
        ax.legend(fontsize=9, frameon=False, loc="upper left", bbox_to_anchor=(0.0, 0.95))

    if show_n:
        # blended transform: x in data units, y in axes fraction, so the labels
        # sit inside the axes and cannot collide with the title
        trans = ax.get_xaxis_transform()
        for c, v in zip(centers, groups):
            ax.text(c, 0.99, f"n={v.size}", transform=trans, ha="center", va="top",
                    fontsize=7, color="0.35")

    ax.set_xlim(edges[0] - bin_min * 0.7, edges[-1] + bin_min * 1.7)
    ax.set_xticks(centers)
    ax.set_xticklabels([f"{lo:g}\u2013{lo + bin_min:g}" for lo in edges],
                       rotation=45, ha="right")
    ax.set_xlabel(f"Acquisition time bin (min, width {bin_min:g})")
    ax.set_ylabel("Nuclear cross-sectional area (\u00b5m\u00b2)")
    ax.set_title(title or f"Nuclear cross-sectional area per {bin_min:g}-min interval")
    if ylim:
        ax.set_ylim(*ylim)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()

    summary = (d.groupby("bin_lo")[area_col]
               .agg(n="size", median="median",
                    q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75),
                    mean="mean", std="std")
               .reset_index().rename(columns={"bin_lo": "bin_start_min"}))
    summary["bin_end_min"] = summary["bin_start_min"] + bin_min
    return summary.round(2)


### 36f. Run the full figure set

In [ ]:
# ── Full figure set ───────────────────────────────────────────────────────
# Every figure is written to <population folder>/figures/ as PNG + PDF, and every
# table here is a *_sel table: the selected single nuclei (§16).
SAVE_FIGURES = True
print("Figures for:", POP_LABEL)

# 0. carried over from v17 — N/C ratio and area vs time
plot_nc_vs_time(halo_sel)
plot_area_vs_time(halo_sel)
plot_largest_cross_sectional_area_vs_time(timed_sel, cfg)

# 1. one-time prep: proper nucleus-surface -> wall radius on the sweep
radial_n_df = add_wall_normalised_radius(radial_sel, cfg)

# 2. rose plots — envelope shape by angle
envelope_peak_df = compute_envelope_radius_by_angle(
    radial_n_df, cfg, search_rho_max=ENVELOPE_SEARCH_RHO_MAX)
_ = plot_rose_individual(envelope_peak_df, tracked_sel, cfg, n=5, selection="longest")
pooled_rose_df = plot_rose_pooled(envelope_peak_df, cfg)

# 3. membrane intensity by angle and distance
_longest = tracked_sel.groupby("track_id")["t"].nunique().idxmax()
plot_membrane_angle_distance(radial_n_df, cfg, track_id=_longest,
                             save_as=f"angle_distance_track{_longest}")
# rbins kept coarse here on purpose: rays are sampled every
# radial_step_size_px (1 px = 0.1625 um), so bins finer than ~1.5 px come back
# part-empty. 8 um / 32 bins = 0.25 um = 1.5 px.
plot_membrane_angle_distance(radial_n_df, cfg, track_id=None, rbins=32,
                             radius_col="r_from_edge_um", r_max_um=8.0,
                             save_as="angle_distance_pooled")

# 4. membrane asymmetry vs time
asym_df = compute_membrane_asymmetry(radial_n_df, cfg, tracked_df=tracked_sel,
                                     band=ASYMMETRY_BAND_UM, band_units="um")
asym_stats = plot_membrane_asymmetry_vs_time(asym_df, cfg, bin_min=AREA_BIN_MIN)
plot_asymmetry_direction_rose(asym_df, cfg)
asym_df.to_csv(POP_EXPORT_DIR / "membrane_asymmetry.csv", index=False)

# 5. cross-sectional area boxed into fixed time intervals
area_summary = plot_area_boxplot_binned(
    halo_sel, cfg, bin_min=AREA_BIN_MIN,
    title=f"Nuclear cross-sectional area per {AREA_BIN_MIN:g}-min interval ({POP_LABEL})")
display(area_summary)
area_summary.to_csv(POP_EXPORT_DIR / "area_by_time_bin.csv", index=False)

print("figures ->", figures_dir(cfg))


---
## 37. Shell-resolved membrane asymmetry

The faceted rose — radial shells down the rows, timepoints across the columns —
plus the normalised-score supplement and the direction-agnostic aggregate.

Four things about the original version change what it says, so each is
implemented as an explicit, switchable choice rather than a silent default.

**1. Summed intensity is biased by the droplet wall.** Rays stop at the wall,
so a shell defined in absolute distance from the nuclear surface is clipped by
different amounts at different θ as soon as the nucleus sits off-centre in its
droplet. A sum then counts more pixels on the far side and reports asymmetry
pointing away from the near wall — *for a perfectly uniform field*. With a
uniform intensity field and a nucleus 6 µm off-centre, the outermost shell
returns R = 0.42; at 9 µm it returns 0.60. §37a bins on **mean per sample**,
which is invariant to how many pixels a cell happens to contain, and §37b adds
`R_geom` — the same resultant computed on the sample counts with intensity
discarded — as a per-facet floor.

**2. Radius encoding exaggerates magnitude.** On a polar bar the eye reads
area, and area goes as r². A linear radius shows a 2× difference as a 4× wedge.
§37c defaults to `radius_scale="sqrt"`, so wedge **area** is proportional to the
value. `"linear"` restores the literal encoding.

**3. The resultant needs a baseline that is not its own minimum.** With a
background pedestal *b* and signal *s*, R = |Σs·e^{iθ}| / (N·b + Σs), so a
bright cytoplasm drives R toward zero however one-sided the membrane is.
Subtracting each profile's own minimum overcorrects in a way that is worse:
R then becomes a pure *shape* statistic, returning **0.525 for a cosine of any
amplitude**, so a nucleus twice as one-sided scores identically. §37b subtracts
a single dataset-wide floor, giving R = (A₁/A₀)/2 — bounded on [0, 1], zero for
an even shell, linear in how one-sided it is.

**4. The resultant needs a null.** Even a uniform shell returns R ≈ 1/√n_eff,
about 0.17 for 36 bins, so every facet looks mildly asymmetric. §37b reports
`R_corr` (debiased), a Rayleigh p on the weighted resultant using Kish's
effective sample size, and `R_crit`, drawn on each facet as a dashed ring: an
arrow inside that ring is noise.

In [ ]:
from typing import Sequence                      # not in the Section 1 import block
from matplotlib.colors import LinearSegmentedColormap

# ── Section 37 parameters ─────────────────────────────────────────────────
# Shell edges, microns outward from the nuclear mask surface. These are the
# edges from the original figure; change them freely, but see 37b on why the
# outermost shell needs the geometry control.
DEFAULT_SHELL_EDGES_UM = (0.0, 2.34, 5.36, 9.55, 23.5)

# Angular bins. 36 bins = 10 degrees. Fewer bins raises the per-nucleus
# detection floor (R_crit goes as 1/sqrt(n_eff)); more bins makes each bin
# noisier without adding information once bins are finer than the ray spacing.
N_THETA_BINS = 36

# Fraction of theta below which a shell is flagged as wall-clipped.
MIN_SHELL_COVERAGE = 0.80


### 37a. Binning into (shell, θ) cells

Returns a **complete** θ grid per unit — cells with no sample are kept with
`n_samples = 0` rather than dropped, because their absence is the geometry
signal that §37b tests for.

The per-cell statistic is `I_mean`, not `I_sum`. `I_sum` is still carried so
the original encoding can be reproduced for comparison.

In [ ]:
def bin_shell_angle(radial_df: pd.DataFrame, config,
                    shell_edges_um: Sequence[float] = DEFAULT_SHELL_EDGES_UM,
                    n_theta_bins: int = N_THETA_BINS,
                    radius_col: str = "r_from_edge_um") -> pd.DataFrame:
    """Bin the sweep into (shell, theta) cells per nucleus per timepoint.

    Returns one row per (t, track_id, shell_idx, theta_bin) on a COMPLETE theta
    grid -- cells with no sample are kept with n_samples = 0 and I_mean = NaN,
    because their absence is itself the geometry signal that
    compute_shell_resultants tests for.

    I_mean, not I_sum, is the per-cell statistic. Rays terminate at the droplet
    wall, so a shell defined in absolute distance is clipped by different
    amounts at different theta whenever the nucleus is off-centre. A sum then
    counts more pixels on the far side and reports asymmetry pointing away from
    the near wall, for a perfectly uniform field. A mean is invariant to how
    many pixels a cell happens to contain.
    """
    if radial_df is None or radial_df.empty:
        return pd.DataFrame()

    df = radial_df
    if radius_col not in df.columns:
        raise ValueError(f"{radius_col} missing -- run add_wall_normalised_radius first.")

    edges = np.asarray(shell_edges_um, dtype=float)
    d = df[["t", "track_id", "theta_rad", radius_col, "intensity"]].copy()
    d["shell_idx"] = pd.cut(d[radius_col], edges, labels=False, include_lowest=True)
    d = d[d["shell_idx"].notna()]
    if d.empty:
        return pd.DataFrame()
    d["shell_idx"] = d["shell_idx"].astype(int)

    dtheta = 2 * np.pi / n_theta_bins
    d["theta_bin"] = ((d["theta_rad"] % (2 * np.pi)) // dtheta).astype(int)
    d["theta_bin"] = d["theta_bin"].clip(0, n_theta_bins - 1)

    g = (d.groupby(["t", "track_id", "shell_idx", "theta_bin"])["intensity"]
         .agg(n_samples="size", I_sum="sum", I_mean="mean")
         .reset_index())

    # complete the theta grid so empty cells are explicit
    units = g[["t", "track_id", "shell_idx"]].drop_duplicates()
    grid = units.merge(pd.DataFrame({"theta_bin": np.arange(n_theta_bins)}), how="cross")
    out = grid.merge(g, on=["t", "track_id", "shell_idx", "theta_bin"], how="left")
    out["n_samples"] = out["n_samples"].fillna(0).astype(int)
    out["I_sum"] = out["I_sum"].fillna(0.0)

    out["theta_rad"] = (out["theta_bin"] + 0.5) * dtheta
    out["shell_lo_um"] = edges[out["shell_idx"].to_numpy()]
    out["shell_hi_um"] = edges[out["shell_idx"].to_numpy() + 1]
    out["shell_label"] = [f"({lo:g}, {hi:g}]" for lo, hi
                          in zip(out.shell_lo_um, out.shell_hi_um)]
    out.attrs["n_theta_bins"] = n_theta_bins
    out.attrs["shell_edges_um"] = tuple(edges)
    return out.sort_values(["t", "track_id", "shell_idx", "theta_bin"]).reset_index(drop=True)


### 37b. Resultant vectors, baselines, and the two nulls

Four resultants per unit, each answering a different question:

| column | weights | what it measures |
|---|---|---|
| `R` | `I_mean − background` | **the score.** (A₁/A₀)/2 for a cosine; 0 = even, 1 = all one side |
| `R_raw` | `I_mean` | what an uncorrected rose shows; deflated by the pedestal |
| `R_shape` | `I_mean −` own 10th pct | concentration only — saturates at 0.525, never quote as magnitude |
| `R_geom` | `n_samples` | asymmetry from droplet-wall clipping alone |

`R` must clear both `R_crit` (the noise floor) and `R_geom` (the geometry
floor) to mean anything.

On `background`: it has to be estimated from outside the profile being scored —
`estimate_background` uses a low quantile of every sweep sample in the dataset,
which is the camera offset plus the isotropic cytoplasmic floor. Pass your own
value if you have a dark-frame measurement.

`n_eff` is Kish's effective sample size, `(Σw)²/Σw²`: the number of
equally-weighted bins carrying the same information. It is what the Rayleigh
statistic needs when observations are intensity weights rather than counts.

In [ ]:
def _resultant(theta: np.ndarray, w: np.ndarray) -> Tuple[float, float, float]:
    """Weighted circular resultant. Returns (R, phi, n_eff).

    n_eff is Kish's effective sample size (sum w)^2 / sum(w^2): the number of
    equally-weighted bins that would carry the same information. It is what the
    Rayleigh statistic needs when the observations are intensity weights rather
    than counts.
    """
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(w) & np.isfinite(theta) & (w > 0)
    if ok.sum() < 3:
        return np.nan, np.nan, np.nan
    th, w = theta[ok], w[ok]
    sw = w.sum()
    if sw <= 0:
        return np.nan, np.nan, np.nan
    C = (w * np.exp(1j * th)).sum() / sw
    n_eff = float(sw ** 2 / (w ** 2).sum())
    return float(np.abs(C)), float(np.angle(C)), n_eff


def _debias_R(R: float, n_eff: float) -> float:
    """Bias-corrected resultant length.

    E[R^2] ~= rho^2 + (1 - rho^2)/n_eff, so even a perfectly uniform shell
    returns R ~ 1/sqrt(n_eff) -- about 0.17 for 36 bins. Reporting raw R makes
    every facet look mildly asymmetric. This inverts the relation and clips at
    zero.
    """
    if not (np.isfinite(R) and np.isfinite(n_eff)) or n_eff <= 1:
        return np.nan
    rho2 = (n_eff * R ** 2 - 1.0) / (n_eff - 1.0)
    return float(np.sqrt(max(rho2, 0.0)))


def _rayleigh_p(R: float, n_eff: float) -> float:
    """Rayleigh test on the weighted resultant (Zar's two-term approximation)."""
    if not (np.isfinite(R) and np.isfinite(n_eff)) or n_eff <= 1:
        return np.nan
    z = n_eff * R ** 2
    p = np.exp(-z) * (1 + (2 * z - z ** 2) / (4 * n_eff)
                      - (24 * z - 132 * z ** 2 + 76 * z ** 3 - 9 * z ** 4)
                      / (288 * n_eff ** 2))
    return float(np.clip(p, 0.0, 1.0))


def R_critical(n_eff: float, alpha: float = 0.05) -> float:
    """Resultant length a uniform shell would exceed with probability alpha."""
    if not np.isfinite(n_eff) or n_eff <= 1:
        return np.nan
    return float(np.sqrt(-np.log(alpha) / n_eff))


def estimate_background(radial_df: pd.DataFrame, q: float = 0.05) -> float:
    """Isotropic intensity floor for the whole dataset: the q-quantile of every
    sweep sample.

    This is the zero of the asymmetry scale, and it has to come from OUTSIDE the
    profile being scored. Subtracting a profile's own minimum instead makes R a
    shape statistic that saturates -- a cosine of ANY amplitude returns
    R = 0.525, so a nucleus twice as one-sided scores the same. Subtracting a
    common floor gives R = (A1/A0)/2, which is what a normalised asymmetry
    score should do.
    """
    if radial_df is None or radial_df.empty:
        return 0.0
    return float(np.quantile(radial_df["intensity"].to_numpy(), q))


def compute_shell_resultants(binned: pd.DataFrame,
                             background: Optional[float] = None,
                             background_q: float = 0.05,
                             min_coverage: float = 0.80,
                             alpha: float = 0.05) -> pd.DataFrame:
    """Per (t, track_id, shell): resultant vector, its null, and a geometry control.

    Four numbers per unit, each answering a different question:

    R        THE SCORE. Resultant of (I_mean - background), background being one
             dataset-wide floor (see estimate_background). For a cosine profile
             this is exactly (A1/A0)/2, so it is bounded on [0, 1], zero for an
             even shell, and linear in how one-sided the shell is.
    R_raw    the same without subtracting anything. Reported because it is what
             an uncorrected rose shows, and it is deflated by the pedestal: a
             bright cytoplasm pushes it toward zero however one-sided the
             membrane is. Compare the two to see how much background was
             masking.
    R_shape  resultant after removing each shell's OWN 10th percentile. This
             discards amplitude entirely and measures only how concentrated the
             bright side is -- a narrow lobe scores above a broad one. Useful,
             but never quote it as an asymmetry magnitude.
    R_geom   resultant of n_samples, intensity discarded. The asymmetry a
             perfectly uniform field would show given only how the droplet wall
             clipped the rays. R has to clear R_geom to mean anything.

    Also: R_corr (R debiased for finite bin count), rayleigh_p, R_crit,
    coverage, and A1_over_A0 = 2R for a cosine, quoted directly for continuity
    with Section 36d.
    """
    cols = ["t", "track_id", "shell_idx", "shell_label", "shell_lo_um", "shell_hi_um",
            "I_bar", "I_bg", "R", "R_corr", "R_raw", "R_shape",
            "phi_rad", "phi_deg", "n_eff", "rayleigh_p", "R_crit", "significant",
            "R_geom", "phi_geom_deg", "R_excess", "coverage", "n_occupied",
            "A1_over_A0", "n_samples_total"]
    if binned is None or binned.empty:
        return pd.DataFrame(columns=cols)

    if background is None:
        occ_vals = binned.loc[binned.n_samples > 0, "I_mean"].to_numpy()
        background = float(np.quantile(occ_vals, background_q)) if occ_vals.size else 0.0
        print(f"background floor estimated from binned data: {background:.1f} "
              f"(q={background_q}). Pass background=... to override.")

    rows = []
    for (t, tid, sh), d in binned.groupby(["t", "track_id", "shell_idx"], sort=True):
        d = d.sort_values("theta_bin")
        th = d["theta_rad"].to_numpy()
        I = d["I_mean"].to_numpy(dtype=float)
        n = d["n_samples"].to_numpy(dtype=float)

        occ = n > 0
        coverage = float(occ.mean())
        if occ.sum() < 3:
            continue

        Ifin = np.where(occ, I, np.nan)
        I_bar = float(np.nanmean(Ifin))

        w_raw = np.where(occ, Ifin, 0.0)
        w_sig = np.clip(w_raw - background, 0.0, None) * occ
        w_shape = np.clip(w_raw - np.nanquantile(Ifin, 0.10), 0.0, None) * occ

        R_raw, _, _ = _resultant(th, w_raw)
        R, phi, n_eff = _resultant(th, w_sig)
        R_shape, _, _ = _resultant(th, w_shape)
        R_geom, phi_g, _ = _resultant(th, n)

        # first harmonic of the background-subtracted profile
        sig = np.where(occ, Ifin - background, np.nan)
        base = np.nanmean(sig)
        if occ.sum() >= 8 and np.isfinite(base) and base > 0:
            c1 = np.nanmean(sig * np.exp(-1j * th))
            A1_A0 = float(2 * np.abs(c1) / base)
        else:
            A1_A0 = np.nan

        R_corr = _debias_R(R, n_eff)
        p = _rayleigh_p(R, n_eff)
        Rc = R_critical(n_eff, alpha)
        rows.append({
            "t": int(t), "track_id": int(tid), "shell_idx": int(sh),
            "shell_label": d["shell_label"].iloc[0],
            "shell_lo_um": float(d["shell_lo_um"].iloc[0]),
            "shell_hi_um": float(d["shell_hi_um"].iloc[0]),
            "I_bar": I_bar, "I_bg": float(background),
            "R": R, "R_corr": R_corr, "R_raw": R_raw, "R_shape": R_shape,
            "phi_rad": phi,
            "phi_deg": float(np.degrees(phi) % 360.0) if np.isfinite(phi) else np.nan,
            "n_eff": n_eff, "rayleigh_p": p, "R_crit": Rc,
            "significant": bool(np.isfinite(p) and p < alpha),
            "R_geom": R_geom,
            "phi_geom_deg": float(np.degrees(phi_g) % 360.0) if np.isfinite(phi_g) else np.nan,
            "R_excess": (R_corr - R_geom) if np.isfinite(R_corr) and np.isfinite(R_geom) else np.nan,
            "coverage": coverage, "n_occupied": int(occ.sum()),
            "A1_over_A0": A1_A0,
            "n_samples_total": float(n.sum()),
        })

    out = pd.DataFrame(rows, columns=cols)
    if not out.empty:
        out["low_coverage"] = out["coverage"] < min_coverage
        n_low = int(out["low_coverage"].sum())
        if n_low:
            print(f"NOTE: {n_low}/{len(out)} shell-units below {min_coverage:.0%} theta "
                  f"coverage -- the droplet wall cuts these shells off. They are kept "
                  f"but flagged; read R_geom for them.")
    return out


### 37c. The faceted rose

Rows = shells, columns = timepoints. Reading the facet:

- **coloured wedges** — mean intensity above the background floor, area-scaled
- **grey wedges** — angles with no sample at all (wall clipping), marked rather
  than left blank
- **red arrow** — the resultant, length = R on the 0–1 radius; solid red when
  p < 0.05, grey otherwise
- **dashed ring** — `R_crit`; an arrow inside it is noise
- **dotted grey arrow** — `R_geom`. Where it rivals the red arrow, the facet is
  showing droplet placement
- **facet label** — `R = score`, `g = R_geom`, `*` = significant

`share_r="row"` puts a shell's timepoints on one radial scale so the columns are
comparable; `"all"` compares shells too; `"none"` autoscales each facet and
should only be used to read shape.

In [ ]:
def _rose_axis_values(d: pd.DataFrame, res_row, value: str) -> np.ndarray:
    occ = d["n_samples"].to_numpy() > 0
    I = d["I_mean"].to_numpy(dtype=float)
    if value == "mean":
        v = np.where(occ, I, 0.0)
    elif value == "mean_bgsub":
        v = np.where(occ, I - float(res_row["I_bg"]), 0.0)
    elif value == "sum":
        v = d["I_sum"].to_numpy(dtype=float)
    else:
        raise ValueError(f"unknown value {value!r}")
    return np.clip(v, 0.0, None)


def plot_shell_rose_facets(binned: pd.DataFrame, resultants: pd.DataFrame, config,
                           track_id: Optional[int] = None,
                           timepoints: Optional[Sequence[int]] = None,
                           value: str = "mean_bgsub",
                           radius_scale: str = "sqrt",
                           share_r: str = "row",
                           show_geom_arrow: bool = True,
                           alpha: float = 0.05,
                           save_as: str = "shell_rose_facets"):
    """Rose facets: rows = radial shells, columns = timepoints.

    Encoding choices, all of which change what the reader concludes:

    radius_scale="sqrt" (default) makes wedge AREA proportional to the value.
        On a polar bar the eye reads area, and area goes as r^2, so a linear
        radius shows a 2x difference as a 4x wedge. Every Nightingale rose has
        this problem; sqrt is the standard correction. Pass "linear" to get the
        literal radius encoding back.

    value="mean_bgsub" (default) plots mean intensity per sample with the
        per-shell baseline removed, so the origin is background rather than
        zero counts and the bars show the part of the signal that can be
        asymmetric. "mean" keeps the pedestal and draws it as a grey ring so
        you can see how much of each bar is background. "sum" reproduces the
        original encoding and is biased wherever coverage < 1.

    share_r="row" puts every timepoint in a shell on one radial scale, so the
        columns are comparable; "all" compares shells too; "none" autoscales
        each facet and should only be used for shape.

    The red arrow is the background-subtracted resultant, length R x r_limit.
    The dashed ring is R_crit, the length a uniform shell clears 5 % of the
    time -- an arrow inside that ring is noise. The open grey arrow is R_geom,
    the resultant of the sample counts alone: the asymmetry produced by the
    droplet wall clipping the rays. Where the grey arrow rivals the red one,
    the facet is showing geometry.
    """
    if binned is None or binned.empty or resultants is None or resultants.empty:
        print("No data to plot."); return

    b, r = binned, resultants
    if track_id is not None:
        b = b[b.track_id == int(track_id)]
        r = r[r.track_id == int(track_id)]
        subtitle = f"track {int(track_id)}"
        if b.empty:
            print(f"track {track_id} not present."); return
    else:
        subtitle = "per-nucleus facets require track_id; pooling shown"

    ts = sorted(b["t"].unique()) if timepoints is None else list(timepoints)
    shells = sorted(b["shell_idx"].unique())
    if not ts or not shells:
        print("No data to plot."); return

    key = ["t", "track_id", "shell_idx"]
    res_lut = r.set_index(key)

    # precompute values so scales can be shared
    vals: Dict[Tuple[int, int], np.ndarray] = {}
    for sh in shells:
        for t in ts:
            try:
                row = res_lut.loc[(t, int(track_id), sh)]
            except KeyError:
                continue
            d = b[(b.t == t) & (b.shell_idx == sh)].sort_values("theta_bin")
            if d.empty:
                continue
            vals[(sh, t)] = _rose_axis_values(d, row, value)

    if not vals:
        print("No data to plot."); return

    def _lim(keys):
        pool = np.concatenate([vals[k] for k in keys if k in vals])
        pool = pool[np.isfinite(pool)]
        return float(np.percentile(pool, 99.5)) if pool.size else 1.0

    if share_r == "all":
        lim_all = _lim(list(vals))
        limits = {sh: lim_all for sh in shells}
    elif share_r == "row":
        limits = {sh: _lim([(sh, t) for t in ts]) for sh in shells}
    else:
        limits = None

    xf = (lambda v, L: np.sqrt(np.clip(v, 0, None) / L) if radius_scale == "sqrt"
          else np.clip(v, 0, None) / L)

    dtheta = 2 * np.pi / b.attrs.get("n_theta_bins", N_THETA_BINS)
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    shell_colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(len(shells), len(ts),
                             figsize=(2.05 * len(ts), 2.25 * len(shells)),
                             subplot_kw={"projection": "polar"}, squeeze=False)

    for i, sh in enumerate(shells):
        for j, t in enumerate(ts):
            ax = axes[i][j]
            ax.set_theta_zero_location("E")
            ax.set_theta_direction(-1)          # image frame: y grows downward
            ax.set_yticklabels([])
            ax.set_xticks(np.radians([0, 90, 180, 270]))
            ax.set_xticklabels([])   # orientation stated once in the subtitle
            ax.tick_params(labelsize=5.5, pad=-2)
            ax.grid(alpha=0.25, lw=0.4)
            ax.set_ylim(0, 1.0)

            if (sh, t) not in vals:
                ax.set_axis_off(); continue
            row = res_lut.loc[(t, int(track_id), sh)]
            d = b[(b.t == t) & (b.shell_idx == sh)].sort_values("theta_bin")
            v = vals[(sh, t)]
            L = limits[sh] if limits else max(float(np.max(v)), 1e-12)

            occ = d["n_samples"].to_numpy() > 0
            rr = xf(v, L)
            colors = [shell_colors[sh] if o else "none" for o in occ]
            ax.bar(d["theta_rad"].to_numpy(), rr, width=dtheta * 0.92,
                   bottom=0.0, color=colors, edgecolor="none", align="center")
            # unsampled angles marked, not silently blank
            if (~occ).any():
                ax.bar(d["theta_rad"].to_numpy()[~occ], np.ones((~occ).sum()),
                       width=dtheta * 0.92, bottom=0.0, color="0.90", zorder=0)

            if value == "mean" and np.isfinite(row["I_bg"]):
                ax.plot(np.linspace(0, 2 * np.pi, 200),
                        np.full(200, xf(row["I_bg"], L)),
                        color="0.45", lw=0.7, ls=":", zorder=4)

            Rv, phi = float(row["R"]), float(row["phi_rad"])
            Rc = float(row["R_crit"])
            if np.isfinite(Rc):
                ax.plot(np.linspace(0, 2 * np.pi, 200), np.full(200, Rc),
                        color="k", lw=0.6, ls="--", alpha=0.5, zorder=4)
            if np.isfinite(Rv) and np.isfinite(phi):
                sig = bool(row["significant"])
                ax.annotate("", xy=(phi, Rv), xytext=(0, 0),
                            arrowprops=dict(arrowstyle="-|>", lw=1.4,
                                            color="#D7263D" if sig else "0.55",
                                            alpha=1.0 if sig else 0.7), zorder=6)
            if show_geom_arrow and np.isfinite(row["R_geom"]) and row["R_geom"] > 0.02:
                ax.annotate("", xy=(np.radians(row["phi_geom_deg"]), row["R_geom"]),
                            xytext=(0, 0),
                            arrowprops=dict(arrowstyle="-|>", lw=1.1, color="0.35",
                                            ls=":", alpha=0.9), zorder=5)

            star = "*" if row["significant"] else ""
            ax.annotate(f"R={row['R']:.2f}{star}  g={row['R_geom']:.2f}",
                        xy=(0.5, -0.13), xycoords="axes fraction",
                        ha="center", fontsize=6.5,
                        color="0.15" if row["significant"] else "0.5")
            if i == 0:
                ax.annotate(f"t = {t}", xy=(0.5, 1.18), xycoords="axes fraction",
                            ha="center", fontsize=9)
            if j == 0:
                ax.annotate(f"{row['shell_label']} \u00b5m", xy=(-0.38, 0.5),
                            xycoords="axes fraction", rotation=90,
                            va="center", ha="center", fontsize=7.5)

    vlab = {"mean_bgsub": "mean intensity above shell baseline",
            "mean": "mean intensity per sample",
            "sum": "summed intensity (biased where coverage < 1)"}[value]
    rlab = ("radius \u221d \u221avalue, so wedge AREA \u221d value"
            if radius_scale == "sqrt" else "radius \u221d value (area \u221d value\u00b2)")
    fig.suptitle("Membrane intensity by angle and distance", fontsize=13, x=0.02, ha="left", y=0.995)
    fig.text(0.02, 0.963, f"10\u00b0 bins \u00b7 {vlab} \u00b7 {rlab} \u00b7 {subtitle}",
             fontsize=8, ha="left", color="0.25")
    fig.text(0.02, 0.941, "0\u00b0 = image +x (right); angle increases clockwise, "
             "matching the displayed image", fontsize=7.5, ha="left", color="0.45")
    fig.text(0.02, 0.012,
             "red arrow = background-subtracted resultant (length = R); dashed ring = R at p=0.05; "
             "dotted grey arrow = R_geom, the resultant of sample counts alone (droplet-wall clipping). "
             "Facet labels: R = asymmetry score, g = R_geom, * = p < %.2f. "
             "Grey wedges = angles with no sample." % alpha,
             fontsize=6.5, color="0.3")
    fig.tight_layout(rect=(0.02, 0.03, 1, 0.925))
    save_figure(fig, save_as, config)
    plt.show()
    return fig


### 37d. Normalised asymmetry score — table and graph

`build_asymmetry_score_table` collapses the per-nucleus vectors by (shell, t):
median score with IQR, the geometry floor, `R_excess` (the part geometry does
not explain), the fraction of nuclei individually clearing Rayleigh, and
`pop_R` — the coherence of φ **across** nuclei.

`pop_R` is the artefact check carried over from §36d. Stochastic delivery
should be strong per nucleus and randomly oriented across them. A `pop_R` above
its p = 0.05 line means the bright side has a preferred lab-frame direction,
which is what an illumination gradient or a tilted coverslip looks like.

In [ ]:
def build_asymmetry_score_table(resultants: pd.DataFrame,
                                by: Sequence[str] = ("shell_label", "t")) -> pd.DataFrame:
    """Aggregate the per-nucleus vectors into a reportable score table.

    Columns
    -------
    n                 shell-units contributing
    R_corr_median     normalised asymmetry score, 0 = even, 1 = all on one side
    R_corr_q1/q3      IQR across nuclei
    R_geom_median     the same statistic on sample counts alone -- the floor
                      set by droplet geometry
    R_excess_median   R_corr - R_geom, the part not explained by geometry
    frac_significant  fraction of nuclei clearing the per-nucleus Rayleigh test
    pop_R             coherence of phi ACROSS nuclei (lab-frame directionality)
    pop_p             Rayleigh p on pop_R
    coverage_min      worst theta coverage in the group
    """
    if resultants is None or resultants.empty:
        return pd.DataFrame()

    def agg(d):
        phi = d["phi_rad"].dropna().to_numpy()
        n = phi.size
        if n:
            C = np.mean(np.exp(1j * phi))
            pop_R = float(np.abs(C))
            pop_dir = float(np.degrees(np.angle(C)) % 360)
            pop_p = float(np.exp(-n * pop_R ** 2))
        else:
            pop_R = pop_dir = pop_p = np.nan
        return pd.Series({
            "n": len(d),
            "R_median": d["R"].median(),
            "R_q1": d["R"].quantile(0.25),
            "R_q3": d["R"].quantile(0.75),
            "R_corr_median": d["R_corr"].median(),
            "R_raw_median": d["R_raw"].median(),
            "R_geom_median": d["R_geom"].median(),
            "R_excess_median": d["R_excess"].median(),
            "R_shape_median": d["R_shape"].median(),
            "A1_over_A0_median": d["A1_over_A0"].median(),
            "frac_significant": d["significant"].mean(),
            "pop_R": pop_R, "pop_dir_deg": pop_dir, "pop_p": pop_p,
            "coverage_min": d["coverage"].min(),
        })

    out = (resultants.groupby(list(by), sort=True)
           .apply(agg, include_groups=False).reset_index())
    return out.round(4)


def plot_asymmetry_score_summary(resultants: pd.DataFrame, config,
                                 score: str = "R",
                                 save_as: str = "asymmetry_score_summary"):
    """Three panels: score vs time per shell, the geometry floor, and lab-frame
    coherence across nuclei."""
    if resultants is None or resultants.empty:
        print("No data to plot."); return

    tab = build_asymmetry_score_table(resultants)
    shells = sorted(resultants["shell_idx"].unique())
    labels = (resultants.drop_duplicates("shell_idx")
              .set_index("shell_idx")["shell_label"].to_dict())
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    ax = axes[0]
    for sh in shells:
        d = (resultants[resultants.shell_idx == sh]
             .groupby("t")[score].agg(["median", lambda s: s.quantile(.25),
                                       lambda s: s.quantile(.75)]))
        d.columns = ["med", "q1", "q3"]
        ax.fill_between(d.index, d.q1, d.q3, color=colors[sh], alpha=0.15, lw=0)
        ax.plot(d.index, d.med, color=colors[sh], lw=2, marker="o", ms=4,
                label=f"{labels[sh]} \u00b5m")
    med_neff = resultants["n_eff"].median()
    if np.isfinite(med_neff):
        ax.axhline(R_critical(med_neff), color="k", lw=0.9, ls="--", alpha=0.6,
                   label="single-nucleus noise floor (p=0.05)")
    ax.set_xlabel("timepoint"); ax.set_ylabel("normalised asymmetry score $R$")
    ax.set_title("(a) Asymmetry score by shell")
    ax.legend(fontsize=7, title="shell", title_fontsize=7)
    ax.set_ylim(0, 1); ax.spines[["top", "right"]].set_visible(False)

    ax = axes[1]
    for sh in shells:
        d = resultants[resultants.shell_idx == sh].groupby("t")[["R", "R_geom"]].median()
        ax.plot(d.index, d.R, color=colors[sh], lw=2, marker="o", ms=4)
        ax.plot(d.index, d.R_geom, color=colors[sh], lw=1.2, ls=":", marker="^", ms=3)
    ax.plot([], [], color="0.3", lw=2, label="$R$ (signal)")
    ax.plot([], [], color="0.3", lw=1.2, ls=":", label="$R_{geom}$ (sampling)")
    ax.set_xlabel("timepoint"); ax.set_ylabel("$R$")
    ax.set_title("(b) Signal vs the geometry floor")
    ax.legend(fontsize=8); ax.set_ylim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)

    ax = axes[2]
    for sh in shells:
        d = tab[tab.shell_label == labels[sh]].sort_values("t")
        ax.plot(d.t, d.pop_R, color=colors[sh], lw=2, marker="o", ms=4)
        ax.plot(d.t, np.sqrt(-np.log(0.05) / d["n"].replace(0, np.nan)),
                color=colors[sh], lw=0.8, ls="--", alpha=0.6)
    ax.set_xlabel("timepoint"); ax.set_ylabel("population resultant of $\\phi$")
    ax.set_title("(c) Do nuclei agree on a lab direction?")
    ax.set_ylim(0, 1)
    ax.plot([], [], color="0.3", lw=0.8, ls="--", label="p = 0.05")
    ax.legend(fontsize=8); ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Normalised asymmetry score", fontsize=13)
    fig.tight_layout()
    save_figure(fig, save_as, config)
    plt.show()
    return tab


### 37e. Aggregate across the dataset, direction discarded

Three views, none of which assume a common direction:

**(a) and (b)** pool the score over every nucleus × every frame per shell —
magnitude only, direction thrown away — against the geometry floor and the
uniform-shell noise line.

**(c) the alignment-averaged profile.** Each nucleus is rotated so its *own*
asymmetry axis sits at 0°, then all are averaged. A population whose bright
sides point in random directions still produces a lobe here, provided each
nucleus is individually one-sided — that is exactly the direction-agnostic
statement, and it also shows the *shape* of the asymmetry: a broad cosine means
gradual enrichment, a narrow spike means a delivery hotspot.

**The null is not optional here.** Choosing the rotation from the same data you
then average is circular: pure noise, aligned to its own noisiest angle,
produces a lobe. `aligned_mean_profile` repeats the entire procedure on
θ-permuted profiles — same values, spatial structure destroyed — `n_perm` times,
building a distribution of *mean* profiles, and returns its 95th percentile.
Comparing an averaged observation against unaveraged null draws understates the
null by √n_units. The test is read at lag 0 only, since both curves peak there
by construction.

In [ ]:
def aligned_mean_profile(binned: pd.DataFrame, resultants: pd.DataFrame,
                         shell_idx: int, n_perm: int = 200,
                         min_coverage: float = 1.0, t_min: Optional[int] = None,
                         seed: int = 0) -> Optional[Dict]:
    """Rotate every nucleus so its own asymmetry axis is at 0, then average.

    Direction-agnostic by construction: no lab frame is assumed, so a
    population whose bright sides point in random directions still produces a
    lobe here as long as each nucleus is individually one-sided.

    The catch, and why the null matters: the rotation is chosen from the same
    data that is then averaged. Pure noise, aligned to its own noisiest angle,
    produces a lobe. The null repeats the ENTIRE procedure -- permute each
    unit's theta bins (same values, spatial structure destroyed), realign each
    to its own permuted axis, average across units -- and does that n_perm
    times to build a distribution of MEAN profiles. The envelope returned is
    the 95th percentile across those replicate means, so it is on the same
    footing as the observed mean. Comparing an averaged observation against
    unaveraged null draws understates the null by a factor of sqrt(n_units).
    """
    if binned is None or binned.empty or resultants is None or resultants.empty:
        return None
    rng = np.random.default_rng(seed)
    n_bins = binned.attrs.get("n_theta_bins", N_THETA_BINS)
    dtheta = 2 * np.pi / n_bins
    th = (np.arange(n_bins) + 0.5) * dtheta

    res = resultants[(resultants.shell_idx == shell_idx)
                     & (resultants.coverage >= min_coverage)]
    if t_min is not None:
        res = res[res.t >= t_min]
    if res.empty:
        return None
    lut = res.set_index(["t", "track_id"])

    units, prof = [], []
    for (t, tid), d in binned[binned.shell_idx == shell_idx].groupby(["t", "track_id"]):
        if (t, tid) not in lut.index:
            continue
        row = lut.loc[(t, tid)]
        d = d.sort_values("theta_bin")
        if not (d["n_samples"].to_numpy() > 0).all():
            continue
        I = d["I_mean"].to_numpy(dtype=float)
        bg = float(row["I_bg"])
        s = I - bg
        base = s.mean()
        if not np.isfinite(base) or base <= 0 or not np.isfinite(row["phi_rad"]):
            continue
        p = s / base                                  # normalised, unitless
        units.append(p)
        prof.append(np.roll(p, -int(np.round(float(row["phi_rad"]) / dtheta))))

    if len(prof) < 3:
        return None
    prof = np.vstack(prof)

    # null: n_perm replicate MEAN profiles, each built the same way as the observation
    null_means = np.empty((n_perm, n_bins))
    for b in range(n_perm):
        acc = np.empty((len(units), n_bins))
        for i, p in enumerate(units):
            q = rng.permutation(p)
            _, phi_q, _ = _resultant(th, np.clip(q, 0, None))
            sh_q = 0 if not np.isfinite(phi_q) else int(np.round(phi_q / dtheta))
            acc[i] = np.roll(q, -sh_q)
        null_means[b] = acc.mean(axis=0)

    lag_deg = (np.degrees(th) + 180) % 360 - 180
    order = np.argsort(lag_deg)
    obs = prof.mean(axis=0)
    null_hi = np.percentile(null_means, 95, axis=0)

    # Score at lag 0 only. Both the observation and the null are aligned so
    # their maximum sits at lag 0 by construction, so that is the one bin the
    # test is about; taking a max over all 36 lags instead re-introduces a
    # multiple-comparisons problem and lets a uniform field clear its own null.
    lag0 = int(np.argmin(np.abs(lag_deg)))
    return {
        "lag_deg": lag_deg[order],
        "mean": obs[order],
        "sem": (prof.std(axis=0, ddof=1) / np.sqrt(len(prof)))[order],
        "null_hi": null_hi[order],
        "null_mean": null_means.mean(axis=0)[order],
        "n_units": len(prof),
        "obs_at_axis": float(obs[lag0]),
        "null_at_axis": float(null_hi[lag0]),
        "peak_excess": float(obs[lag0] - null_hi[lag0]),
    }


def plot_aggregate_asymmetry(binned: pd.DataFrame, resultants: pd.DataFrame, config,
                             score: str = "R", n_perm: int = 200,
                             save_as: str = "aggregate_asymmetry"):
    """Whole-dataset asymmetry with direction discarded.

    (a) distribution of the per-nucleus score by shell, against the geometry
        floor and the uniform-shell null -- magnitude only, no direction.
    (b) ECDF of the same, so the whole distribution is visible rather than a
        median.
    (c) the alignment-averaged profile: what the asymmetry LOOKS like once
        every nucleus is rotated onto its own axis, with the permutation null.
    """
    if resultants is None or resultants.empty:
        print("No data to plot."); return

    shells = sorted(resultants["shell_idx"].unique())
    labels = (resultants.drop_duplicates("shell_idx")
              .set_index("shell_idx")["shell_label"].to_dict())
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))

    # (a) violins of R_corr, with the geometry floor overlaid
    ax = axes[0]
    data = [resultants.loc[resultants.shell_idx == sh, score].dropna().to_numpy()
            for sh in shells]
    pos = np.arange(len(shells))
    keep = [i for i, d in enumerate(data) if d.size > 1]
    if keep:
        vp = ax.violinplot([data[i] for i in keep], positions=pos[keep],
                           showmedians=True, widths=0.75)
        for i, body in zip(keep, vp["bodies"]):
            body.set_facecolor(colors[shells[i]]); body.set_alpha(0.55)
        for k in ("cmins", "cmaxes", "cbars", "cmedians"):
            if k in vp:
                vp[k].set_color("0.25"); vp[k].set_linewidth(1.0)
    for i, sh in enumerate(shells):
        g = resultants.loc[resultants.shell_idx == sh, "R_geom"].median()
        ax.hlines(g, i - 0.38, i + 0.38, color="#D7263D", lw=1.8, ls=":", zorder=5)
    ax.plot([], [], color="#D7263D", lw=1.8, ls=":", label="median $R_{geom}$")
    med_neff = resultants["n_eff"].median()
    if np.isfinite(med_neff):
        ax.axhline(R_critical(med_neff), color="k", lw=0.9, ls="--", alpha=0.6,
                   label="uniform-shell $R$ at p=0.05")
    ax.set_xticks(pos)
    ax.set_xticklabels([f"{labels[sh]}" for sh in shells], fontsize=8)
    ax.set_xlabel("shell (\u00b5m from nuclear surface)")
    ax.set_ylabel("asymmetry score $R$")
    ax.set_title("(a) Score distribution, all nuclei \u00d7 all frames")
    ax.set_ylim(0, 1); ax.legend(fontsize=7.5)
    ax.spines[["top", "right"]].set_visible(False)

    # (b) ECDF
    ax = axes[1]
    for sh in shells:
        d = np.sort(resultants.loc[resultants.shell_idx == sh, score].dropna().to_numpy())
        if d.size == 0:
            continue
        ax.step(d, np.arange(1, d.size + 1) / d.size, where="post",
                color=colors[sh], lw=1.8, label=f"{labels[sh]}  (n={d.size})")
        g = np.sort(resultants.loc[resultants.shell_idx == sh, "R_geom"].dropna().to_numpy())
        if g.size:
            ax.step(g, np.arange(1, g.size + 1) / g.size, where="post",
                    color=colors[sh], lw=1.0, ls=":", alpha=0.8)
    ax.set_xlabel("asymmetry score $R$"); ax.set_ylabel("cumulative fraction")
    ax.set_title("(b) ECDF (dotted = $R_{geom}$)")
    ax.set_xlim(0, 1); ax.legend(fontsize=7)
    ax.spines[["top", "right"]].set_visible(False)

    # (c) alignment-averaged profile
    ax = axes[2]
    any_prof = False
    for sh in shells:
        pr = aligned_mean_profile(binned, resultants, sh, n_perm=n_perm)
        if pr is None:
            continue
        any_prof = True
        ax.plot(pr["lag_deg"], pr["mean"], color=colors[sh], lw=2,
                label=f"{labels[sh]}  (n={pr['n_units']})")
        ax.fill_between(pr["lag_deg"], pr["mean"] - 1.96 * pr["sem"],
                        pr["mean"] + 1.96 * pr["sem"], color=colors[sh], alpha=0.18, lw=0)
        if pr["null_hi"] is not None:
            ax.plot(pr["lag_deg"], pr["null_hi"], color=colors[sh], lw=0.9,
                    ls="--", alpha=0.75)
    if any_prof:
        ax.axhline(1.0, color="0.4", lw=0.8)
        ax.set_xlim(-180, 180); ax.set_xticks([-180, -90, 0, 90, 180])
        ax.plot([], [], color="0.3", lw=0.9, ls="--", label="permutation null (95th pct)")
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, "no fully-covered shell units", ha="center", va="center",
                transform=ax.transAxes, fontsize=9, color="0.4")
    ax.set_xlabel("angle from each nucleus's own asymmetry axis (\u00b0)")
    ax.set_ylabel("intensity / shell mean")
    ax.set_title("(c) Alignment-averaged profile")
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Aggregate membrane asymmetry \u2014 direction discarded", fontsize=13)
    fig.tight_layout()
    save_figure(fig, save_as, config)
    plt.show()


### 37f. Run the shell-resolved analysis

In [ ]:
# ── Shell-resolved asymmetry: full run ────────────────────────────────────
# Reads the *_sel tables bound in §22: the selected single nuclei.
print("Figures for:", POP_LABEL)
radial_n_df = add_wall_normalised_radius(radial_sel, cfg)

# One background floor for the whole dataset — see 37b for why it must not
# come from the profile being scored.
BG = estimate_background(radial_n_df, q=0.05)
print(f"background floor: {BG:.1f} (5th percentile of all sweep samples)")

binned_df = bin_shell_angle(radial_n_df, cfg,
                            shell_edges_um=DEFAULT_SHELL_EDGES_UM,
                            n_theta_bins=N_THETA_BINS)
shell_res_df = compute_shell_resultants(binned_df, background=BG,
                                        min_coverage=MIN_SHELL_COVERAGE)
print(shell_res_df.shape)

# 1. the rose itself, for the longest-lived nucleus
_longest = int(tracked_sel.groupby("track_id")["t"].nunique().idxmax())
plot_shell_rose_facets(binned_df, shell_res_df, cfg, track_id=_longest,
                       value="mean_bgsub", radius_scale="sqrt", share_r="row",
                       save_as=f"shell_rose_track{_longest}")

# 2. normalised asymmetry score — table + graph
shell_score_table = plot_asymmetry_score_summary(shell_res_df, cfg)
display(shell_score_table)

# 3. aggregate across the whole dataset, direction discarded
plot_aggregate_asymmetry(binned_df, shell_res_df, cfg, n_perm=200)

# exports
shell_res_df.to_csv(POP_EXPORT_DIR / "shell_asymmetry_per_nucleus.csv", index=False)
shell_score_table.to_csv(POP_EXPORT_DIR / "shell_asymmetry_score_table.csv", index=False)
# Baseline keeps its original location; a selected population writes beside its figures.
binned_df.to_pickle((cfg.analysis_dir if ANALYSIS_POPULATION == "baseline" else POP_EXPORT_DIR)
                    / "shell_angle_binned.pkl")
print("figures ->", figures_dir(cfg))


In [ ]:
# ── Sanity read of the geometry control ───────────────────────────────────
# Any shell where R does not clear R_geom is reporting droplet placement,
# not membrane. Read this before interpreting the rose.
chk = (shell_res_df.groupby("shell_label")
       .agg(n=("R", "size"),
            R=("R", "median"),
            R_geom=("R_geom", "median"),
            excess=("R_excess", "median"),
            coverage=("coverage", "min"),
            frac_sig=("significant", "mean"))
       .round(3))
display(chk)

flagged = chk[chk.R_geom >= 0.5 * chk.R]
if len(flagged):
    print("\nShells where sampling geometry accounts for half or more of the "
          "measured asymmetry — do not quote these as membrane asymmetry:")
    display(flagged)
else:
    print("\nNo shell has R_geom within 2x of R.")


### §24 Database export

Writes the schema tables for the **selected single nuclei**. `NucleusZStack` lists, for each
nucleus, the selected section plus the neighbouring sections it suppressed in the local
area-maximum step (§16), with `Is_Selected_Max` marking the selected one.

In [ ]:
PIPELINE_VERSION = "v19.0"  # matches this notebook; bump manually per release
DB_EXPORT_DIR = cfg.output_dir / "database_export"
DB_EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def _fmt_slice_id(z: int) -> str:
    """Match the S#### convention used in the schema workbook."""
    return f"S{int(z):04d}"


def _fmt_centroid_xyz(x: float, y: float, z: float) -> str:
    return f"({x:.1f}, {y:.1f}, {z:.1f})"


In [ ]:
def build_nuclei_table(halo_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """One row per Nucleus_ID (track_id) per Time_Frame — analysis-ready table."""
    if halo_df.empty:
        return pd.DataFrame(columns=[
            "Nucleus_ID", "Droplet_ID", "FOV_ID", "Experiment_ID",
            "Time_Frame", "Time_Real_Minutes", "Stage_Classification",
            "Selected_Slice_ID", "Centroid_XYZ_px",
            "Cross_Sectional_Area_um2", "Volume_3D_um3", "NC_Ratio",
            "Segmentation_Pipeline_Version", "Model_Version",
            "QC_Flag", "QC_Notes", "Source_File_Path",
        ])

    experiment_id = Path(cfg.input_image_name).stem
    rows = []
    for row in halo_df.itertuples(index=False):
        qc_pass = bool(getattr(row, "valid_single_nucleus", True))
        rows.append({
            "Nucleus_ID":                    int(row.track_id),
            "Droplet_ID":                    "",  # not tracked as a distinct ID by this pipeline
            "FOV_ID":                        "FOV01",  # single stitched image per run
            "Experiment_ID":                 experiment_id,
            "Time_Frame":                    int(row.t),
            "Time_Real_Minutes":             float(getattr(row, "true_time_min", np.nan)),
            "Stage_Classification":          "",  # not computed by this pipeline
            "Selected_Slice_ID":             _fmt_slice_id(row.z),
            "Centroid_XYZ_px":               _fmt_centroid_xyz(row.centroid_x_px, row.centroid_y_px, row.z),
            "Cross_Sectional_Area_um2":      float(getattr(row, "nucleus_area_um2", np.nan)),
            "Volume_3D_um3":                 np.nan,  # 2D best-Z pipeline only
            "NC_Ratio":                      float(getattr(row, "nc_ratio", np.nan)),
            "Segmentation_Pipeline_Version": PIPELINE_VERSION,
            "Model_Version":                 cfg.model_name,
            "QC_Flag":                       "PASS" if qc_pass else "FAIL",
            "QC_Notes":                      "" if qc_pass else "excluded: multiple nuclei within exclusion radius",
            "Source_File_Path":              cfg.input_image_name,
        })
    return pd.DataFrame(rows)


nuclei_export_df = build_nuclei_table(halo_df, cfg)
print(nuclei_export_df.shape)
display(nuclei_export_df.head())


In [ ]:
def build_zstack_table(sections_df: pd.DataFrame, tracked_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """One row per section per nucleus per timepoint: the selected section and every section it
    suppressed in §16 (its local Z neighbourhood)."""
    cols = ["Nucleus_ID", "Time_Frame", "Slice_ID", "Centroid_XYZ_px",
            "Cross_Sectional_Area_um2", "Is_Selected_Max"]
    if tracked_df.empty:
        return pd.DataFrame(columns=cols)
    sel = sections_df[sections_df.selected]
    key = sel.reset_index().merge(tracked_df[["t", "z", "label", "track_id"]], on=["t", "z", "label"])
    owner = dict(zip(key["index"], key["track_id"]))
    rows = []
    for ix, r in sections_df.iterrows():
        tid = owner.get(ix) if r.selected else owner.get(int(r.suppressed_by))
        if tid is None:
            continue
        rows.append({"Nucleus_ID": int(tid), "Time_Frame": int(r.t), "Slice_ID": _fmt_slice_id(r.z),
                     "Centroid_XYZ_px": _fmt_centroid_xyz(r.centroid_x_px, r.centroid_y_px, r.z),
                     "Cross_Sectional_Area_um2": float(r.area_um2), "Is_Selected_Max": bool(r.selected)})
    return pd.DataFrame(rows, columns=cols)


zstack_export_df = build_zstack_table(sections_df, tracked_df, cfg)
print(zstack_export_df.shape)
display(zstack_export_df.head())


In [ ]:
def build_raw_intensities_table(halo_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """Per-halo, per-channel integrated intensities at the selected Z-slice.

    NOTE: only the Mcherry/NLS channel is currently measured by
    measure_fiji_equivalent_halos (Section 18). NPC and Membrane columns,
    and all Background_* columns, are written as NaN until that function
    is extended to loop over channels.
    """
    cols = [
        "Nucleus_ID", "Time_Frame",
        "Halo1_Mcherry", "Halo2_Mcherry", "Halo3_Mcherry", "Halo4_Mcherry",
        "Halo1_NPC", "Halo2_NPC", "Halo3_NPC", "Halo4_NPC",
        "Halo1_Membrane", "Halo2_Membrane", "Halo3_Membrane", "Halo4_Membrane",
        "Background_Mcherry", "Background_NPC", "Background_Membrane",
    ]
    if halo_df.empty:
        return pd.DataFrame(columns=cols)

    rows = []
    for row in halo_df.itertuples(index=False):
        rd = row._asdict()
        rec = {
            "Nucleus_ID": int(rd["track_id"]),
            "Time_Frame": int(rd["t"]),
        }
        for i in range(1, cfg.n_halos + 1):
            rec[f"Halo{i}_Mcherry"] = rd.get(f"ring_{i}_intden", np.nan)
            rec[f"Halo{i}_NPC"] = np.nan       # not measured yet — see docstring
            rec[f"Halo{i}_Membrane"] = np.nan  # not measured yet — see docstring
        rec["Background_Mcherry"] = np.nan
        rec["Background_NPC"] = np.nan
        rec["Background_Membrane"] = np.nan
        rows.append(rec)
    return pd.DataFrame(rows, columns=cols)


raw_intensities_export_df = build_raw_intensities_table(halo_df, cfg)
print(raw_intensities_export_df.shape)
display(raw_intensities_export_df.head())


In [ ]:
def build_radial_profile_table(radial_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """Long-form radial sweep -> schema RadialProfile (one row per ray sample point).

    W_Wall_Proximity_Index, Is_Ridge_Point, and Cluster_ID are written as
    placeholders (NaN / False / blank) -- wall-proximity index and ridge/DBSCAN
    clustering are not computed in v16.
    """
    cols = [
        "Nucleus_ID", "Time_Frame", "Theta_deg", "Rho_Normalized",
        "W_Wall_Proximity_Index", "Channel", "Intensity",
        "Is_Ridge_Point", "Cluster_ID",
    ]
    if radial_df is None or radial_df.empty:
        return pd.DataFrame(columns=cols)

    channel_label_by_index = {
        cfg.membrane_channel_index: "Membrane",
        cfg.nuclear_channel_index:  "Mcherry",
        cfg.npc_channel_index:      "NPC",
    }
    radial_channel_label = channel_label_by_index.get(cfg.radial_channel_index, "Membrane")

    out = pd.DataFrame({
        "Nucleus_ID":             radial_df["track_id"].astype(int),
        "Time_Frame":             radial_df["t"].astype(int),
        "Theta_deg":              np.degrees(radial_df["theta_rad"].astype(float)),
        "Rho_Normalized":         radial_df["distance_norm"].astype(float),
        "W_Wall_Proximity_Index": np.nan,
        "Channel":                radial_channel_label,
        "Intensity":              radial_df["intensity"].astype(float),
        "Is_Ridge_Point":         False,
        "Cluster_ID":             "",
    })
    return out[cols]


radial_profile_export_df = build_radial_profile_table(radial_df, cfg)
print(radial_profile_export_df.shape)
display(radial_profile_export_df.head())


In [ ]:
def build_experimental_cfg_table(cfg: PipelineConfig) -> pd.DataFrame:
    experiment_id = Path(cfg.input_image_name).stem
    channel_labels = {0: "Membrane", 1: "NLS", 2: "NPC"}
    row = {
        "Experiment_ID":                 experiment_id,
        "FOV_ID":                        "FOV01",
        "Droplet_ID":                    "",
        "Date":                          datetime.utcnow().strftime("%Y-%m-%d"),
        "Pixel_Size_um":                 cfg.pixel_size_um,
        "Z_Step_um":                     cfg.z_step_um,
        "Num_Z_Planes":                  int(img_5d.shape[1]) if "img_5d" in dir() else np.nan,
        "Frame_Interval_min":            cfg.minutes_per_tile * cfg.tile_rows * cfg.tile_cols,
        "Channel0_Label":                channel_labels.get(0, ""),
        "Channel1_Label":                channel_labels.get(1, ""),
        "Channel2_Label":                channel_labels.get(2, ""),
        "Microscope":                    "",
        "Objective":                     "",
        "Operator":                      "",
        "Segmentation_Pipeline_Version": PIPELINE_VERSION,
        "Model_Version":                 cfg.model_name,
        "Raw_File_Path":                 str(cfg.input_image_path),
        "Notes":                         "",
    }
    return pd.DataFrame([row])


experimental_cfg_export_df = build_experimental_cfg_table(cfg)
display(experimental_cfg_export_df)


In [ ]:
def export_database_schema_tables(export_dir: Path) -> dict:
    tables = {"Nuclei": nuclei_export_df, "NucleusZStack": zstack_export_df,
              "RawIntensities": raw_intensities_export_df, "RadialProfile": radial_profile_export_df,
              "Experimental_Cfg": experimental_cfg_export_df}
    written = {}
    for name, df in tables.items():
        out_path = export_dir / f"{name}.csv"
        df.to_csv(out_path, index=False)
        written[name] = str(out_path)
        print(f"{name}: {len(df)} rows -> {out_path}")
    return written


if RUN_DB_SCHEMA_EXPORT:
    written_paths = export_database_schema_tables(DB_EXPORT_DIR)


## 25. Conditions × replicates

Pools the single-nucleus tables (§20 exports) of the three replicates of each condition, fits one
median logistic per condition, and takes uncertainty from a **two-level bootstrap** (replicates,
then nuclei within each replicate): replicates are the experimental unit, so resampling nuclei
alone would make the intervals far too narrow. Per-replicate fits are shown next to the pooled
curve. Phase windows are fixed from the pooled **reference** condition so velocities compare over
the same time windows; each condition's own `tm` and `k` answer whether the transition shifts.

Every replicate must be on the same biological time zero (`time_zero_offset_min` in its §3).
With three replicates the intervals are coarse; plot the replicate points and read them as such.
`VELOCITY_MARGIN` is open: leave it `None` until a biologically meaningful difference is set.

In [ ]:
# §25a CONDITIONS × REPLICATES: point each condition at its replicate run directories (v19 runs).
# Control: the most recent control v19 run that has exported its single-nucleus table
# (override CONTROL_RUN_DIR by hand to pin a specific run).
CONTROL_RUN_DIR = None
if CONTROL_RUN_DIR is None:
    _ctl = sorted((p for p in paths.runs_dir.glob("control_extract_1.1_v19*")
                   if (p / "exports" / "single_nuclei" / "single_nuclei.pkl").exists()),
                  key=lambda p: (p / "exports" / "single_nuclei" / "single_nuclei.pkl").stat().st_mtime)
    assert _ctl, "No control v19 run with an exported single-nucleus table under " + str(paths.runs_dir)
    CONTROL_RUN_DIR = _ctl[-1]
print("control run:", CONTROL_RUN_DIR)
CONDITION_RUNS = {
    "control":  [Path(CONTROL_RUN_DIR)],
    "P150_CC1": [run.run_dir],         # add the other P150 replicates here as they are run
}
REFERENCE_CONDITION = "control"
N_BOOT_COND = 500
VELOCITY_MARGIN = None                 # µm²/min; None = not yet decided (no pass/fail reported)
COMPARISON_DIR = paths.data_root / "Analysis" / ("conditions_" + datetime.now().strftime("%Y%m%d"))
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

DATA, PROVENANCE = {}, {}
for cond, run_dirs in CONDITION_RUNS.items():
    DATA[cond] = {}
    for rd in map(Path, run_dirs):
        tab = pd.read_pickle(rd / "exports" / "single_nuclei" / "single_nuclei.pkl")
        prov = json.loads((rd / "exports" / "single_nuclei" / "provenance.json").read_text())
        DATA[cond][rd.name] = (tab.true_time_min.to_numpy(float), tab.area_um2.to_numpy(float))
        PROVENANCE[(cond, rd.name)] = prov
prov_df = pd.DataFrame({f"{c} | {r}": {k: p.get(k) for k in ("model_sha256", "time_zero_offset_min", "n_nuclei",
                                                               "population_tag", "selection")}
                        for (c, r), p in PROVENANCE.items()}).T
display(prov_df)
if prov_df.model_sha256.nunique() > 1:
    print("WARNING: replicates were segmented with different models.")
if prov_df.selection.astype(str).nunique() > 1:
    print("WARNING: replicates used different selection rules.")


In [ ]:
# §25b Pooled fit per condition, hierarchical bootstrap, per-replicate fits, differences vs reference.
COND = {}
for cond, reps in DATA.items():
    t = np.concatenate([v[0] for v in reps.values()]); y = np.concatenate([v[1] for v in reps.values()])
    p = fit_median_logistic(t, y)
    COND[cond] = dict(p=p, boots=hier_boot_logistic(reps, p, n=N_BOOT_COND),
                      per_rep={r: fit_median_logistic(*v, p) for r, v in reps.items()},
                      t_obs=(float(np.floor(t.min())), float(np.ceil(t.max()))))

SHARED_PHASES = phase_windows(COND[REFERENCE_CONDITION]["p"], t_obs=COND[REFERENCE_CONDITION]["t_obs"],
                              min_len=cfg.frame_interval_min)
print("shared phase windows (from the pooled reference):", SHARED_PHASES)
for cond, R in COND.items():
    R["m"] = phase_velocity_table(R["boots"], SHARED_PHASES)
    R["rep_m"] = phase_velocity_table(np.array(list(R["per_rep"].values())), SHARED_PHASES).set_axis(list(R["per_rep"]))

COND_SUMMARY = pd.DataFrame({c: {k: ci_text(v) for k, v in R["m"].items()} for c, R in COND.items()})
display(COND_SUMMARY); COND_SUMMARY.to_csv(COMPARISON_DIR / "condition_summary.csv")
for cond, R in COND.items():
    print(f"\n{cond}: per-replicate fits"); display(R["rep_m"].round(3))
    R["rep_m"].to_csv(COMPARISON_DIR / f"per_replicate_{cond}.csv")

ref_m = COND[REFERENCE_CONDITION]["m"]
for cond, R in COND.items():
    if cond == REFERENCE_CONDITION:
        continue
    d = R["m"] - ref_m.to_numpy()
    out = pd.DataFrame({"difference (95% CI)": {k: ci_text(v) for k, v in d.items()}})
    if VELOCITY_MARGIN is not None:
        out["equivalent (90% CI within ±margin)"] = {
            k: (lambda lo, hi: "yes" if -VELOCITY_MARGIN < lo and hi < VELOCITY_MARGIN else "no")(
                *np.nanquantile(v, [.05, .95])) for k, v in d.items() if k.startswith("abs vel")}
    print(f"\n{cond} − {REFERENCE_CONDITION} (hierarchical bootstrap)"); display(out)
    out.to_csv(COMPARISON_DIR / f"difference_{cond}_vs_{REFERENCE_CONDITION}.csv")


In [ ]:
# §25c FIGURE: pooled curves (bands) with single-replicate fits (dashed), and both velocities.
t_lo = min(R["t_obs"][0] for R in COND.values()); t_hi = max(R["t_obs"][1] for R in COND.values())
tt = np.linspace(t_lo, t_hi, 300)
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for (cond, R), col in zip(COND.items(), plt.cm.tab10.colors):
    cur = logi(R["boots"], tt)
    for ax, g in ((axs[0], cur), (axs[1], np.gradient(cur, tt, axis=1)), (axs[2], np.gradient(np.log(cur), tt, axis=1))):
        ax.fill_between(tt, *np.quantile(g, [.025, .975], axis=0), color=col, alpha=.2)
        ax.plot(tt, np.median(g, axis=0), c=col, lw=2.5, label=f"{cond} (pooled)")
    for r, q in R["per_rep"].items():
        axs[0].plot(tt, logi(q, tt), c=col, lw=1, ls="--", alpha=.8)
for ax, (ttl, yl) in zip(axs, (("area", "µm²"), ("absolute velocity", "dA/dt (µm²/min)"),
                               ("relative velocity", "d lnA/dt (1/min)"))):
    for (ph, win), shade in zip(SHARED_PHASES.items(), ("#e8f0ff", "#fff3e0", "#eef7ee")):
        if win: ax.axvspan(*win, color=shade, zorder=0)
    ax.set(title=ttl, xlabel="Acquisition time (minutes)", ylabel=yl); ax.grid(alpha=.2)
axs[0].legend(title="dashed = single replicates"); plt.tight_layout()
fig.savefig(COMPARISON_DIR / "conditions_growth.png", dpi=300); plt.show()
print("outputs ->", COMPARISON_DIR)
